In [4]:
# BLOCK 1 — Locate the export and inspect its files.
# Read-only: no training, model loading, or LLM calls.

from pathlib import Path
import json

INPUT_ROOT = Path("/kaggle/input")
EXPORT_CHOICE = None  # Set a number here only if multiple exports are found.
LLM_EXPORT_DIR = LLM_BUNDLE_DIR = LLM_SUMMARY = LLM_SETUP = None
LLM_FILES = {}

if not INPUT_ROOT.is_dir():
    raise FileNotFoundError("Run this cell inside your Kaggle notebook.")

_summaries = sorted(INPUT_ROOT.rglob("llm_export_summary.json"))

if not _summaries:
    raise FileNotFoundError(
        "No llm_export_summary.json found. "
        "Check that models-and-data is attached to this notebook."
    )

print("EXPORTS FOUND")
for _i, _path in enumerate(_summaries):
    print(f"  [{_i}] {_path}")

if EXPORT_CHOICE is None:
    if len(_summaries) != 1:
        raise ValueError(
            "Set EXPORT_CHOICE to the intended export number above, then rerun."
        )
    EXPORT_CHOICE = 0

if not isinstance(EXPORT_CHOICE, int) or not 0 <= EXPORT_CHOICE < len(_summaries):
    raise ValueError("EXPORT_CHOICE must match a number listed above.")

_summary_path = _summaries[EXPORT_CHOICE]
LLM_EXPORT_DIR = _summary_path.parent
LLM_BUNDLE_DIR = LLM_EXPORT_DIR.parent

with _summary_path.open(encoding="utf-8") as _f:
    LLM_SUMMARY = json.load(_f)

print("\nSELECTED EXPORT:", LLM_EXPORT_DIR)
print("Bundle:", LLM_BUNDLE_DIR)
print("Status:", LLM_SUMMARY.get("status"))
print("Rows:", LLM_SUMMARY.get("rows"))
print("Dates:", LLM_SUMMARY.get("start"), "->", LLM_SUMMARY.get("end"))
print("Splits:", LLM_SUMMARY.get("existing_split_counts"))
print("Ensemble weights:", LLM_SUMMARY.get("ensemble_weights"))

# Inspect filenames and sizes without loading model checkpoints.
_paths = sorted(_p for _p in LLM_BUNDLE_DIR.rglob("*") if _p.is_file())
LLM_FILES = {
    str(_p.relative_to(LLM_BUNDLE_DIR)): _p
    for _p in _paths
}

print(f"\nBUNDLE FILES ({len(LLM_FILES)} total; displaying up to 100)")
for _relative, _path in list(LLM_FILES.items())[:100]:
    print(f"  {_relative}  [{_path.stat().st_size / 1024**2:.3f} MiB]")

_setup_path = LLM_BUNDLE_DIR / "setup" / "config.json"
if _setup_path.is_file():
    with _setup_path.open(encoding="utf-8") as _f:
        LLM_SETUP = json.load(_f)
    print("\nSetup config:", _setup_path)
    if isinstance(LLM_SETUP, dict):
        print("Setup keys:", sorted(LLM_SETUP))
else:
    print("\nNo setup/config.json beside this export; see inventory above.")

print("\nEXPORT NOTES")
for _note in LLM_SUMMARY.get("notes", []):
    print(" -", _note)

print("\nBlock 1 complete: paths recorded. Tables and models are not loaded yet.")

EXPORTS FOUND
  [0] /kaggle/input/datasets/nidhishgautam/models-and-data/llm_export/llm_export_summary.json

SELECTED EXPORT: /kaggle/input/datasets/nidhishgautam/models-and-data/llm_export
Bundle: /kaggle/input/datasets/nidhishgautam/models-and-data
Status: PREPARATION_ONLY
Rows: 3243
Dates: 2017-11-09 00:00:00+00:00 -> 2026-09-25 00:00:00+00:00
Splits: {'TRAIN': 1879, 'VAL': 273, 'TEST': 1091}
Ensemble weights: {'Majority': 0.0, 'Mean': 0.0, 'MLP': 0.27919944171617833, 'PPO': 0.7208005582838216}

BUNDLE FILES (35 total; displaying up to 100)
  README.txt  [0.001 MiB]
  cryptobert/config.json  [0.001 MiB]
  cryptobert/model.safetensors  [475.517 MiB]
  cryptobert/tokenizer.json  [3.394 MiB]
  cryptobert/tokenizer_config.json  [0.000 MiB]
  llm_export/archive/existing_splits.csv  [0.167 MiB]
  llm_export/archive/model_inputs_raw.csv  [1.218 MiB]
  llm_export/archive/model_signals_saved.csv  [0.367 MiB]
  llm_export/evaluator_only/outcomes.csv  [0.159 MiB]
  llm_export/evaluator_only/pr

In [6]:
# BLOCK 2 — Load the exported CSVs and check their structure.
# Outcomes/prices stay separate from reviewer inputs. No models or LLM calls.

import numpy as np
import pandas as pd
from IPython.display import display

if globals().get("LLM_EXPORT_DIR") is None or globals().get("LLM_SUMMARY") is None:
    raise RuntimeError("Run Block 1 successfully first.")

LLM_TABLES_READY = False
LLM_CAUSALITY_VERIFIED = False
LLM_TABLES = {}
_file_map = {
    "model_inputs": "archive/model_inputs_raw.csv",
    "features": "reviewer_candidates/features.csv",
    "signals": "archive/model_signals_saved.csv",
    "splits": "archive/existing_splits.csv",
    "prices": "evaluator_only/prices.csv",
    "outcomes": "evaluator_only/outcomes.csv",
}
_loaded, _report = {}, []

for _name, _relative in _file_map.items():
    _path = LLM_EXPORT_DIR / _relative
    if not _path.is_file():
        raise FileNotFoundError(f"Missing export file: {_path}")
    _df = pd.read_csv(_path, index_col=0)
    if _df.empty:
        raise ValueError(f"{_name}: empty table")

    # Require an explicit timezone; do not silently assume local dates are UTC.
    _dates = _df.index.astype(str).str.strip()
    if not _dates.str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", regex=True).all():
        raise ValueError(f"{_name}: timestamps lack an explicit timezone; inspect before continuing")
    _df.index = pd.to_datetime(_dates, utc=True, errors="raise")
    _df.index.name = "source_timestamp_utc"
    if _df.index.hasnans or not _df.index.is_unique or not _df.index.is_monotonic_increasing:
        raise ValueError(f"{_name}: timestamps must be valid, unique, and increasing")

    _numeric = _df.select_dtypes(include=[np.number]).to_numpy(dtype=float)
    _report.append({
        "table": _name, "rows": len(_df), "columns": len(_df.columns),
        "missing_cells": int(_df.isna().sum().sum()),
        "infinite_values": int(np.isinf(_numeric).sum()),
    })
    _loaded[_name] = _df

_index = _loaded["model_inputs"].index
for _row in _report:
    _row["same_timestamps"] = _loaded[_row["table"]].index.equals(_index)
display(pd.DataFrame(_report))

if len(_index) != int(LLM_SUMMARY["rows"]):
    raise ValueError("Row count differs from the export summary")
# Feature/signal/split/outcome rows must match exactly. The full price history
# may include extra bars needed for warm-up or subsequent outcome evaluation.
_bad_tables = [
    _row["table"] for _row in _report
    if _row["table"] != "prices" and not _row["same_timestamps"]
]
if _bad_tables:
    raise ValueError(f"These tables do not match the feature timestamps: {_bad_tables}")

_price_index = _loaded["prices"].index
_missing_prices = _index.difference(_price_index)
_extra_prices = _price_index.difference(_index)

print("\nPRICE COVERAGE")
print("Feature range:", _index.min(), "->", _index.max())
print("Price range:  ", _price_index.min(), "->", _price_index.max())
print("Feature timestamps without prices:", len(_missing_prices))
print("Extra price timestamps:", len(_extra_prices))
for _ts in _extra_prices[:10]:
    _where = ("before feature range" if _ts < _index.min() else
              "after feature range" if _ts > _index.max() else "inside feature range")
    print(f"  {_ts} — {_where}")
if len(_missing_prices):
    raise ValueError(f"Prices are missing for feature timestamps: {list(_missing_prices[:10])}")
print("Full price history preserved; no rows removed or shifted.")

for _name, _key in [("model_inputs", "model_feature_order"),
                    ("features", "reviewer_feature_columns")]:
    if list(_loaded[_name].columns) != LLM_SUMMARY[_key]:
        raise ValueError(f"{_name}: columns/order differ from the export summary")
    if not np.isfinite(_loaded[_name].to_numpy(dtype=float)).all():
        raise ValueError(f"{_name}: missing or infinite features need inspection")

_excluded = set(LLM_SUMMARY.get("excluded_reviewer_columns", []))
if _excluded.intersection(_loaded["features"].columns):
    raise ValueError("An excluded column is present in the reviewer features")

# Parse split membership explicitly: strings such as 'False' must stay false.
_bool_map = {"true": True, "false": False, "1": True, "0": False}
_flags = _loaded["splits"][["TRAIN", "VAL", "TEST"]].apply(
    lambda s: s.astype(str).str.strip().str.lower().map(_bool_map)
)
if _flags.isna().any().any() or not _flags.sum(axis=1).eq(1).all():
    raise ValueError("Every timestamp must belong to exactly one valid original split")
_loaded["splits"][["TRAIN", "VAL", "TEST"]] = _flags.astype(bool)

_split_report = []
_last_end = None
for _split in ["TRAIN", "VAL", "TEST"]:
    _idx = _index[_flags[_split].to_numpy(dtype=bool)]
    if len(_idx) != int(LLM_SUMMARY["existing_split_counts"][_split]) or len(_idx) == 0:
        raise ValueError(f"{_split}: count differs from summary, or split is empty")
    if _last_end is not None and _idx.min() <= _last_end:
        raise ValueError("Original splits are not chronologically separated")
    _last_end = _idx.max()
    _split_report.append({"split": _split, "rows": len(_idx),
                          "first_date": str(_idx.min().date()),
                          "last_date": str(_idx.max().date())})

LLM_TABLES = _loaded
LLM_TABLES_READY = True
print("\nORIGINAL SPLITS")
display(pd.DataFrame(_split_report))
for _name in ["signals", "prices", "outcomes"]:
    print(f"\n{_name.upper()} columns:", list(LLM_TABLES[_name].columns))
print("\nprev_lbl in trading-model inputs:", "prev_lbl" in LLM_TABLES["model_inputs"])
print("prev_lbl in reviewer features:", "prev_lbl" in LLM_TABLES["features"])
print("\nBlock 2 complete: structural checks passed. Timing/leakage checks remain.")
print("Keep prices/outcomes separate from LLM prompts. Experiment periods remain unassigned.")


,table,rows,columns,missing_cells,infinite_values,same_timestamps
0,model_inputs,3243,54,0,0,True
1,features,3243,53,0,0,True
2,signals,3243,11,0,0,True
3,splits,3243,4,0,0,True
4,prices,3244,5,0,0,False
5,outcomes,3243,2,0,0,True



PRICE COVERAGE
Feature range: 2017-11-09 00:00:00+00:00 -> 2026-09-25 00:00:00+00:00
Price range:   2017-11-09 00:00:00+00:00 -> 2026-09-27 00:00:00+00:00
Feature timestamps without prices: 0
Extra price timestamps: 1
  2026-09-27 00:00:00+00:00 — after feature range
Full price history preserved; no rows removed or shifted.

ORIGINAL SPLITS


,split,rows,first_date,last_date
0,TRAIN,1879,2017-11-09,2022-12-31
1,VAL,273,2023-01-01,2023-09-30
2,TEST,1091,2023-10-01,2026-09-25



SIGNALS columns: ['sentiment_rule__maj_sig', 'sentiment_rule__maj_conf', 'sentiment_rule__mean_sig', 'sentiment_rule__mean_conf', 'mlp_direction', 'mlp_max_probability', 'ppo_proposal_saved', 'ensemble_proposal_saved', 'mlp_p_bear', 'mlp_p_neutral', 'mlp_p_bull']

PRICES columns: ['open', 'high', 'low', 'close', 'volume']

OUTCOMES columns: ['y', 'fwd_ret_1']

prev_lbl in trading-model inputs: True
prev_lbl in reviewer features: False

Block 2 complete: structural checks passed. Timing/leakage checks remain.
Keep prices/outcomes separate from LLM prompts. Experiment periods remain unassigned.


In [7]:
# BLOCK 3 — Check time coverage and inspect backtest settings.
# No rows are dropped, filled, shifted, or sent to an LLM.

import json
import numpy as np
import pandas as pd
from IPython.display import display

# Supports either a completed Block 2 or its interrupted/repair state.
_tables = globals().get("LLM_TABLES") or globals().get("_loaded")
_required = {"model_inputs", "features", "signals", "splits", "prices", "outcomes"}

if not isinstance(_tables, dict) or not _required.issubset(_tables):
    raise RuntimeError("Run Block 2 to load the six CSVs first.")

LLM_TABLES_READY = False
LLM_CAUSALITY_VERIFIED = False
_idx = _tables["model_inputs"].index

if len(_idx) != int(LLM_SUMMARY["rows"]):
    raise ValueError("Feature row count differs from the summary")

for _name, _df in _tables.items():
    if not isinstance(_df.index, pd.DatetimeIndex) or _df.index.tz is None:
        raise ValueError(f"{_name}: timezone-aware datetime index required")

    if (
        _df.index.hasnans
        or not _df.index.is_unique
        or not _df.index.is_monotonic_increasing
    ):
        raise ValueError(f"{_name}: invalid, duplicate, or unsorted timestamps")

    if _name != "prices" and not _df.index.equals(_idx):
        raise ValueError(f"{_name}: timestamps differ from model inputs")

if len(_idx.difference(_tables["prices"].index)):
    raise ValueError("Prices do not cover every feature timestamp")

for _name, _key in [
    ("model_inputs", "model_feature_order"),
    ("features", "reviewer_feature_columns"),
]:
    if list(_tables[_name].columns) != LLM_SUMMARY[_key]:
        raise ValueError(f"{_name}: feature columns/order differ from summary")
    if not np.isfinite(_tables[_name].to_numpy(dtype=float)).all():
        raise ValueError(f"{_name}: missing or infinite feature values")

_excluded = set(LLM_SUMMARY.get("excluded_reviewer_columns", []))
if _excluded.intersection(_tables["features"].columns):
    raise ValueError("An excluded column is present in reviewer features")

_flags = _tables["splits"][["TRAIN", "VAL", "TEST"]].apply(
    lambda s: s.astype(str).str.strip().str.lower().map(
        {"true": True, "false": False, "1": True, "0": False}
    )
)

if _flags.isna().any().any() or not _flags.sum(axis=1).eq(1).all():
    raise ValueError("Each row must belong to exactly one original split")

_split_rows, _previous_end = [], None

for _split in ["TRAIN", "VAL", "TEST"]:
    _dates = _idx[_flags[_split].to_numpy(dtype=bool)]
    if (
        len(_dates) == 0
        or len(_dates) != int(LLM_SUMMARY["existing_split_counts"][_split])
    ):
        raise ValueError(f"{_split}: split count differs from summary")

    if _previous_end is not None and _dates.min() <= _previous_end:
        raise ValueError("Original splits are not chronologically separated")

    _previous_end = _dates.max()
    _split_rows.append({
        "split": _split,
        "rows": len(_dates),
        "start": str(_dates.min().date()),
        "end": str(_dates.max().date()),
    })

LLM_TABLES = _tables
LLM_TABLES_READY = True

print("STRUCTURAL CHECKS PASSED")
display(pd.DataFrame(_split_rows))

print("\nDAILY TIMESTAMP COVERAGE")
for _name in ["features", "prices"]:
    _dates = _tables[_name].index
    if not _dates.equals(_dates.normalize()):
        raise ValueError(f"{_name}: expected midnight timestamps; inspect timing")

    _grid = pd.date_range(_dates.min(), _dates.max(), freq="D")
    _gaps = _grid.difference(_dates)
    print(f"{_name}: {len(_gaps)} missing calendar days")
    print("Missing dates (up to 10):", [str(x.date()) for x in _gaps[:10]])

# Diagnostic only: we have not yet established the execution convention.
_no_next_day = _idx[
    ~(_idx + pd.Timedelta(days=1)).isin(_tables["prices"].index)
]
print("\nFeature dates without a price exactly one calendar day later:")
print([str(x.date()) for x in _no_next_day[:10]],
      "| total:", len(_no_next_day))

for _name in ["signals", "prices", "outcomes"]:
    print(f"\n{_name.upper()} columns:", list(_tables[_name].columns))

print("\nprev_lbl in model inputs:",
      "prev_lbl" in _tables["model_inputs"])
print("prev_lbl in reviewer features:",
      "prev_lbl" in _tables["features"])

_keys = [
    "TRAIN_END", "VAL_START", "VAL_END", "OOS_START", "OOS_END",
    "ANN_DAYS", "RF_ANNUAL", "FEE", "SLIPPAGE", "INIT_CASH",
    "PPO_EPISODE_LEN", "PPO_REWARD", "TBL_MIN_HOLD", "TBL_EWMA_SPAN",
    "TBL_LOOKBACK_MONTHS", "TARGET_OOS_SHARPE", "QUICK_MODE",
]

print("\nBACKTEST SETTINGS")
_cfg = globals().get("LLM_SETUP")
if isinstance(_cfg, dict):
    print(json.dumps(
        {k: _cfg[k] for k in _keys if k in _cfg},
        indent=2,
        default=str,
    ))
else:
    print("Setup config was not loaded in Block 1.")

print("\nBlock 3 complete. Execution timing still requires verification.")
print("No performance calculated; no experiment periods assigned.")

STRUCTURAL CHECKS PASSED


,split,rows,start,end
0,TRAIN,1879,2017-11-09,2022-12-31
1,VAL,273,2023-01-01,2023-09-30
2,TEST,1091,2023-10-01,2026-09-25



DAILY TIMESTAMP COVERAGE
features: 0 missing calendar days
Missing dates (up to 10): []
prices: 1 missing calendar days
Missing dates (up to 10): ['2026-09-26']

Feature dates without a price exactly one calendar day later:
['2026-09-25'] | total: 1

SIGNALS columns: ['sentiment_rule__maj_sig', 'sentiment_rule__maj_conf', 'sentiment_rule__mean_sig', 'sentiment_rule__mean_conf', 'mlp_direction', 'mlp_max_probability', 'ppo_proposal_saved', 'ensemble_proposal_saved', 'mlp_p_bear', 'mlp_p_neutral', 'mlp_p_bull']

PRICES columns: ['open', 'high', 'low', 'close', 'volume']

OUTCOMES columns: ['y', 'fwd_ret_1']

prev_lbl in model inputs: True
prev_lbl in reviewer features: False

BACKTEST SETTINGS
{
  "TRAIN_END": "2022-12-31",
  "VAL_START": "2023-01-01",
  "VAL_END": "2023-09-30",
  "OOS_START": "2023-10-01",
  "OOS_END": "2026-09-27",
  "ANN_DAYS": 365,
  "RF_ANNUAL": 0.04,
  "FEE": 0.001,
  "SLIPPAGE": 0.0005,
  "INIT_CASH": 100000,
  "PPO_EPISODE_LEN": 252,
  "PPO_REWARD": "diff_sharpe

In [8]:
# BLOCK 4 — Audit prev_lbl and saved forward returns.
# These are evaluator diagnostics, not LLM inputs.

import numpy as np
import pandas as pd
from IPython.display import display

if not globals().get("LLM_TABLES_READY", False):
    raise RuntimeError("Run Block 3 successfully first.")

LLM_CAUSALITY_VERIFIED = False

_x = LLM_TABLES["model_inputs"]
_out = LLM_TABLES["outcomes"]
_px = LLM_TABLES["prices"]
_idx = _x.index

_prev = pd.to_numeric(_x["prev_lbl"], errors="raise")
_y = pd.to_numeric(_out["y"], errors="raise")
_saved_ret = pd.to_numeric(
    _out["fwd_ret_1"], errors="raise"
).reindex(_idx)

# 1. Does prev_lbl equal the previous calendar day's target label?
_y_previous_day = pd.Series(
    _y.reindex(_idx - pd.Timedelta(days=1)).to_numpy(),
    index=_idx,
)

_valid_label = np.isfinite(_prev) & np.isfinite(_y_previous_day)
_label_match = pd.Series(
    np.isclose(_prev, _y_previous_day, rtol=0, atol=1e-10),
    index=_idx,
)

_split_flags = LLM_TABLES["splits"][["TRAIN", "VAL", "TEST"]].apply(
    lambda s: s.astype(str).str.strip().str.lower().map(
        {"true": True, "false": False, "1": True, "0": False}
    )
)
if _split_flags.isna().any().any():
    raise ValueError("Invalid split membership")

_label_report = []

for _split in ["TRAIN", "VAL", "TEST"]:
    _mask = _valid_label & _split_flags[_split].astype(bool)
    # Separate comparisons where at least one value is nonzero.
    _nonzero = _mask & ((_prev != 0) | (_y_previous_day != 0))

    _label_report.append({
        "split": _split,
        "compared_rows": int(_mask.sum()),
        "match_pct": (
            100 * _label_match[_mask].mean()
            if _mask.any() else np.nan
        ),
        "nonzero_comparisons": int(_nonzero.sum()),
        "nonzero_match_pct": (
            100 * _label_match[_nonzero].mean()
            if _nonzero.any() else np.nan
        ),
    })

LLM_LABEL_AUDIT = pd.DataFrame(_label_report)

print("PREV_LBL VERSUS PREVIOUS CALENDAR DAY'S Y")
display(LLM_LABEL_AUDIT)
print("Label values:", sorted(_y.dropna().unique().tolist()))

# 2. Recompute two possible forward-return definitions.
_close = pd.to_numeric(_px["close"], errors="raise")

if not np.isfinite(_close).all() or (_close <= 0).any():
    raise ValueError("Close prices must be finite and positive")

_current_close = _close.reindex(_idx)
_next_day_close = pd.Series(
    _close.reindex(_idx + pd.Timedelta(days=1)).to_numpy(),
    index=_idx,
)

_calendar_ret = _next_day_close / _current_close - 1
_next_available_ret = (
    _close.shift(-1) / _close - 1
).reindex(_idx)

_return_report = []

for _name, _candidate in [
    ("close -> next calendar day's close", _calendar_ret),
    ("close -> next available row's close", _next_available_ret),
]:
    _valid = np.isfinite(_saved_ret) & np.isfinite(_candidate)
    _matches = np.isclose(
        _saved_ret[_valid], _candidate[_valid],
        rtol=1e-6, atol=1e-10,
    )

    _return_report.append({
        "candidate_formula": _name,
        "compared_rows": int(_valid.sum()),
        "missing_candidate_rows": int((~np.isfinite(_candidate)).sum()),
        "match_pct": (
            100 * _matches.mean() if _valid.any() else np.nan
        ),
        "max_abs_error": (
            float((_saved_ret[_valid] - _candidate[_valid]).abs().max())
            if _valid.any() else np.nan
        ),
    })

LLM_RETURN_AUDIT = pd.DataFrame(_return_report)

print("\nSAVED FWD_RET_1 VERSUS RECOMPUTED RETURNS")
display(LLM_RETURN_AUDIT)

# 3. Inspect the final dates, including the missing September 26 bar.
_next_price_time = pd.Series(
    _close.index, index=_close.index
).shift(-1).reindex(_idx)

_elapsed_days = (
    _next_price_time - pd.Series(_idx, index=_idx)
).dt.total_seconds() / 86400

LLM_RETURN_TAIL_AUDIT = pd.DataFrame({
    "saved_fwd_ret_1": _saved_ret,
    "calendar_day_return": _calendar_ret,
    "next_available_return": _next_available_ret,
    "days_to_next_price": _elapsed_days,
}).tail(3)

print("\nFINAL THREE FEATURE DATES — DIAGNOSTIC ONLY")
display(LLM_RETURN_TAIL_AUDIT)

print("\nBlock 4 complete. No features, models, or saved returns changed.")
print("Matching a label does not establish when that label became available.")
print("Matching a return formula does not establish executable trade timing.")

PREV_LBL VERSUS PREVIOUS CALENDAR DAY'S Y


,split,compared_rows,match_pct,nonzero_comparisons,nonzero_match_pct
0,TRAIN,1878,100.0,1362,100.0
1,VAL,273,100.0,242,100.0
2,TEST,1091,100.0,878,100.0


Label values: [-1.0, 0.0, 1.0]

SAVED FWD_RET_1 VERSUS RECOMPUTED RETURNS


,candidate_formula,compared_rows,missing_candidate_rows,match_pct,max_abs_error
0,close -> next calendar day's close,3242,1,100.0,5.655199e-16
1,close -> next available row's close,3243,0,100.0,5.655199e-16



FINAL THREE FEATURE DATES — DIAGNOSTIC ONLY


,saved_fwd_ret_1,calendar_day_return,next_available_return,days_to_next_price
source_timestamp_utc,,,,
2026-09-23 00:00:00+00:00,0.000972,0.000972,0.000972,1.0
2026-09-24 00:00:00+00:00,0.001181,0.001181,0.001181,1.0
2026-09-25 00:00:00+00:00,0.004451,NaN,0.004451,2.0



Block 4 complete. No features, models, or saved returns changed.
Matching a label does not establish when that label became available.
Matching a return formula does not establish executable trade timing.


In [9]:
# BLOCK 5 — Prepare existing checkpoints for a PIPELINE PROTOTYPE.
# No training, package installation, or LLM calls.

from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import json
import os
import sys
import tempfile
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display

if not globals().get("LLM_TABLES_READY", False):
    raise RuntimeError("Run Block 3 successfully first.")

LLM_EXPERIMENT_MODE = "PIPELINE_PROTOTYPE"
LLM_CAUSALITY_VERIFIED = False
LLM_BASELINE_VALID_FOR_PERFORMANCE_CLAIMS = False

LLM_MODEL_DIR = LLM_EXPORT_DIR / "models"
LLM_WORK_DIR = Path("/kaggle/working/llm_memory_experiment")
LLM_CHECKPOINT_DIR = LLM_WORK_DIR / "checkpoints"
LLM_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LLM_MLP_PATH = LLM_MODEL_DIR / "mlp.pt"
if not LLM_MLP_PATH.is_file():
    raise FileNotFoundError(LLM_MLP_PATH)

# Preserve the trained model's complete feature order for faithful loading.
# Removing prev_lbl here would not repair its training.
LLM_MODEL_FEATURE_ORDER = list(LLM_SUMMARY["model_feature_order"])

with np.load(
    LLM_MODEL_DIR / "normalization.npz", allow_pickle=False
) as _npz:
    print("Normalization keys:", _npz.files)

    if not {"mu", "sd"}.issubset(_npz.files):
        raise ValueError("Expected mu and sd in normalization.npz")

    LLM_MU = np.asarray(_npz["mu"], dtype=np.float32).reshape(-1)
    LLM_SD = np.asarray(_npz["sd"], dtype=np.float32).reshape(-1)

_n = len(LLM_MODEL_FEATURE_ORDER)

if LLM_MU.shape != (_n,) or LLM_SD.shape != (_n,):
    raise ValueError("Normalization dimensions do not match model features")

if (
    not np.isfinite(LLM_MU).all()
    or not np.isfinite(LLM_SD).all()
    or (LLM_SD <= 0).any()
):
    raise ValueError("Invalid normalization values")

LLM_X_MODEL = (
    (
        LLM_TABLES["model_inputs"][LLM_MODEL_FEATURE_ORDER]
        .to_numpy(dtype=np.float32)
        - LLM_MU
    )
    / LLM_SD
)

if not np.isfinite(LLM_X_MODEL).all():
    raise ValueError("Normalized model inputs contain nonfinite values")

print("Normalized model input shape:", LLM_X_MODEL.shape)

# Recreate PPO ZIPs with their files at the archive root.
# Read ordinary JSON metadata without decoding serialized Python objects.
_folders = sorted(
    p for p in LLM_MODEL_DIR.glob("ppo_seed*") if p.is_dir()
)
if not _folders:
    raise FileNotFoundError("No unpacked ppo_seed* folders found")

_expected_seeds = (LLM_SETUP or {}).get("PPO_SEEDS")
if _expected_seeds is not None:
    _expected_names = {f"ppo_seed{int(s)}" for s in _expected_seeds}
    if {p.name for p in _folders} != _expected_names:
        raise ValueError("PPO folders differ from the configured seed list")

LLM_PPO_PATHS = []
LLM_PPO_METADATA = {}
_ppo_report = []

for _folder in _folders:
    for _required_file in [
        "data", "policy.pth", "pytorch_variables.pth",
        "_stable_baselines3_version",
    ]:
        if not (_folder / _required_file).is_file():
            raise FileNotFoundError(_folder / _required_file)

    _meta = json.loads(
        (_folder / "data").read_text(encoding="utf-8")
    )
    _saved_version = (
        _folder / "_stable_baselines3_version"
    ).read_text().strip()

    _destination = LLM_CHECKPOINT_DIR / f"{_folder.name}.zip"
    _fd, _tmp = tempfile.mkstemp(
        dir=LLM_CHECKPOINT_DIR, suffix=".tmp"
    )
    os.close(_fd)

    try:
        with zipfile.ZipFile(
            _tmp, "w", compression=zipfile.ZIP_STORED
        ) as _z:
            for _file in sorted(_folder.rglob("*")):
                if _file.is_file():
                    _z.write(
                        _file,
                        arcname=_file.relative_to(_folder).as_posix(),
                    )

        with zipfile.ZipFile(_tmp) as _z:
            if _z.testzip() is not None:
                raise ValueError(f"ZIP checksum failed: {_folder.name}")

        os.replace(_tmp, _destination)
    finally:
        if os.path.exists(_tmp):
            os.unlink(_tmp)

    LLM_PPO_PATHS.append(_destination)
    LLM_PPO_METADATA[_folder.name] = _meta
    _ppo_report.append({
        "checkpoint": _folder.name,
        "saved_SB3": _saved_version,
        "training_steps": _meta.get("num_timesteps"),
        "seed": _meta.get("seed"),
        "zip_KiB": round(_destination.stat().st_size / 1024, 1),
    })

print("\nPREPARED PPO CHECKPOINTS")
display(pd.DataFrame(_ppo_report))
print("Working checkpoint folder:", LLM_CHECKPOINT_DIR)

LLM_RUNTIME_VERSIONS = {}

print("\nCURRENT NOTEBOOK ENVIRONMENT")
print("Python:", sys.version.split()[0])

for _package in [
    "numpy", "pandas", "torch",
    "stable-baselines3", "gymnasium", "cloudpickle",
]:
    try:
        LLM_RUNTIME_VERSIONS[_package] = version(_package)
    except PackageNotFoundError:
        LLM_RUNTIME_VERSIONS[_package] = "NOT INSTALLED"

    print(f"{_package}: {LLM_RUNTIME_VERSIONS[_package]}")

_python_file = LLM_BUNDLE_DIR / "setup" / "python_version.txt"
if _python_file.is_file():
    print("\nSAVED TRAINING PYTHON:",
          _python_file.read_text().strip())

print("\nBlock 5 complete: checkpoints prepared; models not loaded yet.")
print("Mode:", LLM_EXPERIMENT_MODE)
print("Known future-label leakage remains in the existing trading model.")

Normalization keys: ['mu', 'sd']
Normalized model input shape: (3243, 54)

PREPARED PPO CHECKPOINTS


,checkpoint,saved_SB3,training_steps,seed,zip_KiB
0,ppo_seed0,2.9.0,200704,0,215.3
1,ppo_seed1,2.9.0,200704,1,215.3
2,ppo_seed2,2.9.0,200704,2,215.3


Working checkpoint folder: /kaggle/working/llm_memory_experiment/checkpoints

CURRENT NOTEBOOK ENVIRONMENT
Python: 3.12.13
numpy: 2.0.2
pandas: 2.3.3
torch: 2.10.0+cu128
stable-baselines3: NOT INSTALLED
gymnasium: 1.2.0
cloudpickle: 3.1.2

SAVED TRAINING PYTHON: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Block 5 complete: checkpoints prepared; models not loaded yet.
Mode: PIPELINE_PROTOTYPE
Known future-label leakage remains in the existing trading model.


In [10]:
# BLOCK 6 — Load frozen MLP/PPO models and verify inference.
# No training or LLM calls.

import sys
import subprocess
import importlib
from importlib.metadata import version, PackageNotFoundError
import numpy as np
import pandas as pd
from IPython.display import display

_required = [
    "LLM_X_MODEL", "LLM_MLP_PATH",
    "LLM_PPO_PATHS", "LLM_MODEL_FEATURE_ORDER",
]
if any(name not in globals() for name in _required):
    raise RuntimeError("Run Block 5 successfully first.")

LLM_MODELS_READY = False
LLM_MLP_MODEL = None
LLM_PPO_MODELS = []

# Install only SB3, preserving your existing CPU PyTorch.
_target_sb3 = "2.9.0"
_already_imported = sys.modules.get("stable_baselines3")

if (
    _already_imported is not None
    and _already_imported.__version__ != _target_sb3
):
    raise RuntimeError(
        "Another SB3 version is already imported. "
        "Restart the session and rerun Blocks 1–6."
    )

try:
    _installed_sb3 = version("stable-baselines3")
except PackageNotFoundError:
    _installed_sb3 = None

if _installed_sb3 != _target_sb3:
    print("Installing stable-baselines3==2.9.0 ...", flush=True)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--disable-pip-version-check", "--no-deps",
        "stable-baselines3==2.9.0",
    ])
    importlib.invalidate_caches()

import torch
from torch import nn
import stable_baselines3
from stable_baselines3 import PPO

if stable_baselines3.__version__ != _target_sb3:
    raise RuntimeError("Imported SB3 version does not match 2.9.0")

print(
    "SB3:", stable_baselines3.__version__,
    "| Torch:", torch.__version__,
    "| Device: CPU",
)

# Exact MLP architecture from your original notebook.
class LoadedMetaMLP(nn.Module):
    def __init__(self, n_in, h=128, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, h), nn.GELU(), nn.Dropout(p),
            nn.Linear(h, h), nn.GELU(), nn.Dropout(p),
            nn.Linear(h, 3),
        )

    def forward(self, x):
        return self.net(x)

_n_features = len(LLM_MODEL_FEATURE_ORDER)

_mlp = LoadedMetaMLP(
    _n_features,
    h=int(LLM_SUMMARY["mlp_hidden"]),
    p=float(LLM_SUMMARY["mlp_dropout"]),
)

_state = torch.load(
    LLM_MLP_PATH, map_location="cpu", weights_only=True
)
_mlp.load_state_dict(_state, strict=True)
_mlp.eval()
_mlp.requires_grad_(False)

with torch.inference_mode():
    _probs = torch.softmax(
        _mlp(torch.from_numpy(LLM_X_MODEL)), dim=-1
    ).cpu().numpy()

if (
    _probs.shape != (len(LLM_X_MODEL), 3)
    or not np.isfinite(_probs).all()
):
    raise ValueError("Invalid MLP probability output")

if not np.allclose(_probs.sum(axis=1), 1.0, atol=1e-6):
    raise ValueError("MLP probabilities do not sum to one")

# Verify the restored MLP against your exported probabilities.
_prob_columns = [
    "mlp_p_bear", "mlp_p_neutral", "mlp_p_bull"
]
_saved = LLM_TABLES["signals"][_prob_columns].to_numpy(dtype=float)

_max_error = float(np.max(np.abs(_probs - _saved)))
_matches = bool(np.allclose(
    _probs, _saved, rtol=2e-5, atol=2e-5
))

print("\nMLP RECONSTRUCTION CHECK")
print("Maximum probability difference:", _max_error)
print("Matches saved probabilities within tolerance:", _matches)

if not _matches:
    raise ValueError(
        "MLP predictions differ from the export. "
        "Inspect checkpoint/normalization before proceeding."
    )

# PPO expects 54 features + position + previous net return = 56.
# Smoke test only: one observation with position=0 and previous return=0.
_smoke_obs = np.concatenate([
    LLM_X_MODEL[0],
    np.array([0.0, 0.0], dtype=np.float32),
]).astype(np.float32)

_loaded_ppo, _ppo_report = [], []

for _path in LLM_PPO_PATHS:
    print(f"Loading {_path.name} ...", flush=True)
    _model = PPO.load(str(_path), device="cpu")

    if _model.observation_space.shape != (_n_features + 2,):
        raise ValueError(
            f"{_path.name}: unexpected observation shape "
            f"{_model.observation_space.shape}"
        )

    if _model.action_space.shape != (1,):
        raise ValueError(f"{_path.name}: expected one continuous action")

    if not (
        np.allclose(_model.action_space.low, -1)
        and np.allclose(_model.action_space.high, 1)
    ):
        raise ValueError(f"{_path.name}: unexpected action bounds")

    _model.policy.set_training_mode(False)
    _model.policy.requires_grad_(False)

    _action, _ = _model.predict(
        _smoke_obs, deterministic=True
    )
    _action = np.asarray(_action, dtype=float).reshape(-1)

    if (
        _action.shape != (1,)
        or not np.isfinite(_action).all()
        or abs(_action[0]) > 1.000001
    ):
        raise ValueError(f"{_path.name}: invalid test action")

    _loaded_ppo.append(_model)
    _ppo_report.append({
        "checkpoint": _path.stem,
        "observation_shape": str(_model.observation_space.shape),
        "action_shape": str(_model.action_space.shape),
        "smoke_action": float(_action[0]),
    })

LLM_MLP_MODEL = _mlp
LLM_PPO_MODELS = _loaded_ppo

LLM_MLP_PROBS = pd.DataFrame(
    _probs,
    index=LLM_TABLES["model_inputs"].index,
    columns=_prob_columns,
)
LLM_MLP_DIRECTION = pd.Series(
    _probs.argmax(axis=1) - 1,
    index=LLM_MLP_PROBS.index,
    name="mlp_direction",
)
LLM_MLP_CONFIDENCE = pd.Series(
    _probs.max(axis=1),
    index=LLM_MLP_PROBS.index,
    name="mlp_max_probability",
)

LLM_MODELS_READY = True

print("\nPPO LOAD / INFERENCE CHECK")
display(pd.DataFrame(_ppo_report))

print(
    f"\nBlock 6 complete: 1 MLP + "
    f"{len(LLM_PPO_MODELS)} PPO models loaded and frozen."
)
print("PPO smoke actions are a loading check, not a portfolio replay.")
print("Experiment mode:", LLM_EXPERIMENT_MODE)

Installing stable-baselines3==2.9.0 ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 7.6 MB/s eta 0:00:00
SB3: 2.9.0 | Torch: 2.10.0+cu128 | Device: CPU

MLP RECONSTRUCTION CHECK
Maximum probability difference: 4.205381775035377e-07
Matches saved probabilities within tolerance: True
Loading ppo_seed0.zip ...


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Loading ppo_seed1.zip ...
Loading ppo_seed2.zip ...

PPO LOAD / INFERENCE CHECK


,checkpoint,observation_shape,action_shape,smoke_action
0,ppo_seed0,"(56,)","(1,)",0.725989
1,ppo_seed1,"(56,)","(1,)",-0.739768
2,ppo_seed2,"(56,)","(1,)",-0.890201



Block 6 complete: 1 MLP + 3 PPO models loaded and frozen.
PPO smoke actions are a loading check, not a portfolio replay.
Experiment mode: PIPELINE_PROTOTYPE


In [11]:
# BLOCK 7 — Reproduce the original trading proposals.
# No training, portfolio backtest, or LLM calls.

import json
import time
import numpy as np
import pandas as pd
import torch
from IPython.display import display

if not globals().get("LLM_MODELS_READY", False):
    raise RuntimeError("Run Block 6 successfully first.")

LLM_PROPOSALS_READY = False

# 1. Check the saved ensemble weights.
_weights = {
    k: float(v)
    for k, v in LLM_SUMMARY["ensemble_weights"].items()
}

if any(
    abs(_weights.get(k, 0.0)) > 1e-12
    for k in ["Majority", "Mean"]
):
    raise ValueError(
        "This block expects the confirmed zero direct-sentiment weights."
    )

if not all(
    np.isfinite(_weights[k]) and _weights[k] >= 0
    for k in ["MLP", "PPO"]
):
    raise ValueError("Invalid ensemble weights.")

if not np.isclose(_weights["MLP"] + _weights["PPO"], 1.0):
    raise ValueError(
        "MLP/PPO weights must sum to one; do not silently renormalize."
    )

_idx = LLM_TABLES["model_inputs"].index
_n = len(_idx)
_seed_names = [p.stem for p in LLM_PPO_PATHS]

if (
    _n < 2
    or len(_seed_names) != len(LLM_PPO_MODELS)
    or not _seed_names
):
    raise ValueError("Invalid number of rows or checkpoints.")

_per_seed = np.zeros(
    (_n, len(LLM_PPO_MODELS)),
    dtype=np.float64,
)
_started = time.perf_counter()

# 2. Reproduce the original PPO inference loop.
# Each policy receives:
#   54 normalized features + its own previous proposal + zero return.
_old_threads = torch.get_num_threads()

try:
    torch.set_num_threads(1)

    with torch.inference_mode():
        for _j, (_name, _model) in enumerate(
            zip(_seed_names, LLM_PPO_MODELS)
        ):
            print(f"Reconstructing {_name} ...", flush=True)
            _previous_proposal = 0.0

            for _i in range(_n - 1):
                _obs = np.concatenate([
                    LLM_X_MODEL[_i],
                    np.array(
                        [_previous_proposal, 0.0],
                        dtype=np.float32,
                    ),
                ]).astype(np.float32)

                _action, _ = _model.predict(
                    _obs,
                    deterministic=True,
                )
                _values = np.asarray(_action).reshape(-1)

                if (
                    _values.size != 1
                    or not np.isfinite(_values[0])
                ):
                    raise ValueError(
                        f"Invalid action from {_name} at row {_i}"
                    )

                _previous_proposal = float(
                    np.clip(_values[0], -1.0, 1.0)
                )
                _per_seed[_i, _j] = _previous_proposal

                if (_i + 1) % 1000 == 0:
                    print(
                        f"  {_i + 1:,}/{_n - 1:,} rows",
                        flush=True,
                    )

            # Preserve the original notebook's final-row behavior.
            _per_seed[-1, _j] = _per_seed[-2, _j]
            print(f"  {_name} complete", flush=True)

finally:
    torch.set_num_threads(_old_threads)

# 3. Combine the three PPO policies and the MLP.
_ppo = _per_seed.mean(axis=1)

_mlp = (
    LLM_MLP_DIRECTION.reindex(_idx).to_numpy(dtype=float)
    * LLM_MLP_CONFIDENCE.reindex(_idx).to_numpy(dtype=float)
)

_ensemble = np.clip(
    _weights["MLP"] * _mlp + _weights["PPO"] * _ppo,
    -1.0,
    1.0,
)

# 4. Compare recomputed proposals with the exported proposals.
_checks = []

for _name, _actual, _saved_column in [
    ("PPO", _ppo, "ppo_proposal_saved"),
    ("Ensemble", _ensemble, "ensemble_proposal_saved"),
]:
    _saved = (
        LLM_TABLES["signals"][_saved_column]
        .reindex(_idx)
        .to_numpy(dtype=float)
    )

    _same = np.isclose(
        _actual,
        _saved,
        rtol=2e-5,
        atol=2e-5,
    )

    _checks.append({
        "component": _name,
        "match_pct": float(100 * _same.mean()),
        "max_abs_difference": float(
            np.max(np.abs(_actual - _saved))
        ),
        "all_within_tolerance": bool(_same.all()),
    })

LLM_PROPOSAL_RECONSTRUCTION = pd.DataFrame(_checks)

print("\nPROPOSAL RECONSTRUCTION CHECK")
display(LLM_PROPOSAL_RECONSTRUCTION)

if not all(c["all_within_tolerance"] for c in _checks):
    raise ValueError(
        "Proposals differ from the export. "
        "Inspect these differences before continuing."
    )

# 5. Store one proposal sequence shared by all reviewer conditions.
LLM_PROPOSAL_TAPE = LLM_MLP_PROBS.copy()
LLM_PROPOSAL_TAPE["mlp_direction"] = LLM_MLP_DIRECTION
LLM_PROPOSAL_TAPE["mlp_max_probability"] = LLM_MLP_CONFIDENCE
LLM_PROPOSAL_TAPE["mlp_signed_proposal"] = _mlp

for _j, _name in enumerate(_seed_names):
    LLM_PROPOSAL_TAPE[_name + "_proposal"] = _per_seed[:, _j]

LLM_PROPOSAL_TAPE["ppo_proposal"] = _ppo
LLM_PROPOSAL_TAPE["ensemble_proposal"] = _ensemble
LLM_PROPOSAL_TAPE.index.name = "source_timestamp_utc"


def get_trading_proposal(source_timestamp):
    """Look up an original proposal by its UTC source-row timestamp."""
    ts = pd.Timestamp(source_timestamp)
    ts = (
        ts.tz_localize("UTC")
        if ts.tzinfo is None
        else ts.tz_convert("UTC")
    )

    if ts not in LLM_PROPOSAL_TAPE.index:
        raise KeyError(f"No exported proposal for {ts}")

    row = LLM_PROPOSAL_TAPE.loc[ts]

    return {
        "source_timestamp_utc": ts.isoformat(),
        "ensemble_proposal": float(row["ensemble_proposal"]),
        "ppo_proposal": float(row["ppo_proposal"]),
        "ppo_seed_proposals": {
            name: float(row[name + "_proposal"])
            for name in _seed_names
        },
        "mlp_direction": int(row["mlp_direction"]),
        "mlp_max_probability": float(
            row["mlp_max_probability"]
        ),
        "mlp_signed_proposal": float(
            row["mlp_signed_proposal"]
        ),
    }


_output_path = LLM_WORK_DIR / "verified_model_proposals.csv"
LLM_PROPOSAL_TAPE.to_csv(_output_path)

LLM_PROPOSAL_INTERFACE_MODE = "FIXED_ORIGINAL_PROPOSALS"
LLM_PROPOSALS_READY = True

_example_ts = _idx[
    LLM_TABLES["splits"]["TEST"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
    .to_numpy()
][0]

print("\nEXAMPLE PROPOSAL — SOURCE ROW, NOT VERIFIED EXECUTION TIME")
print(json.dumps(get_trading_proposal(_example_ts), indent=2))

print(f"\nBlock 7 complete in {time.perf_counter() - _started:.1f}s.")
print("Saved:", _output_path)
print(
    "The reviewer will maintain its own portfolio state; "
    "this baseline proposal sequence stays fixed."
)
print(
    "Proposal reproduction is not a reproduction "
    "of the original Sharpe/backtest."
)

Reconstructing ppo_seed0 ...
  1,000/3,242 rows
  2,000/3,242 rows
  3,000/3,242 rows
  ppo_seed0 complete
Reconstructing ppo_seed1 ...
  1,000/3,242 rows
  2,000/3,242 rows
  3,000/3,242 rows
  ppo_seed1 complete
Reconstructing ppo_seed2 ...
  1,000/3,242 rows
  2,000/3,242 rows
  3,000/3,242 rows
  ppo_seed2 complete

PROPOSAL RECONSTRUCTION CHECK


,component,match_pct,max_abs_difference,all_within_tolerance
0,PPO,100.0,2.483527e-07,True
1,Ensemble,100.0,1.790128e-07,True



EXAMPLE PROPOSAL — SOURCE ROW, NOT VERIFIED EXECUTION TIME
{
  "source_timestamp_utc": "2023-10-01T00:00:00+00:00",
  "ensemble_proposal": 0.8605075840726089,
  "ppo_proposal": 1.0,
  "ppo_seed_proposals": {
    "ppo_seed0": 1.0,
    "ppo_seed1": 1.0,
    "ppo_seed2": 1.0
  },
  "mlp_direction": 1,
  "mlp_max_probability": 0.5003843307495117,
  "mlp_signed_proposal": 0.5003843307495117
}

Block 7 complete in 3.3s.
Saved: /kaggle/working/llm_memory_experiment/verified_model_proposals.csv
The reviewer will maintain its own portfolio state; this baseline proposal sequence stays fixed.
Proposal reproduction is not a reproduction of the original Sharpe/backtest.


In [12]:
# BLOCK 8 — Build market/model inputs and a reviewer-state function.
# No LLM calls, memory updates, or portfolio simulation yet.

import json
import numpy as np
import pandas as pd

if not globals().get("LLM_PROPOSALS_READY", False):
    raise RuntimeError("Run Block 7 successfully first.")

LLM_REVIEWER_INPUTS_READY = False

# Explicit allowlist: no sentiment, prev_lbl, y, or forward returns.
# Original vol_8 and vol_30 = rolling return standard deviation * 100.
# Convert them back to fractions: 0.02 means 2% daily volatility.
_market_spec = {
    "ret_1": ("return_1d_frac", 1.0),
    "ret_5": ("return_5d_frac", 1.0),
    "ret_10": ("return_10d_frac", 1.0),
    "rsi_14": ("rsi_14", 1.0),
    "macd_n": ("macd_histogram_over_close", 1.0),
    "vol_8": ("daily_volatility_8d_frac", 0.01),
    "vol_30": ("daily_volatility_30d_frac", 0.01),
}

_features = LLM_TABLES["features"]
_missing = set(_market_spec) - set(_features.columns)

if _missing:
    raise ValueError(
        f"Missing required market features: {sorted(_missing)}"
    )

if not _features.index.equals(LLM_PROPOSAL_TAPE.index):
    raise ValueError(
        "Market features and proposals have different timestamps."
    )

LLM_REVIEWER_MARKET = pd.DataFrame(index=_features.index)

for _source, (_target, _scale) in _market_spec.items():
    LLM_REVIEWER_MARKET[_target] = (
        pd.to_numeric(_features[_source], errors="raise") * _scale
    )

if not np.isfinite(
    LLM_REVIEWER_MARKET.to_numpy(dtype=float)
).all():
    raise ValueError(
        "Reviewer market features contain missing/infinite values."
    )

LLM_REVIEWER_SCHEMA = {
    "mode": LLM_EXPERIMENT_MODE,
    "market_fields": list(LLM_REVIEWER_MARKET.columns),
    "units": {
        "fraction": "0.01 means 1%; volatility is daily, not annualized",
        "rsi_14": "0 to 100",
        "macd_histogram_over_close": "MACD histogram divided by close",
        "position": "signed exposure; +0.5 means 50% long exposure",
        "drawdown_frac": "nonnegative fraction below peak equity",
    },
    "timestamp_status": (
        "source-row key; execution timing not yet verified"
    ),
    "omitted_market_fields": {
        "roc_10": "duplicates ret_10 after rescaling",
        "sigma": "zero values have unresolved availability meaning",
    },
    "direct_sentiment_fields": False,
    "base_models_unchanged": True,
}


def build_reviewer_state(source_timestamp, *, portfolio_state):
    """Combine one source row with this reviewer's PRE-decision account state.

    portfolio_state must come from the caller's own replay, using only
    outcomes already observed. This function cannot verify that provenance.
    """
    ts = pd.Timestamp(source_timestamp)
    ts = (
        ts.tz_localize("UTC")
        if ts.tzinfo is None
        else ts.tz_convert("UTC")
    )

    if ts not in LLM_REVIEWER_MARKET.index:
        raise KeyError(f"No reviewer market features for {ts}")

    required = (
        "current_position",
        "previous_portfolio_return",
        "equity",
        "peak_equity",
    )
    missing = set(required) - set(portfolio_state)

    if missing:
        raise ValueError(
            f"Missing portfolio state: {sorted(missing)}"
        )

    account = {
        key: float(portfolio_state[key])
        for key in required
    }

    if not all(np.isfinite(value) for value in account.values()):
        raise ValueError(
            "Portfolio state must contain finite numbers."
        )

    if (
        account["equity"] <= 0
        or account["peak_equity"] < account["equity"]
    ):
        raise ValueError("Require peak_equity >= equity > 0.")

    account["drawdown_frac"] = (
        1.0 - account["equity"] / account["peak_equity"]
    )

    market = {
        key: float(value)
        for key, value in LLM_REVIEWER_MARKET.loc[ts].items()
    }

    quality_notes = []
    for key in (
        "daily_volatility_8d_frac",
        "daily_volatility_30d_frac",
    ):
        if market[key] <= 0:
            # Mark unusable volatility; do not interpret it as low risk.
            market[key] = None
            quality_notes.append(
                f"{key}: nonpositive stored value treated as unavailable"
            )

    proposal = get_trading_proposal(ts)
    proposal.pop("source_timestamp_utc")

    return {
        "source_timestamp_utc": ts.isoformat(),
        "model": proposal,
        "market": market,
        "portfolio": account,
        "data_quality_notes": quality_notes,
    }


# Save the static inputs and their definitions.
_market_path = LLM_WORK_DIR / "reviewer_market_features.csv"
LLM_REVIEWER_MARKET.to_csv(_market_path)

(LLM_WORK_DIR / "reviewer_input_schema.json").write_text(
    json.dumps(LLM_REVIEWER_SCHEMA, indent=2),
    encoding="utf-8",
)

# Demonstration: start a fresh, flat account at the first TEST source row.
# Later replay decisions must pass their actual evolving account state.
_example_ts = LLM_TABLES["splits"].index[
    LLM_TABLES["splits"]["TEST"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
][0]

_initial_cash = float(LLM_SETUP["INIT_CASH"])

_example_state = build_reviewer_state(
    _example_ts,
    portfolio_state={
        "current_position": 0.0,
        "previous_portfolio_return": 0.0,
        "equity": _initial_cash,
        "peak_equity": _initial_cash,
    },
)

print("EXAMPLE REVIEWER STATE — DEMONSTRATION WITH A FRESH ACCOUNT")
print(json.dumps(_example_state, indent=2, allow_nan=False))

print(
    f"\nPrepared {len(LLM_REVIEWER_MARKET):,} rows "
    "and 7 market features."
)
print("Fraction units: 0.01 means 1%; daily volatility is not annualized.")
print("No direct sentiment fields or outcome labels in reviewer state.")
print("Existing base-model inputs remain unchanged.")
print("Saved:", _market_path)
print("Block 8 complete. Execution timing and replay are next.")

LLM_REVIEWER_INPUTS_READY = True

EXAMPLE REVIEWER STATE — DEMONSTRATION WITH A FRESH ACCOUNT
{
  "source_timestamp_utc": "2023-10-01T00:00:00+00:00",
  "model": {
    "ensemble_proposal": 0.8605075840726089,
    "ppo_proposal": 1.0,
    "ppo_seed_proposals": {
      "ppo_seed0": 1.0,
      "ppo_seed1": 1.0,
      "ppo_seed2": 1.0
    },
    "mlp_direction": 1,
    "mlp_max_probability": 0.5003843307495117,
    "mlp_signed_proposal": 0.5003843307495117
  },
  "market": {
    "return_1d_frac": 0.0374880260695311,
    "return_5d_frac": 0.0881082401242783,
    "return_10d_frac": 0.0943651813213446,
    "rsi_14": 71.71390757850875,
    "macd_histogram_over_close": 0.0092811905524306,
    "daily_volatility_8d_frac": 0.0164126082192689,
    "daily_volatility_30d_frac": 0.014996342852381911
  },
  "portfolio": {
    "current_position": 0.0,
    "previous_portfolio_return": 0.0,
    "equity": 100000.0,
    "peak_equity": 100000.0,
    "drawdown_frac": 0.0
  },
  "data_quality_notes": []
}

Prepared 3,243 rows and 7 market feat

In [13]:
# BLOCK 8 — Build market/model inputs and a reviewer-state function.
# No LLM calls, memory updates, or portfolio simulation yet.

import json
import numpy as np
import pandas as pd

if not globals().get("LLM_PROPOSALS_READY", False):
    raise RuntimeError("Run Block 7 successfully first.")

LLM_REVIEWER_INPUTS_READY = False

# Explicit allowlist: no sentiment, prev_lbl, y, or forward returns.
# Original vol_8 and vol_30 = rolling return standard deviation * 100.
# Convert them back to fractions: 0.02 means 2% daily volatility.
_market_spec = {
    "ret_1": ("return_1d_frac", 1.0),
    "ret_5": ("return_5d_frac", 1.0),
    "ret_10": ("return_10d_frac", 1.0),
    "rsi_14": ("rsi_14", 1.0),
    "macd_n": ("macd_histogram_over_close", 1.0),
    "vol_8": ("daily_volatility_8d_frac", 0.01),
    "vol_30": ("daily_volatility_30d_frac", 0.01),
}

_features = LLM_TABLES["features"]
_missing = set(_market_spec) - set(_features.columns)

if _missing:
    raise ValueError(
        f"Missing required market features: {sorted(_missing)}"
    )

if not _features.index.equals(LLM_PROPOSAL_TAPE.index):
    raise ValueError(
        "Market features and proposals have different timestamps."
    )

LLM_REVIEWER_MARKET = pd.DataFrame(index=_features.index)

for _source, (_target, _scale) in _market_spec.items():
    LLM_REVIEWER_MARKET[_target] = (
        pd.to_numeric(_features[_source], errors="raise") * _scale
    )

if not np.isfinite(
    LLM_REVIEWER_MARKET.to_numpy(dtype=float)
).all():
    raise ValueError(
        "Reviewer market features contain missing/infinite values."
    )

LLM_REVIEWER_SCHEMA = {
    "mode": LLM_EXPERIMENT_MODE,
    "market_fields": list(LLM_REVIEWER_MARKET.columns),
    "units": {
        "fraction": "0.01 means 1%; volatility is daily, not annualized",
        "rsi_14": "0 to 100",
        "macd_histogram_over_close": "MACD histogram divided by close",
        "position": "signed exposure; +0.5 means 50% long exposure",
        "drawdown_frac": "nonnegative fraction below peak equity",
    },
    "timestamp_status": (
        "source-row key; execution timing not yet verified"
    ),
    "omitted_market_fields": {
        "roc_10": "duplicates ret_10 after rescaling",
        "sigma": "zero values have unresolved availability meaning",
    },
    "direct_sentiment_fields": False,
    "base_models_unchanged": True,
}


def build_reviewer_state(source_timestamp, *, portfolio_state):
    """Combine one source row with this reviewer's PRE-decision account state.

    portfolio_state must come from the caller's own replay, using only
    outcomes already observed. This function cannot verify that provenance.
    """
    ts = pd.Timestamp(source_timestamp)
    ts = (
        ts.tz_localize("UTC")
        if ts.tzinfo is None
        else ts.tz_convert("UTC")
    )

    if ts not in LLM_REVIEWER_MARKET.index:
        raise KeyError(f"No reviewer market features for {ts}")

    required = (
        "current_position",
        "previous_portfolio_return",
        "equity",
        "peak_equity",
    )
    missing = set(required) - set(portfolio_state)

    if missing:
        raise ValueError(
            f"Missing portfolio state: {sorted(missing)}"
        )

    account = {
        key: float(portfolio_state[key])
        for key in required
    }

    if not all(np.isfinite(value) for value in account.values()):
        raise ValueError(
            "Portfolio state must contain finite numbers."
        )

    if (
        account["equity"] <= 0
        or account["peak_equity"] < account["equity"]
    ):
        raise ValueError("Require peak_equity >= equity > 0.")

    account["drawdown_frac"] = (
        1.0 - account["equity"] / account["peak_equity"]
    )

    market = {
        key: float(value)
        for key, value in LLM_REVIEWER_MARKET.loc[ts].items()
    }

    quality_notes = []
    for key in (
        "daily_volatility_8d_frac",
        "daily_volatility_30d_frac",
    ):
        if market[key] <= 0:
            # Mark unusable volatility; do not interpret it as low risk.
            market[key] = None
            quality_notes.append(
                f"{key}: nonpositive stored value treated as unavailable"
            )

    proposal = get_trading_proposal(ts)
    proposal.pop("source_timestamp_utc")

    return {
        "source_timestamp_utc": ts.isoformat(),
        "model": proposal,
        "market": market,
        "portfolio": account,
        "data_quality_notes": quality_notes,
    }


# Save the static inputs and their definitions.
_market_path = LLM_WORK_DIR / "reviewer_market_features.csv"
LLM_REVIEWER_MARKET.to_csv(_market_path)

(LLM_WORK_DIR / "reviewer_input_schema.json").write_text(
    json.dumps(LLM_REVIEWER_SCHEMA, indent=2),
    encoding="utf-8",
)

# Demonstration: start a fresh, flat account at the first TEST source row.
# Later replay decisions must pass their actual evolving account state.
_example_ts = LLM_TABLES["splits"].index[
    LLM_TABLES["splits"]["TEST"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
][0]

_initial_cash = float(LLM_SETUP["INIT_CASH"])

_example_state = build_reviewer_state(
    _example_ts,
    portfolio_state={
        "current_position": 0.0,
        "previous_portfolio_return": 0.0,
        "equity": _initial_cash,
        "peak_equity": _initial_cash,
    },
)

print("EXAMPLE REVIEWER STATE — DEMONSTRATION WITH A FRESH ACCOUNT")
print(json.dumps(_example_state, indent=2, allow_nan=False))

print(
    f"\nPrepared {len(LLM_REVIEWER_MARKET):,} rows "
    "and 7 market features."
)
print("Fraction units: 0.01 means 1%; daily volatility is not annualized.")
print("No direct sentiment fields or outcome labels in reviewer state.")
print("Existing base-model inputs remain unchanged.")
print("Saved:", _market_path)
print("Block 8 complete. Execution timing and replay are next.")

LLM_REVIEWER_INPUTS_READY = True

EXAMPLE REVIEWER STATE — DEMONSTRATION WITH A FRESH ACCOUNT
{
  "source_timestamp_utc": "2023-10-01T00:00:00+00:00",
  "model": {
    "ensemble_proposal": 0.8605075840726089,
    "ppo_proposal": 1.0,
    "ppo_seed_proposals": {
      "ppo_seed0": 1.0,
      "ppo_seed1": 1.0,
      "ppo_seed2": 1.0
    },
    "mlp_direction": 1,
    "mlp_max_probability": 0.5003843307495117,
    "mlp_signed_proposal": 0.5003843307495117
  },
  "market": {
    "return_1d_frac": 0.0374880260695311,
    "return_5d_frac": 0.0881082401242783,
    "return_10d_frac": 0.0943651813213446,
    "rsi_14": 71.71390757850875,
    "macd_histogram_over_close": 0.0092811905524306,
    "daily_volatility_8d_frac": 0.0164126082192689,
    "daily_volatility_30d_frac": 0.014996342852381911
  },
  "portfolio": {
    "current_position": 0.0,
    "previous_portfolio_return": 0.0,
    "equity": 100000.0,
    "peak_equity": 100000.0,
    "drawdown_frac": 0.0
  },
  "data_quality_notes": []
}

Prepared 3,243 rows and 7 market feat

In [14]:
# BLOCK 9 — Define the prototype replay clock and evaluator-only outcomes.
# No LLM calls, portfolio returns, memory updates, or period assignments.

import json
import numpy as np
import pandas as pd
from IPython.display import display

if not globals().get("LLM_REVIEWER_INPUTS_READY", False):
    raise RuntimeError("Run Block 8 successfully first.")

LLM_REPLAY_CLOCK_READY = False

LLM_REPLAY_PROTOCOL = {
    "mode": LLM_EXPERIMENT_MODE,
    "proposal_source": "fixed verified model proposals from Block 7",
    "information_rule": (
        "review source row D and own account state before acting"
    ),
    "fill_assumption": (
        "idealized fill at close[D] after observing row D features; "
        "zero processing latency assumed"
    ),
    "outcome_rule": "close[D + 1 calendar day] / close[D] - 1",
    "timestamp_meaning": (
        "source price-row keys, not verified execution timestamps"
    ),
    "causality_verified": False,
    "missing_price_rule": (
        "exclude decision; never substitute next available row"
    ),
    "fee_rate": float(LLM_SETUP["FEE"]),
    "slippage_rate": float(LLM_SETUP["SLIPPAGE"]),
    "cost_rule": (
        "linear fee plus slippage approximation on traded notional"
    ),
    "initial_cash": float(LLM_SETUP["INIT_CASH"]),
    "execution_scope": (
        "daily exposure replay to be implemented next; "
        "does not reproduce original TP/SL or VectorBT execution"
    ),
    "funding_and_borrow_costs": "not modeled in this prototype",
}

for _key in ("fee_rate", "slippage_rate", "initial_cash"):
    _value = LLM_REPLAY_PROTOCOL[_key]
    if not np.isfinite(_value) or _value < 0:
        raise ValueError(f"Invalid replay setting: {_key}={_value}")

if LLM_REPLAY_PROTOCOL["initial_cash"] <= 0:
    raise ValueError("Initial cash must be positive.")

_source_idx = LLM_REVIEWER_MARKET.index
_prices = LLM_TABLES["prices"]

if (
    _prices.index.has_duplicates
    or not _prices.index.is_monotonic_increasing
):
    raise ValueError("Price timestamps must be unique and sorted.")

_close = pd.to_numeric(_prices["close"], errors="raise")

if (
    not np.isfinite(_close.to_numpy(dtype=float)).all()
    or (_close <= 0).any()
):
    raise ValueError("Close prices must be finite and strictly positive.")

# Exact calendar dates: never use the next available price row.
_next_price_idx = _source_idx + pd.Timedelta(days=1)

_entry_close = _close.reindex(_source_idx)
if _entry_close.isna().any():
    raise ValueError("A source feature date has no entry close.")

_exit_close = pd.Series(
    _close.reindex(_next_price_idx).to_numpy(dtype=float),
    index=_source_idx,
)

# These future prices and returns belong ONLY to the evaluator.
LLM_EVALUATOR_DAILY = pd.DataFrame(index=_source_idx)
LLM_EVALUATOR_DAILY.index.name = "source_timestamp_utc"

LLM_EVALUATOR_DAILY["entry_price_row_utc"] = _source_idx
LLM_EVALUATOR_DAILY["exit_price_row_utc"] = _next_price_idx
LLM_EVALUATOR_DAILY["entry_close"] = _entry_close
LLM_EVALUATOR_DAILY["exit_close"] = _exit_close
LLM_EVALUATOR_DAILY["market_return_1d"] = (
    _exit_close / _entry_close - 1.0
)
LLM_EVALUATOR_DAILY["eligible"] = _exit_close.notna()

LLM_EVALUATOR_DAILY["exclusion_reason"] = np.where(
    LLM_EVALUATOR_DAILY["eligible"],
    "",
    "missing close on exact next calendar day",
)

LLM_REPLAY_SOURCE_TIMESTAMPS = _source_idx[
    LLM_EVALUATOR_DAILY["eligible"].to_numpy()
]

# Report availability within the original splits.
# These are not yet the memory-development / gate / evaluation periods.
_report = []

for _split in ("TRAIN", "VAL", "TEST"):
    _mask = (
        LLM_TABLES["splits"][_split]
        .reindex(_source_idx)
        .astype(str)
        .str.lower()
        .isin(["true", "1"])
    )

    _usable = _source_idx[
        (_mask & LLM_EVALUATOR_DAILY["eligible"]).to_numpy()
    ]

    _report.append({
        "original_split": _split,
        "source_rows": int(_mask.sum()),
        "eligible_daily_decisions": len(_usable),
        "excluded": int(_mask.sum()) - len(_usable),
        "first_source_date": (
            str(_usable.min().date()) if len(_usable) else None
        ),
        "last_source_date": (
            str(_usable.max().date()) if len(_usable) else None
        ),
    })

LLM_REPLAY_COVERAGE = pd.DataFrame(_report)

print("DAILY REPLAY COVERAGE — ORIGINAL SPLITS, NOT MEMORY PERIODS")
display(LLM_REPLAY_COVERAGE)

_excluded = LLM_EVALUATOR_DAILY.loc[
    ~LLM_EVALUATOR_DAILY["eligible"],
    ["exit_price_row_utc", "exclusion_reason"],
]

print("\nEXCLUDED SOURCE ROWS")
display(_excluded)

# Save evaluator data separately from reviewer inputs.
_evaluator_dir = LLM_WORK_DIR / "evaluator_only"
_evaluator_dir.mkdir(parents=True, exist_ok=True)

LLM_EVALUATOR_DAILY.to_csv(
    _evaluator_dir / "daily_replay_outcomes.csv"
)

(LLM_WORK_DIR / "replay_protocol.json").write_text(
    json.dumps(LLM_REPLAY_PROTOCOL, indent=2),
    encoding="utf-8",
)

_cost = (
    LLM_REPLAY_PROTOCOL["fee_rate"]
    + LLM_REPLAY_PROTOCOL["slippage_rate"]
)

print(f"\nFee + slippage: {_cost:.4%} of traded notional per trade.")
print("Evaluator-only prices/returns are not added to the reviewer state.")
print("Source tables preserved; exclusions apply only to this daily replay.")
print("Block 9 complete: prototype clock defined; live timing remains unverified.")
print("No performance calculated. Memory periods remain unassigned.")

LLM_REPLAY_CLOCK_READY = True

DAILY REPLAY COVERAGE — ORIGINAL SPLITS, NOT MEMORY PERIODS


,original_split,source_rows,eligible_daily_decisions,excluded,first_source_date,last_source_date
0,TRAIN,1879,1879,0,2017-11-09,2022-12-31
1,VAL,273,273,0,2023-01-01,2023-09-30
2,TEST,1091,1090,1,2023-10-01,2026-09-24



EXCLUDED SOURCE ROWS


,exit_price_row_utc,exclusion_reason
source_timestamp_utc,,
2026-09-25 00:00:00+00:00,2026-09-26 00:00:00+00:00,missing close on exact next calendar day



Fee + slippage: 0.1500% of traded notional per trade.
Evaluator-only prices/returns are not added to the reviewer state.
Source tables preserved; exclusions apply only to this daily replay.
Block 9 complete: prototype clock defined; live timing remains unverified.
No performance calculated. Memory periods remain unassigned.


In [15]:
# BLOCK 10 — Portfolio replay with KEEP / HALVE / FLAT actions.
# The three-day demonstration is scripted; no LLM or memory learning yet.

import json
import numpy as np
import pandas as pd
from IPython.display import display

if not globals().get("LLM_REPLAY_CLOCK_READY", False):
    raise RuntimeError("Run Block 9 successfully first.")

LLM_REPLAY_ACCOUNTING_READY = False


class LLMDailyReplay:
    ACTIONS = {"KEEP": 1.0, "HALVE": 0.5, "FLAT": 0.0}

    def __init__(self, source_timestamps, *, initial_cash=None):
        self.dates = pd.DatetimeIndex(
            pd.to_datetime(source_timestamps, utc=True)
        )

        if len(self.dates) == 0 or self.dates.has_duplicates:
            raise ValueError(
                "Provide a nonempty sequence of unique dates."
            )

        if not self.dates.is_monotonic_increasing:
            raise ValueError(
                "Replay dates must be in chronological order."
            )

        if len(self.dates) > 1 and not (
            self.dates[1:] - self.dates[:-1]
            == pd.Timedelta(days=1)
        ).all():
            raise ValueError(
                "One replay episode must contain consecutive daily decisions."
            )

        if not (
            LLM_EVALUATOR_DAILY["eligible"]
            .reindex(self.dates)
            .eq(True)
            .all()
        ):
            raise ValueError(
                "Replay includes a date without a valid daily outcome."
            )

        cash = float(
            LLM_REPLAY_PROTOCOL["initial_cash"]
            if initial_cash is None else initial_cash
        )

        if not np.isfinite(cash) or cash <= 0:
            raise ValueError(
                "Initial cash must be finite and positive."
            )

        self.cost_rate = (
            LLM_REPLAY_PROTOCOL["fee_rate"]
            + LLM_REPLAY_PROTOCOL["slippage_rate"]
        )

        # Each instance owns its own account.
        self._portfolio = {
            "current_position": 0.0,
            "previous_portfolio_return": 0.0,
            "equity": cash,
            "peak_equity": cash,
        }

        self._i = 0
        self.records = []

    @property
    def done(self):
        return self._i >= len(self.dates)

    def get_state(self):
        """Information available to the reviewer before this decision."""
        if self.done:
            raise RuntimeError(
                "Replay is complete; there is no next decision."
            )

        return build_reviewer_state(
            self.dates[self._i],
            portfolio_state=self._portfolio,
        )

    def step(self, action):
        if action not in self.ACTIONS:
            raise ValueError(
                f"Action must be one of {list(self.ACTIONS)}."
            )

        before = self.get_state()
        ts = self.dates[self._i]

        base = float(before["model"]["ensemble_proposal"])
        if not np.isfinite(base) or abs(base) > 1.0 + 1e-8:
            raise ValueError(
                "Invalid base-model exposure proposal."
            )

        target = base * self.ACTIONS[action]
        equity_before = self._portfolio["equity"]
        previous_exposure = self._portfolio["current_position"]

        # Target signed ETH notional uses PRE-trade equity.
        previous_notional = previous_exposure * equity_before
        target_notional = target * equity_before

        traded_notional = abs(
            target_notional - previous_notional
        )
        trade_cost = self.cost_rate * traded_notional

        # Future outcome is read only after the action is selected.
        outcome = LLM_EVALUATOR_DAILY.loc[ts]
        market_return = float(outcome["market_return_1d"])

        if (
            not np.isfinite(market_return)
            or market_return <= -1
        ):
            raise ValueError("Invalid evaluator return.")

        gross_pnl = target_notional * market_return
        marked_notional = (
            target_notional * (1.0 + market_return)
        )

        # Close remaining exposure at the end of each episode.
        terminal = self._i == len(self.dates) - 1
        closing_cost = (
            self.cost_rate * abs(marked_notional)
            if terminal else 0.0
        )

        total_cost = trade_cost + closing_cost
        equity_after = equity_before + gross_pnl - total_cost

        if (
            not np.isfinite(equity_after)
            or equity_after <= 0
        ):
            raise RuntimeError(
                f"Replay insolvent or invalid at {ts}; "
                "record this as a failure, not a skipped day."
            )

        net_return = equity_after / equity_before - 1.0
        peak = max(
            self._portfolio["peak_equity"],
            equity_after,
        )

        # Current exposure changes with both price and account equity.
        next_exposure = (
            0.0 if terminal else marked_notional / equity_after
        )
        drawdown = 1.0 - equity_after / peak

        record = {
            "source_timestamp_utc": ts.isoformat(),
            "outcome_price_row_utc": pd.Timestamp(
                outcome["exit_price_row_utc"]
            ).isoformat(),
            "action": action,
            "base_proposal": base,
            "previous_exposure": previous_exposure,
            "target_exposure": target,
            "market_return": market_return,
            "traded_notional": traded_notional,
            "trade_cost": trade_cost,
            "closing_cost": closing_cost,
            "total_cost": total_cost,
            "gross_pnl": gross_pnl,
            "net_return": net_return,
            "equity_before": equity_before,
            "equity_after": equity_after,
            "next_exposure": next_exposure,
            "drawdown_frac": drawdown,
            "terminal": terminal,
            "state_before": before,
        }

        # This becomes the portfolio input for the following decision.
        self._portfolio = {
            "current_position": next_exposure,
            "previous_portfolio_return": net_return,
            "equity": equity_after,
            "peak_equity": peak,
        }

        self.records.append(record)
        self._i += 1
        return record


LLM_REPLAY_PROTOCOL.update({
    "execution_scope": (
        "daily signed exposure replay; "
        "no intraday TP/SL or margin liquidation"
    ),
    "position_sizing": (
        "target notional = action-adjusted proposal * pre-trade equity"
    ),
    "position_marking": (
        "current exposure = marked signed notional / current equity"
    ),
    "terminal_rule": (
        "close remaining exposure at final outcome price and charge costs"
    ),
    "cash_interest_rate": 0.0,
})

(LLM_WORK_DIR / "replay_protocol.json").write_text(
    json.dumps(LLM_REPLAY_PROTOCOL, indent=2),
    encoding="utf-8",
)

# Demonstrate the mechanics on three consecutive TEST source rows.
_test_mask = (
    LLM_TABLES["splits"]["TEST"]
    .reindex(LLM_EVALUATOR_DAILY.index)
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
)

_demo_dates = LLM_EVALUATOR_DAILY.index[
    (_test_mask & LLM_EVALUATOR_DAILY["eligible"]).to_numpy()
][:3]

if len(_demo_dates) != 3:
    raise ValueError(
        "Need three eligible TEST dates for the demonstration."
    )

LLM_REPLAY_DEMO = LLMDailyReplay(_demo_dates)

# These actions are deliberately scripted.
for _action in ("KEEP", "HALVE", "FLAT"):
    LLM_REPLAY_DEMO.step(_action)

LLM_REPLAY_DEMO_LEDGER = pd.DataFrame([
    {
        key: value
        for key, value in record.items()
        if key != "state_before"
    }
    for record in LLM_REPLAY_DEMO.records
])

print(
    "SCRIPTED THREE-DAY REPLAY "
    "— THESE ACTIONS WERE NOT CHOSEN BY AN LLM"
)

display(LLM_REPLAY_DEMO_LEDGER[[
    "source_timestamp_utc",
    "action",
    "base_proposal",
    "target_exposure",
    "total_cost",
    "net_return",
    "equity_after",
    "next_exposure",
    "drawdown_frac",
]])

_demo_path = (
    LLM_WORK_DIR
    / "evaluator_only"
    / "scripted_replay_demo.jsonl"
)

with _demo_path.open("w", encoding="utf-8") as _handle:
    for _record in LLM_REPLAY_DEMO.records:
        _handle.write(
            json.dumps(_record, allow_nan=False) + "\n"
        )

print("\nFinished:", LLM_REPLAY_DEMO.done)
print("Saved:", _demo_path)
print(
    "Each reviewer condition will get a separate "
    "LLMDailyReplay account."
)
print(
    "Block 10 complete. No LLM calls or memory updates; "
    "no full-period score calculated."
)

LLM_REPLAY_ACCOUNTING_READY = True

SCRIPTED THREE-DAY REPLAY — THESE ACTIONS WERE NOT CHOSEN BY AN LLM


,source_timestamp_utc,action,base_proposal,target_exposure,total_cost,net_return,equity_after,next_exposure,drawdown_frac
0,2023-10-01T00:00:00+00:00,KEEP,0.860508,0.860508,129.076138,-0.036123,96387.677737,0.856619,0.036123
1,2023-10-02T00:00:00+00:00,HALVE,-0.761648,-0.380824,178.911380,-0.000267,96361.934248,-0.379336,0.036381
2,2023-10-03T00:00:00+00:00,FLAT,-0.720801,-0.000000,54.830360,-0.000569,96307.103888,0.000000,0.036929



Finished: True
Saved: /kaggle/working/llm_memory_experiment/evaluator_only/scripted_replay_demo.jsonl
Each reviewer condition will get a separate LLMDailyReplay account.
Block 10 complete. No LLM calls or memory updates; no full-period score calculated.


In [16]:
# BLOCK 11 — Assign chronological periods for the LLM memory experiment.
# No model training, LLM calls, or performance scoring.

import json
import pandas as pd
from IPython.display import display

if not globals().get("LLM_REPLAY_ACCOUNTING_READY", False):
    raise RuntimeError("Run Block 10 successfully first.")

LLM_MEMORY_PERIODS_READY = False

# Inclusive decision-date ranges.
LLM_STAGE_PLAN = {
    "MEMORY_BUILD": ("2023-10-01", "2024-06-29"),
    "MEMORY_GATE": ("2024-07-01", "2024-12-30"),
    "FROZEN_EVAL": ("2025-01-01", "2026-09-24"),
}

# These dates separate stages. Their prices can settle the preceding
# stage's final decision, but no new decisions are made on these dates.
LLM_BOUNDARY_BUFFER_DATES = pd.to_datetime(
    ["2024-06-30", "2024-12-31"], utc=True
)

_idx = LLM_EVALUATOR_DAILY.index
_splits = LLM_TABLES["splits"].reindex(_idx)

_is_test = (
    _splits["TEST"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
)
_eligible = LLM_EVALUATOR_DAILY["eligible"].astype(bool)

# Separate assignment: original split table remains unchanged.
LLM_MEMORY_ASSIGNMENT = pd.Series(
    "BASE_MODEL_PERIOD",
    index=_idx,
    name="memory_experiment_stage",
)

LLM_MEMORY_ASSIGNMENT.loc[
    _is_test & ~_eligible
] = "NO_DAILY_OUTCOME"

LLM_MEMORY_ASSIGNMENT.loc[
    _is_test & _eligible
] = "UNASSIGNED"

LLM_STAGE_DATES = {}
_summary = []

_purposes = {
    "MEMORY_BUILD": "Generate candidate lessons from replay feedback",
    "MEMORY_GATE": "Validate/select memories; no new lessons here",
    "FROZEN_EVAL": "Evaluate fixed reviewers and frozen memories",
}

for _stage, (_start, _end) in LLM_STAGE_PLAN.items():
    _dates = pd.date_range(_start, _end, freq="D", tz="UTC")

    if not _dates.isin(_idx).all():
        raise ValueError(f"{_stage}: a planned source date is missing.")

    if not (_is_test.loc[_dates] & _eligible.loc[_dates]).all():
        raise ValueError(
            f"{_stage}: dates must be eligible original TEST rows."
        )

    if not LLM_MEMORY_ASSIGNMENT.loc[_dates].eq("UNASSIGNED").all():
        raise ValueError(f"{_stage}: overlapping or invalid assignment.")

    LLM_MEMORY_ASSIGNMENT.loc[_dates] = _stage
    LLM_STAGE_DATES[_stage] = _dates

    _last_outcome = pd.Timestamp(
        LLM_EVALUATOR_DAILY.loc[
            _dates, "exit_price_row_utc"
        ].max()
    )

    _summary.append({
        "stage": _stage,
        "decisions": len(_dates),
        "first_source_date": str(_dates.min().date()),
        "last_source_date": str(_dates.max().date()),
        "last_outcome_price_row": str(_last_outcome.date()),
        "purpose": _purposes[_stage],
    })

# Validate and assign the boundary buffers.
if not LLM_BOUNDARY_BUFFER_DATES.isin(_idx).all():
    raise ValueError("A planned boundary buffer date is missing.")

if not LLM_MEMORY_ASSIGNMENT.loc[
    LLM_BOUNDARY_BUFFER_DATES
].eq("UNASSIGNED").all():
    raise ValueError(
        "Boundary buffers overlap a period or an ineligible row."
    )

LLM_MEMORY_ASSIGNMENT.loc[
    LLM_BOUNDARY_BUFFER_DATES
] = "BOUNDARY_BUFFER"

if LLM_MEMORY_ASSIGNMENT.eq("UNASSIGNED").any():
    raise ValueError("Some eligible TEST rows were not assigned.")

# Check when each stage's final outcome becomes available.
for _earlier, _later in [
    ("MEMORY_BUILD", "MEMORY_GATE"),
    ("MEMORY_GATE", "FROZEN_EVAL"),
]:
    _last_outcome = pd.Timestamp(
        LLM_EVALUATOR_DAILY.loc[
            LLM_STAGE_DATES[_earlier],
            "exit_price_row_utc",
        ].max()
    )

    if _last_outcome >= LLM_STAGE_DATES[_later].min():
        raise ValueError(
            f"{_earlier} outcomes reach into {_later}."
        )

# Variables used by the upcoming experiment blocks.
LLM_MEMORY_BUILD_DATES = LLM_STAGE_DATES["MEMORY_BUILD"]
LLM_MEMORY_GATE_DATES = LLM_STAGE_DATES["MEMORY_GATE"]
LLM_FROZEN_EVAL_DATES = LLM_STAGE_DATES["FROZEN_EVAL"]
LLM_MEMORY_PERIOD_SUMMARY = pd.DataFrame(_summary)

LLM_MEMORY_PERIOD_PROTOCOL = {
    "mode": LLM_EXPERIMENT_MODE,
    "date_ranges_inclusive": LLM_STAGE_PLAN,
    "boundary_buffer_source_dates": [
        str(d.date()) for d in LLM_BOUNDARY_BUFFER_DATES
    ],
    "candidate_generation": "MEMORY_BUILD only",
    "gate_policy": (
        "Freeze candidate text before MEMORY_GATE; "
        "use gate scores only for selection; "
        "predeclare selection rule and candidate budget before gate runs."
    ),
    "evaluation_policy": (
        "Freeze prompts, memory contents/retrieval rules, model choice, "
        "and settings before FROZEN_EVAL; "
        "portfolio state still evolves chronologically."
    ),
    "account_policy": (
        "Fresh independent account per condition and stage; "
        "close at stage end."
    ),
    "buffer_policy": (
        "No decisions, reward scoring, or memory updates on buffer "
        "source rows; their close can settle the preceding stage's "
        "last decision."
    ),
    "claims_scope": (
        "Held out from this LLM memory selection; exploratory prototype "
        "because the underlying baseline has known future-label leakage "
        "and prior TEST inspection."
    ),
}

_assignment_path = LLM_WORK_DIR / "memory_experiment_periods.csv"
LLM_MEMORY_ASSIGNMENT.to_csv(_assignment_path)

(LLM_WORK_DIR / "memory_period_protocol.json").write_text(
    json.dumps(LLM_MEMORY_PERIOD_PROTOCOL, indent=2),
    encoding="utf-8",
)

print("LLM MEMORY EXPERIMENT PERIODS")
display(LLM_MEMORY_PERIOD_SUMMARY)

print("\nALL SOURCE ROWS ACCOUNTED FOR")
display(
    LLM_MEMORY_ASSIGNMENT
    .value_counts()
    .rename("rows")
    .to_frame()
)

print("\nBoundary checks passed.")
print("No memory updates are allowed during FROZEN_EVAL.")
print("Original tables unchanged. No LLM calls or scoring performed.")
print("Saved:", _assignment_path)

LLM_MEMORY_PERIODS_READY = True
print("\nBlock 11 complete. You can now run Block 12.")

LLM MEMORY EXPERIMENT PERIODS


,stage,decisions,first_source_date,last_source_date,last_outcome_price_row,purpose
0,MEMORY_BUILD,273,2023-10-01,2024-06-29,2024-06-30,Generate candidate lessons from replay feedback
1,MEMORY_GATE,183,2024-07-01,2024-12-30,2024-12-31,Validate/select memories; no new lessons here
2,FROZEN_EVAL,632,2025-01-01,2026-09-24,2026-09-25,Evaluate fixed reviewers and frozen memories



ALL SOURCE ROWS ACCOUNTED FOR


,rows
memory_experiment_stage,
BASE_MODEL_PERIOD,2152
FROZEN_EVAL,632
MEMORY_BUILD,273
MEMORY_GATE,183
BOUNDARY_BUFFER,2
NO_DAILY_OUTCOME,1



Boundary checks passed.
No memory updates are allowed during FROZEN_EVAL.
Original tables unchanged. No LLM calls or scoring performed.
Saved: /kaggle/working/llm_memory_experiment/memory_experiment_periods.csv

Block 11 complete. You can now run Block 12.


In [17]:
# BLOCK 12 — DeepSeek Flash setup for parallel independent replays.
# Checks the model catalog; requests no LLM completion.

import json
import os
import threading
import time
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

if not globals().get("LLM_MEMORY_PERIODS_READY", False):
    raise RuntimeError("Run Block 11 successfully first.")

LLM_API_SETUP_READY = False

LLM_DEEPSEEK_API_KEY = os.environ.get(
    "DEEPSEEK_API_KEY", ""
).strip()

if not LLM_DEEPSEEK_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient

        LLM_DEEPSEEK_API_KEY = (
            UserSecretsClient()
            .get_secret("DEEPSEEK_API_KEY")
            .strip()
        )
    except Exception:
        raise RuntimeError(
            "Add a Kaggle secret named DEEPSEEK_API_KEY "
            "and grant this notebook access. Then rerun this cell. "
            "Do not put the key in chat or notebook output."
        ) from None

if not LLM_DEEPSEEK_API_KEY:
    raise RuntimeError("DEEPSEEK_API_KEY is empty.")

# Configuration for the request/replay runner added next.
# The budget target is not yet an enforced billing cap.
LLM_API_CONFIG = {
    "provider": "deepseek",
    "base_url": "https://api.deepseek.com",
    "model": "deepseek-flash",
    "run_id": "eth_memory_flash_v1",
    "thinking": {"type": "disabled"},
    "temperature": 0.0,
    "response_format": {"type": "json_object"},
    "decision_max_tokens": 512,
    "reflection_max_tokens": 768,
    "max_parallel_requests": 6,
    "max_attempts": 3,
    "request_timeout_seconds": 60,
    "budget_target_usd": 10.0,
    "price_reference_date": "2026-09-29",
    "peak_input_usd_per_million_uncached": 0.30,
    "peak_input_usd_per_million_cached": 0.006,
    "peak_output_usd_per_million": 1.20,
}


def check_deepseek_model_access():
    request = Request(
        LLM_API_CONFIG["base_url"] + "/models",
        headers={
            "Authorization": "Bearer " + LLM_DEEPSEEK_API_KEY,
            "Accept": "application/json",
        },
        method="GET",
    )

    for attempt in range(3):
        try:
            with urlopen(request, timeout=30) as response:
                payload = json.loads(response.read())

            return {
                item["id"]
                for item in payload.get("data", [])
            }

        except HTTPError as error:
            if (
                error.code in (429, 500, 502, 503, 504)
                and attempt < 2
            ):
                time.sleep(2 ** attempt)
                continue

            raise RuntimeError(
                "DeepSeek model-list request failed with "
                f"HTTP {error.code}. "
                "Check API access/key or retry later."
            ) from None

        except (URLError, TimeoutError):
            if attempt < 2:
                time.sleep(2 ** attempt)
                continue

            raise RuntimeError(
                "Could not reach DeepSeek. "
                "Check Kaggle internet access and retry."
            ) from None


_available_models = check_deepseek_model_access()

if LLM_API_CONFIG["model"] not in _available_models:
    raise RuntimeError(
        "Requested model is not in the returned catalog: "
        f"{sorted(_available_models)}"
    )

# The upcoming request runner will acquire one slot per API call.
LLM_API_REQUEST_SLOTS = threading.BoundedSemaphore(
    LLM_API_CONFIG["max_parallel_requests"]
)

LLM_API_RUN_DIR = (
    LLM_WORK_DIR
    / "llm_runs"
    / LLM_API_CONFIG["run_id"]
)
LLM_API_CACHE_DIR = LLM_API_RUN_DIR / "responses"
LLM_API_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Save only nonsecret settings. The key stays in runtime memory.
(LLM_API_RUN_DIR / "api_config.json").write_text(
    json.dumps(LLM_API_CONFIG, indent=2),
    encoding="utf-8",
)

print("DeepSeek API access: model catalog checked successfully")
print("Model:", LLM_API_CONFIG["model"])
print("Thinking mode: disabled")
print(
    "Configured maximum concurrent requests:",
    LLM_API_CONFIG["max_parallel_requests"],
)
print(
    "Sequential inside each account; "
    "independent accounts may run in parallel."
)
print("Run folder:", LLM_API_RUN_DIR)
print(
    "Budget target: $10; "
    "the request runner must enforce the spending guard."
)
print(
    "Block 12 complete. "
    "No LLM decisions or memory updates requested."
)

LLM_API_SETUP_READY = True

DeepSeek API access: model catalog checked successfully
Model: deepseek-flash
Thinking mode: disabled
Configured maximum concurrent requests: 6
Sequential inside each account; independent accounts may run in parallel.
Run folder: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1
Budget target: $10; the request runner must enforce the spending guard.
Block 12 complete. No LLM decisions or memory updates requested.


In [18]:
# BLOCK 12B — Checkpoint storage and downloadable backup.
# Run this while experiment workers are stopped.
# This cell does not enable automatic cloud backups.

import hashlib
import json
import os
import threading
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import FileLink, display

if "LLM_WORK_DIR" not in globals():
    raise RuntimeError("Run the earlier setup cells first.")

LLM_STORAGE_ROOT = Path(LLM_WORK_DIR).resolve()

LLM_STORAGE_RUN_DIR = Path(globals().get(
    "LLM_API_RUN_DIR",
    LLM_STORAGE_ROOT / "llm_runs" / "eth_memory_flash_v1"
)).resolve()

if not LLM_STORAGE_RUN_DIR.is_relative_to(LLM_STORAGE_ROOT):
    raise ValueError("Run folder must be inside the experiment folder.")

LLM_MEMORY_DIR = LLM_STORAGE_RUN_DIR / "memory"
LLM_RUN_CHECKPOINT_DIR = LLM_STORAGE_RUN_DIR / "checkpoints"

for _folder in [
    LLM_STORAGE_RUN_DIR / "responses",
    LLM_MEMORY_DIR / "candidates",
    LLM_MEMORY_DIR / "versions",
    LLM_RUN_CHECKPOINT_DIR,
    LLM_STORAGE_RUN_DIR / "events",
]:
    _folder.mkdir(parents=True, exist_ok=True)

# Reuse the lock when rerunning this cell.
if "LLM_DISK_LOCK" not in globals():
    LLM_DISK_LOCK = threading.RLock()


def llm_atomic_write_json(path, payload):
    """Save a JSON file without leaving a partially written final file."""
    path = Path(path).resolve()

    if not path.is_relative_to(LLM_STORAGE_ROOT):
        raise ValueError("Checkpoint must be inside the experiment folder.")

    text = json.dumps(payload, indent=2, allow_nan=False)

    with LLM_DISK_LOCK:
        path.parent.mkdir(parents=True, exist_ok=True)

        temporary = path.with_name(
            path.name + "." + uuid.uuid4().hex + ".tmp"
        )

        try:
            with temporary.open("x", encoding="utf-8") as handle:
                handle.write(text)
                handle.flush()
                os.fsync(handle.fileno())

            os.replace(temporary, path)

            directory_fd = os.open(path.parent, os.O_RDONLY)
            try:
                os.fsync(directory_fd)
            finally:
                os.close(directory_fd)
        finally:
            temporary.unlink(missing_ok=True)


def backup_llm_experiment():
    """
    Back up saved experiment files.
    Does not capture Python variables or notebook code.
    """
    stamp = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )

    # Outside the source folder, so backups do not include previous ZIPs.
    destination = (
        LLM_STORAGE_ROOT.parent / f"eth_llm_backup_{stamp}.zip"
    )
    temporary = destination.with_suffix(".zip.partial")

    manifest = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "root_directory": LLM_STORAGE_ROOT.name,
        "notebook_code_included": False,
        "original_input_bundle_included": False,
        "files": [],
    }

    try:
        with LLM_DISK_LOCK:
            with zipfile.ZipFile(
                temporary, "w", zipfile.ZIP_DEFLATED
            ) as archive:

                for path in sorted(LLM_STORAGE_ROOT.rglob("*")):
                    relative = path.relative_to(LLM_STORAGE_ROOT)

                    if (
                        not path.is_file()
                        or path.is_symlink()
                        or any(
                            part.startswith(".")
                            for part in relative.parts
                        )
                        or path.name.endswith((".tmp", ".partial"))
                    ):
                        continue

                    contents = path.read_bytes()
                    name = (
                        Path(LLM_STORAGE_ROOT.name) / relative
                    ).as_posix()

                    archive.writestr(name, contents)

                    manifest["files"].append({
                        "path": name,
                        "bytes": len(contents),
                        "sha256": hashlib.sha256(
                            contents
                        ).hexdigest(),
                    })

                archive.writestr(
                    "BACKUP_MANIFEST.json",
                    json.dumps(manifest, indent=2),
                )

            # Check the ZIP before publishing the completed backup.
            with zipfile.ZipFile(temporary) as archive:
                damaged = archive.testzip()
                if damaged:
                    raise RuntimeError(
                        f"Backup verification failed: {damaged}"
                    )

            os.replace(temporary, destination)

    finally:
        temporary.unlink(missing_ok=True)

    return destination, len(manifest["files"])


_backup, _file_count = backup_llm_experiment()
LLM_LOCAL_CHECKPOINT_HELPERS_READY = True

print("Memory folder:", LLM_MEMORY_DIR)
print("Replay checkpoint folder:", LLM_RUN_CHECKPOINT_DIR)
print(
    f"Backup created: {_file_count} files, "
    f"{_backup.stat().st_size / 2**20:.2f} MiB"
)
print("Download:", _backup)

display(FileLink(os.path.relpath(_backup, Path.cwd())))

print("\nDownload the ZIP to your computer.")
print("A ZIP left in Kaggle is still on the session's disk.")
print("Save the notebook itself separately.")
print("Automatic response saving/resumption will be added next.")
print("\nBlock 12B complete.")

Memory folder: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/memory
Replay checkpoint folder: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/checkpoints
Backup created: 12 files, 0.92 MiB
Download: /kaggle/working/eth_llm_backup_20260929T115649_273856Z.zip


/kaggle/working/eth_llm_backup_20260929T115649_273856Z.zip


Download the ZIP to your computer.
A ZIP left in Kaggle is still on the session's disk.
Save the notebook itself separately.
Automatic response saving/resumption will be added next.

Block 12B complete.


In [19]:
# BLOCK 13 — Resumable DeepSeek JSON requests. No API calls in this cell.
# One Python kernel per run folder. Independent requests may use threads.
# Local checkpoints must still be backed up outside the Kaggle runtime.

import copy
import hashlib
import json
import math
import random
import threading
import time
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
from http.client import HTTPException

if not globals().get("LLM_API_SETUP_READY", False):
    raise RuntimeError("Run Block 12 successfully first.")
if not globals().get("LLM_LOCAL_CHECKPOINT_HELPERS_READY", False):
    raise RuntimeError("Run Block 12B successfully first.")
if globals().get("LLM_REQUEST_CLIENT_READY", False):
    if LLM_REQUEST_CLIENT.active:
        raise RuntimeError("Stop active request workers before rerunning Block 13.")
LLM_REQUEST_CLIENT_READY = False


def _llm_canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False, allow_nan=False)


def _llm_digest(value):
    return hashlib.sha256(_llm_canonical(value).encode("utf-8")).hexdigest()


def _llm_now():
    return datetime.now(timezone.utc).isoformat()


def _llm_strict_json(text):
    def reject(value):
        raise ValueError("Non-finite JSON value")
    return json.loads(text, parse_constant=reject)


class LLMRequestClient:
    VERSION = "json-client-v1"
    RETRYABLE_HTTP = {429, 500, 502, 503, 504}

    def __init__(self, config, api_key, run_dir, disk_lock, slots, writer):
        self.config = copy.deepcopy(config)
        self.api_key = api_key
        self.run_dir = Path(run_dir).resolve()
        self.cache_dir = self.run_dir / "responses"
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.disk_lock, self.slots, self.writer = disk_lock, slots, writer
        self.records, self.key_locks = {}, {}
        self.active = 0

        if self.config["base_url"].rstrip("/") != "https://api.deepseek.com":
            raise ValueError("This client is configured for the direct DeepSeek API.")
        if self.config["thinking"] != {"type": "disabled"}:
            raise ValueError("This experiment currently uses thinking disabled.")
        if self.config["response_format"] != {"type": "json_object"}:
            raise ValueError("This client requires JSON output mode.")
        for field in ["budget_target_usd", "request_timeout_seconds",
                      "peak_input_usd_per_million_uncached",
                      "peak_input_usd_per_million_cached",
                      "peak_output_usd_per_million"]:
            value = float(self.config[field])
            if not math.isfinite(value) or value <= 0:
                raise ValueError(f"Invalid positive setting: {field}")
        if int(self.config["max_attempts"]) < 1:
            raise ValueError("max_attempts must be positive.")

        # Freeze settings for this run. Never write the API key to disk.
        manifest = {"client_version": self.VERSION, "config": self.config}
        manifest_path = self.run_dir / "request_client_config.json"
        with self.disk_lock:
            if manifest_path.exists():
                if _llm_strict_json(manifest_path.read_text()) != manifest:
                    raise RuntimeError(
                        "Saved request settings differ. Use a new run_id for a "
                        "different experiment; do not overwrite the existing run."
                    )
            else:
                self.writer(manifest_path, manifest)
            self.config_hash = _llm_digest(manifest)
            for path in sorted(self.cache_dir.glob("*.json")):
                record = _llm_strict_json(path.read_text())
                expected = _llm_digest({
                    "config_hash": self.config_hash,
                    "purpose": record["purpose"],
                    "payload": record["payload"],
                })
                if record["request_id"] != path.stem or expected != path.stem:
                    raise RuntimeError(f"Cache identity mismatch: {path.name}")
                for attempt in record["attempts"]:
                    charge = attempt["accounted_usd"]
                    if not math.isfinite(charge) or charge < 0:
                        raise RuntimeError(f"Invalid cost record: {path.name}")
                self.records[path.stem] = record

    def _save(self, record):
        # Caller holds disk_lock. Update RAM only after the atomic disk write.
        self.writer(self.cache_dir / (record["request_id"] + ".json"), record)
        self.records[record["request_id"]] = copy.deepcopy(record)

    def status(self):
        with self.disk_lock:
            attempts = [a for r in self.records.values() for a in r["attempts"]]
            return {
                "active_requests": self.active,
                "saved_responses": sum(a["status"] == "response" for a in attempts),
                "attempts": len(attempts),
                "usage_based_usd": sum(a["accounted_usd"] for a in attempts
                                       if a["cost_basis"] == "reported_usage_peak_rates"),
                "reserved_or_uncertain_usd": sum(a["accounted_usd"] for a in attempts
                                                 if a["cost_basis"] != "reported_usage_peak_rates"),
                "accounted_usd": sum(a["accounted_usd"] for a in attempts),
                "budget_target_usd": float(self.config["budget_target_usd"]),
            }

    def _usage_cost(self, response, reserve):
        # completion_tokens already includes reasoning; never count it twice.
        usage = response.get("usage", {}) if isinstance(response, dict) else {}
        try:
            prompt, output = usage["prompt_tokens"], usage["completion_tokens"]
            hit = usage.get("prompt_cache_hit_tokens", 0)
            if any(type(n) is not int or n < 0 for n in (prompt, output, hit)):
                raise ValueError
            if hit > prompt:
                raise ValueError
            cost = (
                (prompt - hit) * self.config["peak_input_usd_per_million_uncached"]
                + hit * self.config["peak_input_usd_per_million_cached"]
                + output * self.config["peak_output_usd_per_million"]
            ) / 1_000_000
            return cost, "reported_usage_peak_rates"
        except (KeyError, TypeError, ValueError):
            return reserve, "usage_unavailable_reservation"

    def _result(self, record, cached):
        attempt = record["attempts"][-1]
        try:
            response = attempt["response"]
            choice = response["choices"][0]
            if choice["finish_reason"] != "stop":
                raise ValueError
            data = _llm_strict_json(choice["message"]["content"])
            if not isinstance(data, dict):
                raise ValueError
        except (KeyError, IndexError, TypeError, ValueError):
            raise RuntimeError(
                f"Request {record['request_id'][:12]}: saved response is incomplete "
                "or is not a JSON object. No trading action was substituted. "
                "Inspect the saved response before changing the prompt/token limit."
            ) from None
        return {
            "data": data, "request_id": record["request_id"], "cache_hit": cached,
            "usage": response.get("usage", {}),
            "provider_model": response.get("model"),
            "system_fingerprint": response.get("system_fingerprint"),
            "accounted_usd": attempt["accounted_usd"],
            "cost_basis": attempt["cost_basis"],
        }

    def request(self, messages, *, purpose, max_tokens, retry_uncertain=False):
        if not isinstance(purpose, str) or not purpose.strip():
            raise ValueError("Give every decision/reflection a stable purpose ID.")
        if type(max_tokens) is not int or not 1 <= max_tokens <= 8192:
            raise ValueError("This non-thinking experiment allows 1–8192 output tokens.")
        if not messages or any(
            m.get("role") not in {"system", "user", "assistant"}
            or not isinstance(m.get("content"), str) for m in messages
        ):
            raise ValueError("Provide text chat messages with valid roles.")
        if "json" not in " ".join(m["content"] for m in messages).lower():
            raise ValueError("The prompt must explicitly request JSON output.")
        payload = {
            "model": self.config["model"], "messages": copy.deepcopy(messages),
            "thinking": copy.deepcopy(self.config["thinking"]),
            "temperature": self.config["temperature"],
            "response_format": {"type": "json_object"},
            "max_tokens": max_tokens, "stream": False,
        }
        key = _llm_digest({"config_hash": self.config_hash,
                           "purpose": purpose, "payload": payload})
        body = _llm_canonical(payload).encode("utf-8")
        # Deliberately cautious input estimate; provider usage replaces it later.
        # This is an application spending guard, not a provider billing guarantee.
        reserve = (
            (len(body) + 2048) * self.config["peak_input_usd_per_million_uncached"]
            + max_tokens * self.config["peak_output_usd_per_million"]
        ) / 1_000_000
        with self.disk_lock:
            key_lock = self.key_locks.setdefault(key, threading.Lock())
            self.active += 1
        try:
            with key_lock:
                with self.disk_lock:
                    record = copy.deepcopy(self.records.get(key, {
                        "request_id": key, "purpose": purpose,
                        "payload": payload, "attempts": [],
                    }))
                if record["attempts"]:
                    last = record["attempts"][-1]
                    if last["status"] == "response":
                        print(f"[cache] {purpose} | {key[:12]}", flush=True)
                        return self._result(record, True)
                    if last["status"] in {"started", "transport_unknown"}:
                        if not retry_uncertain:
                            raise RuntimeError(
                                "Previous attempt has uncertain delivery/billing. "
                                "It remains reserved. Inspect it before explicitly "
                                "using retry_uncertain=True; a retry may cost again."
                            )
                    elif last["status"] != "retryable_http":
                        raise RuntimeError("Previous request failed; inspect its saved status.")

                while len(record["attempts"]) < int(self.config["max_attempts"]):
                    if record["attempts"]:
                        wait = max(0, record["attempts"][-1].get("retry_after_epoch", 0) - time.time())
                        if wait > 30:
                            raise RuntimeError(
                                f"Provider requested a longer wait ({wait:.0f}s remaining). "
                                "Progress is saved; rerun after that interval."
                            )
                        if wait:
                            print(f"[retry wait] {purpose} | {wait:.1f}s", flush=True)
                            time.sleep(wait)

                    with self.slots:
                        with self.disk_lock:
                            if self.status()["accounted_usd"] + reserve > float(self.config["budget_target_usd"]):
                                raise RuntimeError("Spending guard stopped this request before dispatch.")
                            attempt = {
                                "number": len(record["attempts"]) + 1,
                                "started_utc": _llm_now(), "status": "started",
                                "reservation_usd": reserve, "accounted_usd": reserve,
                                "cost_basis": "pending_or_uncertain_reservation",
                            }
                            record["attempts"].append(attempt)
                            self._save(record)
                        print(f"[request] {purpose} | attempt {attempt['number']} | {key[:12]}", flush=True)
                        request = Request(
                            self.config["base_url"].rstrip("/") + "/chat/completions",
                            data=body, method="POST",
                            headers={"Authorization": "Bearer " + self.api_key,
                                     "Content-Type": "application/json"},
                        )
                        started = time.monotonic()
                        try:
                            with urlopen(request, timeout=float(self.config["request_timeout_seconds"])) as response:
                                raw = response.read().decode("utf-8")
                        except HTTPError as error:
                            code = error.code
                            retry_header = error.headers.get("Retry-After") if error.headers else None
                            error.close()
                            delay = min(20, 2 ** attempt["number"]) + random.random()
                            if retry_header:
                                try:
                                    delay = max(delay, float(retry_header))
                                except ValueError:
                                    try:
                                        delay = max(delay, parsedate_to_datetime(retry_header).timestamp() - time.time())
                                    except (ValueError, TypeError, OverflowError):
                                        pass
                            attempt.update({
                                "status": "retryable_http" if code in self.RETRYABLE_HTTP else "http_error",
                                "http_status": code, "finished_utc": _llm_now(),
                                "retry_after_epoch": time.time() + delay,
                            })
                            with self.disk_lock:
                                self._save(record)
                            print(f"[HTTP {code}] {purpose} | saved", flush=True)
                            if code not in self.RETRYABLE_HTTP:
                                raise RuntimeError(f"DeepSeek HTTP {code}; no action executed.") from None
                            continue
                        except (URLError, TimeoutError, ConnectionError, HTTPException, UnicodeError):
                            attempt.update({"status": "transport_unknown", "finished_utc": _llm_now()})
                            with self.disk_lock:
                                self._save(record)
                            raise RuntimeError(
                                "Response not received reliably. Checkpoint saved; billing "
                                "may have occurred. Automatic duplicate dispatch paused."
                            ) from None

                        # Persist the provider response BEFORE parsing decision JSON.
                        try:
                            provider_response = _llm_strict_json(raw)
                        except ValueError:
                            provider_response = {"unparseable_response_text": raw}
                        cost, basis = self._usage_cost(provider_response, reserve)
                        attempt.update({
                            "status": "response", "finished_utc": _llm_now(),
                            "latency_seconds": time.monotonic() - started,
                            "response": provider_response,
                            "accounted_usd": cost, "cost_basis": basis,
                        })
                        with self.disk_lock:
                            self._save(record)
                        total = self.status()["accounted_usd"]
                        print(f"[saved] {purpose} | {attempt['latency_seconds']:.1f}s | "
                              f"accounted total ${total:.4f}", flush=True)
                        return self._result(record, False)
                raise RuntimeError("Maximum attempts reached for this request; progress is saved.")
        finally:
            with self.disk_lock:
                self.active -= 1


LLM_REQUEST_CLIENT = LLMRequestClient(
    LLM_API_CONFIG, LLM_DEEPSEEK_API_KEY, LLM_API_RUN_DIR,
    LLM_DISK_LOCK, LLM_API_REQUEST_SLOTS, llm_atomic_write_json,
)


def llm_json_request(messages, *, purpose, max_tokens=None, retry_uncertain=False):
    return LLM_REQUEST_CLIENT.request(
        messages, purpose=purpose,
        max_tokens=(LLM_REQUEST_CLIENT.config["decision_max_tokens"] if max_tokens is None else max_tokens),
        retry_uncertain=retry_uncertain,
    )


def llm_request_status():
    return LLM_REQUEST_CLIENT.status()


LLM_REQUEST_CLIENT_READY = True
print("Request runner ready. Thinking: disabled.")
print("Cached responses:", LLM_API_CACHE_DIR)
print(json.dumps(llm_request_status(), indent=2))
print("Costs use recorded peak rates; uncertain attempts retain their reservation.")
print("No API calls made by this cell. No replay or memory updates started.")
print("Block 13 complete. Next: a small real reviewer pilot on MEMORY_BUILD dates.")


Request runner ready. Thinking: disabled.
Cached responses: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/responses
{
  "active_requests": 0,
  "saved_responses": 0,
  "attempts": 0,
  "usage_based_usd": 0,
  "reserved_or_uncertain_usd": 0,
  "accounted_usd": 0,
  "budget_target_usd": 10.0
}
Costs use recorded peak rates; uncertain attempts retain their reservation.
No API calls made by this cell. No replay or memory updates started.
Block 13 complete. Next: a small real reviewer pilot on MEMORY_BUILD dates.


In [20]:
# BLOCK 14 — Five real LLM reviews on MEMORY_BUILD dates, with no memory.
# This cell makes paid DeepSeek calls. It never places real trades.
# Rerun unchanged to restore saved steps and reuse saved responses.

import copy
import json
import os
import threading
import time
from pathlib import Path
import pandas as pd
from IPython.display import display, FileLink

for _flag in ["LLM_REQUEST_CLIENT_READY", "LLM_MEMORY_PERIODS_READY",
              "LLM_REPLAY_ACCOUNTING_READY", "LLM_LOCAL_CHECKPOINT_HELPERS_READY"]:
    if not globals().get(_flag, False):
        raise RuntimeError(f"Missing setup: {_flag}. Run the preceding blocks first.")

LLM_PILOT_READY = False
LLM_PILOT_ID = "pilot_no_memory_v1"
LLM_PILOT_DATES = pd.DatetimeIndex(LLM_MEMORY_BUILD_DATES[:5])
if len(LLM_PILOT_DATES) != 5 or not LLM_MEMORY_ASSIGNMENT.loc[
    LLM_PILOT_DATES
].eq("MEMORY_BUILD").all():
    raise ValueError("Pilot must use five MEMORY_BUILD dates.")

LLM_PILOT_CHECKPOINT_PATH = LLM_RUN_CHECKPOINT_DIR / (LLM_PILOT_ID + ".json")

LLM_REVIEW_PROMPT_V1 = """You review a fixed ETH trading model's daily exposure proposal.
Use only the supplied state and available memories. Do not use outside knowledge
of historical ETH prices, news, sentiment, or events. Do not invent missing data.
The objective is to retain useful model exposure while limiting downside risk and
unnecessary turnover, accounting for the stated transaction costs. No action is
guaranteed to improve performance. KEEP is appropriate when evidence for reducing
the proposal is weak. Avoid assuming a single indicator establishes a future move.

Choose exactly one action:
KEEP: target exposure = 1.0 * model.ensemble_proposal. This follows the model;
it does NOT mean holding the current portfolio position unchanged.
HALVE: target exposure = 0.5 * model.ensemble_proposal. This can still reverse
the current position when the model changes direction.
FLAT: target exposure = 0.0. Closing an existing position incurs trading costs.
Never increase, independently reverse, or replace the model's proposal.
Exposure means signed ETH notional divided by equity; positive is long, negative
is short. Target notional uses pre-trade equity. The position can drift with prices.
Returns, volatility, and drawdown are fractions: 0.01 means 1%. Volatility is daily.
MLP probabilities are class scores, not a guarantee of profit or calibrated risk.
Recent portfolio return and drawdown describe already realized performance.
No sentiment evidence is supplied. No future outcome is available to you.

Return only one JSON object with exactly these four keys:
{"action":"KEEP", "reason":"Brief justification using the supplied evidence.",
 "evidence_fields":["model.ensemble_proposal"], "used_memory_ids":[]}
The example illustrates format only; select your own action from the actual state.
reason: 1-350 characters; summarize the decision, not a long reasoning trace.
evidence_fields: 1-4 distinct field paths from allowed_evidence_fields.
used_memory_ids: IDs of supplied memories you actually used; [] when none are supplied.
Do not claim that a cited field or a memory guarantees a future return.
"""


def _pilot_evidence_fields(state):
    paths = []

    def walk(value, prefix):
        if isinstance(value, dict):
            for key, item in value.items():
                walk(item, prefix + "." + key if prefix else key)
        elif value is not None:
            paths.append(prefix)

    for section in ("model", "market", "portfolio"):
        walk(state[section], section)
    if state.get("data_quality_notes"):
        paths.append("data_quality_notes")
    return sorted(paths)


def _pilot_messages(state, terminal):
    # Explicit allowlist of prompt content: checkpoint/outcomes are never sent.
    if set(state) != {"source_timestamp_utc", "model", "market", "portfolio",
                      "data_quality_notes"}:
        raise ValueError("Reviewer state schema changed; inspect before sending.")
    payload = {
        "state": state,
        "available_memories": [],
        "execution": {
            "fee_plus_slippage_frac": float(LLM_REPLAY_PROTOCOL["fee_rate"])
                                      + float(LLM_REPLAY_PROTOCOL["slippage_rate"]),
            "cost_basis": "absolute traded notional",
            "target_notional_basis": "pre-trade equity",
            "holding_interval": "one daily replay interval",
            "flatten_at_end_of_this_interval": bool(terminal),
        },
        "allowed_evidence_fields": _pilot_evidence_fields(state),
    }
    return [{"role": "system", "content": LLM_REVIEW_PROMPT_V1},
            {"role": "user", "content": _llm_canonical(payload)}]


def _validate_pilot_decision(data, state):
    if not isinstance(data, dict) or set(data) != {
        "action", "reason", "evidence_fields", "used_memory_ids"
    }:
        raise ValueError("Decision must have exactly the four requested JSON keys.")
    if not isinstance(data["action"], str) or data["action"] not in LLMDailyReplay.ACTIONS:
        raise ValueError("Action must be KEEP, HALVE, or FLAT.")
    if not isinstance(data["reason"], str) or not 1 <= len(data["reason"].strip()) <= 350:
        raise ValueError("Reason must contain 1-350 characters.")
    fields = data["evidence_fields"]
    if (not isinstance(fields, list) or not 1 <= len(fields) <= 4
            or any(not isinstance(field, str) for field in fields)
            or len(set(fields)) != len(fields)
            or not set(fields).issubset(_pilot_evidence_fields(state))):
        raise ValueError("Evidence references must be 1-4 distinct supplied fields.")
    if data["used_memory_ids"] != []:
        raise ValueError("This pilot has no memories; used_memory_ids must be [].")
    return copy.deepcopy(data)


def _pilot_purpose(index):
    return LLM_PILOT_ID + "/" + LLM_PILOT_DATES[index].isoformat() + "/review"


def _pilot_snapshot(replay, plan, steps):
    return {
        "schema_version": 1, "plan": plan, "plan_hash": _llm_digest(plan),
        "saved_utc": _llm_now(), "next_index": len(steps), "done": replay.done,
        "portfolio_after": copy.deepcopy(replay._portfolio),
        "steps": copy.deepcopy(steps),
        "scope": "evaluator-only checkpoint; never include this whole file in prompts",
    }


def _run_reviewer_pilot():
    # Fingerprints detect changed inputs, fees, prices or prompts on resumption.
    # Outcome fingerprints are evaluator-only and never included in API messages.
    replay = LLMDailyReplay(LLM_PILOT_DATES)
    initial_portfolio = copy.deepcopy(replay._portfolio)
    source_rows = []
    for date in LLM_PILOT_DATES:
        row = LLM_EVALUATOR_DAILY.loc[date]
        source_rows.append({
            "state_with_fresh_account": build_reviewer_state(
                date, portfolio_state=initial_portfolio),
            "outcome_price_row_utc": pd.Timestamp(row["exit_price_row_utc"]).isoformat(),
            "market_return_1d": float(row["market_return_1d"]),
        })
    plan = {
        "pilot_id": LLM_PILOT_ID, "dates": [d.isoformat() for d in LLM_PILOT_DATES],
        "mode": LLM_EXPERIMENT_MODE, "memory_version": "EMPTY",
        "system_prompt": LLM_REVIEW_PROMPT_V1,
        "request_client_config_hash": LLM_REQUEST_CLIENT.config_hash,
        "replay_protocol": copy.deepcopy(LLM_REPLAY_PROTOCOL),
        "source_fingerprint": _llm_digest(source_rows),
        "request_payload_version": "pilot-state-v1",
        "decision_schema_version": "four-fields-v1",
        "max_tokens": LLM_REQUEST_CLIENT.config["decision_max_tokens"],
    }
    steps = []
    if LLM_PILOT_CHECKPOINT_PATH.exists():
        saved = _llm_strict_json(LLM_PILOT_CHECKPOINT_PATH.read_text())
        if saved["plan_hash"] != _llm_digest(plan) or saved["plan"] != plan:
            raise RuntimeError("Pilot settings or data changed. Keep this checkpoint; inspect the mismatch.")
        steps = saved["steps"]
        if not isinstance(steps, list) or not 0 <= len(steps) <= 5 or saved["next_index"] != len(steps):
            raise RuntimeError("Pilot checkpoint has an invalid step count.")

        # Reconstruct the account by replaying saved actions, without API calls.
        for index, entry in enumerate(steps):
            state = replay.get_state()
            decision = _validate_pilot_decision(entry["decision"], state)
            cached = LLM_REQUEST_CLIENT.records.get(entry["request_id"])
            if not cached or cached["attempts"][-1]["status"] != "response":
                raise RuntimeError("A saved step's API response is missing. Restore the complete backup.")
            if (cached["purpose"] != _pilot_purpose(index)
                    or cached["payload"]["messages"] != _pilot_messages(state, index == 4)
                    or LLM_REQUEST_CLIENT._result(cached, True)["data"] != decision):
                raise RuntimeError("Saved request does not match the reconstructed decision state.")
            if _llm_digest(replay.step(decision["action"])) != _llm_digest(entry["replay_record"]):
                raise RuntimeError("Reconstructed accounting differs from the saved replay.")
        if (replay._portfolio != saved["portfolio_after"]
                or replay.done != saved["done"]):
            raise RuntimeError("Restored portfolio differs from the saved checkpoint.")
    else:
        llm_atomic_write_json(LLM_PILOT_CHECKPOINT_PATH, _pilot_snapshot(replay, plan, steps))

    print(f"Pilot: {LLM_PILOT_DATES[0].date()} to {LLM_PILOT_DATES[-1].date()} | "
          f"restored {len(steps)}/5 decisions | memory: empty", flush=True)
    initial_status = llm_request_status()
    started = time.monotonic()
    session_completed = 0
    while not replay.done:
        index = len(steps)
        state = replay.get_state()
        messages = _pilot_messages(state, index == 4)
        stop_heartbeat = threading.Event()
        call_started = time.monotonic()

        def heartbeat():
            while not stop_heartbeat.wait(10):
                print(f"[waiting] day {index + 1}/5 | "
                      f"{time.monotonic() - call_started:.0f}s elapsed | "
                      f"{index}/5 decisions checkpointed", flush=True)

        reporter = threading.Thread(target=heartbeat, daemon=True)
        reporter.start()
        try:
            result = llm_json_request(messages, purpose=_pilot_purpose(index),
                                      max_tokens=plan["max_tokens"])
        finally:
            stop_heartbeat.set()
            reporter.join(timeout=1)
        try:
            decision = _validate_pilot_decision(result["data"], state)
        except ValueError as error:
            print("Saved response for inspection:", json.dumps(result["data"], indent=2))
            raise RuntimeError(f"Pilot paused before executing the action: {error}") from None

        # Only now does the evaluator apply the selected action to the next return.
        record = replay.step(decision["action"])
        steps.append({
            "request_id": result["request_id"], "decision": decision,
            "provider_model": result["provider_model"],
            "usage": result["usage"], "response_reused": result["cache_hit"],
            "replay_record": record,
        })
        llm_atomic_write_json(LLM_PILOT_CHECKPOINT_PATH, _pilot_snapshot(replay, plan, steps))
        session_completed += 1
        status = llm_request_status()
        elapsed = time.monotonic() - started
        eta = elapsed / session_completed * (5 - len(steps))
        date_text = pd.Timestamp(record["source_timestamp_utc"]).date()
        print(f"[day {len(steps)}/5 saved] {date_text} | {decision['action']} | "
              f"target={record['target_exposure']:+.3f} | "
              f"equity=${record['equity_after']:,.2f} | "
              f"accounted API total=${status['accounted_usd']:.4f} | "
              f"ETA~{eta:.0f}s", flush=True)

    ledger = pd.DataFrame([{
        "date": pd.Timestamp(e["replay_record"]["source_timestamp_utc"]).date().isoformat(),
        "action": e["decision"]["action"],
        "base_proposal": e["replay_record"]["base_proposal"],
        "target_exposure": e["replay_record"]["target_exposure"],
        "net_return": e["replay_record"]["net_return"],
        "equity_after": e["replay_record"]["equity_after"],
        "drawdown_frac": e["replay_record"]["drawdown_frac"],
        "reason": e["decision"]["reason"],
    } for e in steps])
    print("\nREAL LLM DECISIONS — FIVE-DAY PIPELINE PILOT, NOT A PERFORMANCE STUDY")
    display(ledger)
    print("Action counts:", ledger["action"].value_counts().to_dict())
    print("Checkpoint:", LLM_PILOT_CHECKPOINT_PATH)
    print("New HTTP attempts this execution:", llm_request_status()["attempts"] - initial_status["attempts"])
    print("Memory remains empty. Re-running unchanged restores all five steps without new API calls.")
    return replay, ledger


# Prevent two copies of this cell from running the same account concurrently.
if "LLM_PILOT_LOCK" not in globals():
    LLM_PILOT_LOCK = threading.Lock()
if not LLM_PILOT_LOCK.acquire(blocking=False):
    raise RuntimeError("This pilot is already running in this kernel.")
try:
    LLM_PILOT_REPLAY, LLM_PILOT_LEDGER = _run_reviewer_pilot()
    LLM_PILOT_READY = True
finally:
    LLM_PILOT_LOCK.release()

# Capture the paid responses and replay checkpoint in a fresh ZIP.
_pilot_backup, _pilot_file_count = backup_llm_experiment()
print("\nDownload the updated backup:", _pilot_backup)
display(FileLink(os.path.relpath(_pilot_backup, Path.cwd())))
print("Block 14 complete. No reflections or memory updates performed yet.")


Pilot: 2023-10-01 to 2023-10-05 | restored 0/5 decisions | memory: empty
[request] pilot_no_memory_v1/2023-10-01T00:00:00+00:00/review | attempt 1 | 4238db158ac0
[saved] pilot_no_memory_v1/2023-10-01T00:00:00+00:00/review | 2.1s | accounted total $0.0002
[day 1/5 saved] 2023-10-01 | KEEP | target=+0.861 | equity=$96,387.68 | accounted API total=$0.0002 | ETA~8s
[request] pilot_no_memory_v1/2023-10-02T00:00:00+00:00/review | attempt 1 | 0219f459a4a8
[saved] pilot_no_memory_v1/2023-10-02T00:00:00+00:00/review | 1.7s | accounted total $0.0005
[day 2/5 saved] 2023-10-02 | KEEP | target=-0.762 | equity=$96,460.04 | accounted API total=$0.0005 | ETA~6s
[request] pilot_no_memory_v1/2023-10-03T00:00:00+00:00/review | attempt 1 | fddcb6556bb7
[saved] pilot_no_memory_v1/2023-10-03T00:00:00+00:00/review | 1.2s | accounted total $0.0007
[day 3/5 saved] 2023-10-03 | KEEP | target=-0.721 | equity=$96,825.99 | accounted API total=$0.0007 | ETA~3s
[request] pilot_no_memory_v1/2023-10-04T00:00:00+00:00

,date,action,base_proposal,target_exposure,net_return,equity_after,drawdown_frac,reason
0,2023-10-01,KEEP,0.860508,0.860508,-0.036123,96387.677737,0.036123,Model ensemble proposal is strongly long (0.86...
1,2023-10-02,KEEP,-0.761648,-0.761648,0.000751,96460.042027,0.035400,"Model ensemble proposal is -0.7616, a strong s..."
2,2023-10-03,KEEP,-0.720801,-0.720801,0.003794,96825.990868,0.031740,"Model ensemble proposal is -0.72, consistent w..."
3,2023-10-04,KEEP,-0.887716,-0.887716,0.019328,98697.481780,0.013025,"Model ensemble proposal is -0.888, with all PP..."
4,2023-10-05,KEEP,-0.720801,-0.720801,-0.016667,97052.460026,0.029475,"Model ensemble proposal is -0.72, consistent w..."


Action counts: {'KEEP': 5}
Checkpoint: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/checkpoints/pilot_no_memory_v1.json
New HTTP attempts this execution: 5
Memory remains empty. Re-running unchanged restores all five steps without new API calls.

Download the updated backup: /kaggle/working/eth_llm_backup_20260929T115708_103404Z.zip


/kaggle/working/eth_llm_backup_20260929T115708_103404Z.zip

Block 14 complete. No reflections or memory updates performed yet.


In [21]:
# BLOCK 15 — Outcome feedback for the completed five-day pilot.
# No API calls. No memory updates. Alternatives are one-step counterfactuals,
# each starting from the SAME actual pre-decision portfolio for that date.

import copy
import json
import math
import pandas as pd
from IPython.display import display

if not globals().get("LLM_PILOT_READY", False):
    raise RuntimeError("Complete Block 14 successfully first.")
LLM_PILOT_FEEDBACK_READY = False

_saved = _llm_strict_json(LLM_PILOT_CHECKPOINT_PATH.read_text())
_dates = pd.DatetimeIndex(pd.to_datetime(_saved["plan"]["dates"], utc=True))
if (not _saved["done"] or _saved["next_index"] != 5
        or len(_saved["steps"]) != 5 or len(_dates) != 5):
    raise RuntimeError("The five-day pilot must be fully checkpointed first.")
if not LLM_MEMORY_ASSIGNMENT.loc[_dates].eq("MEMORY_BUILD").all():
    raise RuntimeError("Feedback generation is restricted to MEMORY_BUILD dates.")
if (_saved["plan_hash"] != _llm_digest(_saved["plan"])
        or _saved["plan"]["replay_protocol"] != LLM_REPLAY_PROTOCOL
        or not _dates.equals(LLM_PILOT_DATES)):
    raise RuntimeError("Pilot plan or replay settings changed; inspect before continuing.")


def _feedback_get_field(state, path):
    value = state
    for part in path.split("."):
        value = value[part]
    return copy.deepcopy(value)


_replay = LLMDailyReplay(_dates)
_cases, _table_rows = [], []

for _index, _entry in enumerate(_saved["steps"]):
    _state = _replay.get_state()
    _decision = _validate_pilot_decision(_entry["decision"], _state)
    if _llm_digest(_state) != _llm_digest(_entry["replay_record"]["state_before"]):
        raise RuntimeError("Decision state differs from the saved pilot.")

    # Each branch copies the actual account BEFORE today's decision.
    # Keep the original five-day episode so terminal costs occur only on day 5.
    _alternatives = {}
    for _action in LLMDailyReplay.ACTIONS:
        _branch = copy.deepcopy(_replay)
        _branch_record = _branch.step(_action)
        _alternatives[_action] = {
            key: _branch_record[key] for key in [
                "target_exposure", "traded_notional", "trade_cost", "closing_cost",
                "total_cost", "gross_pnl", "net_return", "equity_after",
                "next_exposure", "drawdown_frac", "terminal",
            ]
        }

    # Advance ONLY along the action actually chosen by the LLM.
    _actual = _replay.step(_decision["action"])
    if _llm_digest(_actual) != _llm_digest(_entry["replay_record"]):
        raise RuntimeError("Replayed outcome differs from the saved pilot.")
    if any(_alternatives[_decision["action"]][key] != _actual[key]
           for key in _alternatives[_decision["action"]]):
        raise RuntimeError("Chosen-action counterfactual does not match actual accounting.")

    _best_return = max(item["net_return"] for item in _alternatives.values())
    _best_actions = [
        action for action, item in _alternatives.items()
        if math.isclose(item["net_return"], _best_return, rel_tol=0, abs_tol=1e-12)
    ]
    _gap_bps = max(0.0, (_best_return - _actual["net_return"]) * 10000)
    _case = {
        "case_id": LLM_PILOT_ID + "/" + _actual["source_timestamp_utc"],
        "source_timestamp_utc": _actual["source_timestamp_utc"],
        "outcome_price_row_utc": _actual["outcome_price_row_utc"],
        "request_id": _entry["request_id"],
        "state_at_decision": _state,
        "reviewer_decision": _decision,
        "cited_evidence_values": {
            field: _feedback_get_field(_state, field)
            for field in _decision["evidence_fields"]
        },
        "observed_market_return": _actual["market_return"],
        "chosen_outcome": copy.deepcopy(_alternatives[_decision["action"]]),
        "one_step_counterfactuals": _alternatives,
        "hindsight_best_actions_by_net_return": _best_actions,
        "one_step_gap_bps": _gap_bps,
        "structural_checks": {
            "action_and_response_schema_valid": True,
            "cited_field_paths_exist": True,
            "replay_matches_saved_accounting": True,
        },
        "verified_reasoning_error": None,
        "reasoning_error_status": "Not assessed; a loss or hindsight gap alone is not an error.",
    }
    _cases.append(_case)
    _table_rows.append({
        "date": _dates[_index].date().isoformat(),
        "chosen": _decision["action"],
        "KEEP_net_pct": 100 * _alternatives["KEEP"]["net_return"],
        "HALVE_net_pct": 100 * _alternatives["HALVE"]["net_return"],
        "FLAT_net_pct": 100 * _alternatives["FLAT"]["net_return"],
        "chosen_net_pct": 100 * _actual["net_return"],
        "hindsight_best": "/".join(_best_actions),
        "one_step_gap_bps": _gap_bps,
    })

_latest_outcome = max(pd.Timestamp(case["outcome_price_row_utc"]) for case in _cases)
LLM_PILOT_FEEDBACK = {
    "schema_version": 1,
    "pilot_id": LLM_PILOT_ID,
    "mode": LLM_EXPERIMENT_MODE,
    "source_checkpoint_hash": _llm_digest(_saved),
    "latest_outcome_price_row_utc": _latest_outcome.isoformat(),
    "interpretation": {
        "purpose": "Offline MEMORY_BUILD reflection input; not a reviewer's decision state.",
        "counterfactual_scope": "One interval from the same actual portfolio; not full alternative policies.",
        "metric": "Realized net return including the replay's fees and slippage; diagnostic only.",
        "error_rule": "Do not label an action wrong solely because it lost or another action won afterward.",
        "selection_rule": "Do not turn hindsight-best actions into deterministic training labels.",
        "chronology_rule": (
            "A lesson derived from this whole batch must not be applied to a decision "
            "at or before the latest outcome price-row key. Execution timing remains unverified."
        ),
        "limitations": (
            "Five cases; simulator counterfactuals assume the same market return across actions. "
            "Cited field checks do not establish that the natural-language reasoning is correct."
        ),
    },
    "cases": _cases,
}
LLM_PILOT_FEEDBACK_HASH = _llm_digest(LLM_PILOT_FEEDBACK)
LLM_PILOT_FEEDBACK_PATH = (
    LLM_API_RUN_DIR / "evaluator_only" / (LLM_PILOT_ID + "_feedback.json")
)
if LLM_PILOT_FEEDBACK_PATH.exists():
    if _llm_strict_json(LLM_PILOT_FEEDBACK_PATH.read_text()) != LLM_PILOT_FEEDBACK:
        raise RuntimeError("Existing feedback differs. Preserve it and inspect the change.")
else:
    llm_atomic_write_json(LLM_PILOT_FEEDBACK_PATH, LLM_PILOT_FEEDBACK)

LLM_PILOT_COUNTERFACTUAL_TABLE = pd.DataFrame(_table_rows)
print("ONE-STEP HINDSIGHT COMPARISONS — RETURNS IN PERCENT; GAP IN BASIS POINTS")
display(LLM_PILOT_COUNTERFACTUAL_TABLE)
print("1 basis point = 0.01 percentage points. These rows are not a hindsight trading strategy.")

print("\nFULL REVIEWER EXPLANATIONS AND CITED INPUT VALUES")
for _case in _cases:
    print("\n" + _case["source_timestamp_utc"][:10] + " | "
          + _case["reviewer_decision"]["action"])
    print(_case["reviewer_decision"]["reason"])
    print("Cited inputs:", json.dumps(_case["cited_evidence_values"], ensure_ascii=False))

LLM_PILOT_FEEDBACK_READY = True
print("\nSaved evaluator-only feedback:", LLM_PILOT_FEEDBACK_PATH)
print("Block 15 complete. No API calls, candidate lessons, or memory updates yet.")


ONE-STEP HINDSIGHT COMPARISONS — RETURNS IN PERCENT; GAP IN BASIS POINTS


,date,chosen,KEEP_net_pct,HALVE_net_pct,FLAT_net_pct,chosen_net_pct,hindsight_best,one_step_gap_bps
0,2023-10-01,KEEP,-3.612322,-1.806161,0.000000,-3.612322,FLAT,361.232226
1,2023-10-02,KEEP,0.075076,-0.026708,-0.128493,0.075076,KEEP,0.000000
2,2023-10-03,KEEP,0.379379,0.132847,-0.113685,0.379379,KEEP,0.000000
3,2023-10-04,KEEP,1.932839,0.938873,-0.107136,1.932839,KEEP,0.000000
4,2023-10-05,KEEP,-1.666731,-0.897241,-0.127750,-1.666731,FLAT,153.898135


1 basis point = 0.01 percentage points. These rows are not a hindsight trading strategy.

FULL REVIEWER EXPLANATIONS AND CITED INPUT VALUES

2023-10-01 | KEEP
Model ensemble proposal is strongly long (0.86), with all PPO seeds at 1.0 and MLP direction long. No drawdown or negative portfolio return to justify reducing exposure; recent returns are positive. Volatility is moderate. Following the model at full exposure is appropriate.
Cited inputs: {"model.ensemble_proposal": 0.8605075840726089, "model.ppo_proposal": 1.0, "model.mlp_direction": 1, "portfolio.drawdown_frac": 0.0}

2023-10-02 | KEEP
Model ensemble proposal is -0.7616, a strong short signal supported by all PPO seeds and MLP direction. Current position is long 0.8566, so following the model requires reversing to short. Recent 1d return is -4.05% and drawdown is 3.61%, but no supplied evidence justifies overriding the model's directional call. KEEP retains full model exposure.
Cited inputs: {"model.ensemble_proposal": -0.76164

In [22]:
# BLOCK 16 — Verified cost feedback -> one candidate memory proposal.
# One paid LLM request if not already cached (bounded HTTP retries may add attempts).
# A candidate is saved as PENDING_VALIDATION. No active memory is changed.

import copy
import json
import math
import os
import threading
import time
from pathlib import Path
import pandas as pd
from IPython.display import display, FileLink

if not globals().get("LLM_PILOT_FEEDBACK_READY", False):
    raise RuntimeError("Run Block 15 successfully first.")
if not globals().get("LLM_REQUEST_CLIENT_READY", False):
    raise RuntimeError("Run Block 13 successfully first.")

LLM_CANDIDATE_PROPOSAL_READY = False
LLM_PILOT_CANDIDATE = None
LLM_PILOT_CANDIDATE_PATH = None


def _save_immutable_lesson_artifact(path, value):
    with LLM_DISK_LOCK:
        if path.exists():
            if _llm_strict_json(path.read_text()) != value:
                raise RuntimeError(f"Saved artifact differs: {path.name}. Preserve it and inspect.")
        else:
            llm_atomic_write_json(path, value)


# Ground the feedback in the saved completed pilot, rather than a pasted example.
_feedback = _llm_strict_json(LLM_PILOT_FEEDBACK_PATH.read_text())
_checkpoint = _llm_strict_json(LLM_PILOT_CHECKPOINT_PATH.read_text())
if (_llm_digest(_feedback) != LLM_PILOT_FEEDBACK_HASH
        or _feedback["source_checkpoint_hash"] != _llm_digest(_checkpoint)
        or _checkpoint["plan"]["replay_protocol"] != LLM_REPLAY_PROTOCOL):
    raise RuntimeError("Feedback, checkpoint or replay settings changed.")

_case = _feedback["cases"][0]
_date = pd.Timestamp(_case["source_timestamp_utc"])
if LLM_MEMORY_ASSIGNMENT.loc[_date] != "MEMORY_BUILD":
    raise RuntimeError("Candidate generation must use MEMORY_BUILD data.")

# The assistant identified this particular claim in the displayed explanation.
# Code verifies the numerical contradiction; this is an assisted pilot audit,
# not an automatic detector of every possible natural-language error.
_claim = "halving/flattening would add turnover cost"
if _claim not in _case["reviewer_decision"]["reason"].lower():
    raise RuntimeError("The identified claim is absent. Inspect the actual explanation first.")

_state = copy.deepcopy(_case["state_at_decision"])
_equity = float(_state["portfolio"]["equity"])
_position = float(_state["portfolio"]["current_position"])
_base = float(_state["model"]["ensemble_proposal"])
_rate = float(LLM_REPLAY_PROTOCOL["fee_rate"]) + float(LLM_REPLAY_PROTOCOL["slippage_rate"])
if (not all(math.isfinite(x) for x in (_equity, _position, _base, _rate))
        or _equity <= 0 or _rate <= 0 or not 0 < _base <= 1
        or not math.isclose(_position, 0, rel_tol=0, abs_tol=1e-12)):
    raise RuntimeError("This audit requires the observed fresh, flat account and positive proposal.")

_cost_rows = []
for _action, _multiplier in LLMDailyReplay.ACTIONS.items():
    _target = _multiplier * _base
    _notional = _equity * abs(_target - _position)
    _entry_cost = _rate * _notional
    _saved_alternative = _case["one_step_counterfactuals"][_action]
    if (_saved_alternative["terminal"]
            or not math.isclose(_entry_cost, _saved_alternative["trade_cost"],
                                rel_tol=1e-9, abs_tol=1e-8)):
        raise RuntimeError("Calculated entry cost differs from the saved nonterminal replay.")
    _cost_rows.append({
        "action": _action, "target_exposure": _target,
        "traded_notional_usd": _notional, "entry_cost_usd": _entry_cost,
    })
_costs = {row["action"]: row["entry_cost_usd"] for row in _cost_rows}
if not (_costs["KEEP"] > _costs["HALVE"] > _costs["FLAT"] == 0):
    raise RuntimeError("The expected numerical contradiction was not established.")

LLM_PILOT_COST_AUDIT = {
    "schema_version": 1,
    "audit_type": "assistant_identified_claim_with_programmatic_cost_verification",
    "case_id": _case["case_id"],
    "source_feedback_hash": LLM_PILOT_FEEDBACK_HASH,
    "claim_span": _claim,
    "interpretation": "The stated cost justification for KEEP over HALVE/FLAT is incorrect here.",
    "verified_error": "incorrect_entry_turnover_cost_comparison",
    "formula": "entry_cost = (fee + slippage) * equity * abs(target_exposure - current_position)",
    "fee_plus_slippage_frac": _rate,
    "cost_comparison": _cost_rows,
    "scope": (
        "Verify the entry-cost comparison only. This does not establish which action "
        "was best ex ante, or whether changing the decision would generalize."
    ),
}
_audit_hash = _llm_digest(LLM_PILOT_COST_AUDIT)
_audit_path = LLM_API_RUN_DIR / "events" / ("cost_audit_" + _audit_hash[:16] + ".json")
_save_immutable_lesson_artifact(_audit_path, LLM_PILOT_COST_AUDIT)

print("VERIFIED ENTRY-COST COMPARISON — KNOWN AT DECISION TIME")
display(pd.DataFrame(_cost_rows))
print("A cost-reasoning error is verified; the trading action itself is not labelled wrong.")

LLM_REFLECTION_PROMPT_V1 = """You improve the review procedure of a frozen ETH risk reviewer.
You are given one reviewer decision, its pre-decision state, and a verified audit.
Propose at most ONE short reusable memory instruction addressing the verified error.
Alternatively return NO_CHANGE if the evidence does not justify a procedural lesson.

This is a lesson proposal, not acceptance or validation. Do not claim it improves
returns, Sharpe, drawdown, or future performance. Do not invent events or sentiment.
Use the audited cost arithmetic. No future returns or hindsight-winning actions
are supplied. Correcting a cost explanation does not automatically justify changing
the trade. Do not prescribe always choosing the lowest-cost action, going flat from
a fresh account, or halving after a loss. Entry costs depend on current position and
target position; the ordering can change in other states. Account for separate exit
costs when the execution protocol requires them, without claiming they are known
from this nonterminal example. Do not memorize dates or specific exposure values.
The instruction should apply prospectively using available state/protocol values.

Return only a JSON object with exactly these three keys:
{"status":"PROPOSE", "reason":"Why a procedural correction is justified.",
 "candidate":{
   "title":"Short lesson title",
   "instruction":"A concise prospective review instruction.",
   "supporting_case_ids":["the supplied case_id"],
   "expected_benefit":"The reasoning behavior this aims to improve.",
   "limitation":"What remains uncertain and requires validation."
 }}
For NO_CHANGE, use {"status":"NO_CHANGE", "reason":"Explanation", "candidate":null}.
Limits: reason 250 characters, title 80, instruction 650, expected_benefit 200,
limitation 250. Cite only the supplied case_id. No other keys.
"""

# Explicit allowlist: omit realized market returns, best actions and hindsight gaps.
_reflection_input = {
    "case_id": _case["case_id"],
    "state_at_decision": _state,
    "original_reviewer_decision": _case["reviewer_decision"],
    "verified_audit": {
        key: LLM_PILOT_COST_AUDIT[key] for key in [
            "claim_span", "verified_error", "formula", "fee_plus_slippage_frac",
            "cost_comparison", "scope",
        ]
    },
    "available_memories": [],
}
_messages = [
    {"role": "system", "content": LLM_REFLECTION_PROMPT_V1},
    {"role": "user", "content": _llm_canonical(_reflection_input)},
]

_reflection_stop = threading.Event()
_reflection_start = time.monotonic()


def _reflection_heartbeat():
    while not _reflection_stop.wait(10):
        print(f"[reflection waiting] {time.monotonic() - _reflection_start:.0f}s elapsed", flush=True)


_reporter = threading.Thread(target=_reflection_heartbeat, daemon=True)
_reporter.start()
try:
    _result = llm_json_request(
        _messages,
        purpose=LLM_PILOT_ID + "/cost_reflection_v1/" + _audit_hash,
        max_tokens=LLM_REQUEST_CLIENT.config["reflection_max_tokens"],
    )
finally:
    _reflection_stop.set()
    _reporter.join(timeout=1)


def _validate_reflection(data, case_id):
    if not isinstance(data, dict) or set(data) != {"status", "reason", "candidate"}:
        raise ValueError("Reflection must have status, reason and candidate only.")
    if not isinstance(data["reason"], str) or not 1 <= len(data["reason"].strip()) <= 300:
        raise ValueError("Reflection reason must contain 1-300 characters.")
    if data["status"] == "NO_CHANGE":
        if data["candidate"] is not None:
            raise ValueError("NO_CHANGE requires candidate=null.")
        return None
    if data["status"] != "PROPOSE":
        raise ValueError("Reflection status must be PROPOSE or NO_CHANGE.")
    candidate = data["candidate"]
    limits = {"title": 80, "instruction": 650, "expected_benefit": 200, "limitation": 250}
    if not isinstance(candidate, dict) or set(candidate) != set(limits) | {"supporting_case_ids"}:
        raise ValueError("Candidate has missing or unexpected fields.")
    for key, limit in limits.items():
        if not isinstance(candidate[key], str) or not 1 <= len(candidate[key].strip()) <= limit:
            raise ValueError(f"Candidate {key} must contain 1-{limit} characters.")
    if candidate["supporting_case_ids"] != [case_id]:
        raise ValueError("Candidate must reference exactly the supplied case_id.")
    return copy.deepcopy(candidate)


try:
    _candidate_text = _validate_reflection(_result["data"], _case["case_id"])
except ValueError as _error:
    print("Saved reflection for inspection:", json.dumps(_result["data"], indent=2))
    raise RuntimeError(f"Candidate was not admitted: {_error}") from None

_reflection_record = {
    "schema_version": 1,
    "request_id": _result["request_id"],
    "audit_hash": _audit_hash,
    "source_feedback_hash": LLM_PILOT_FEEDBACK_HASH,
    "provider_model": _result["provider_model"],
    "usage": _result["usage"],
    "reflection": _result["data"],
}
_reflection_path = LLM_API_RUN_DIR / "events" / ("reflection_" + _result["request_id"] + ".json")
_save_immutable_lesson_artifact(_reflection_path, _reflection_record)

if _candidate_text is not None:
    _candidate_id = "mem_" + _llm_digest({"audit_hash": _audit_hash,
                                         "candidate": _candidate_text})[:16]
    LLM_PILOT_CANDIDATE = {
        "schema_version": 1, "candidate_id": _candidate_id,
        "status": "PENDING_VALIDATION", "active": False,
        "origin": "assisted_pilot_cost_audit",
        "source_feedback_hash": LLM_PILOT_FEEDBACK_HASH,
        "source_audit_hash": _audit_hash,
        "proposer_request_id": _result["request_id"],
        "source_stage": "MEMORY_BUILD",
        "latest_source_outcome_price_row_utc": _feedback["latest_outcome_price_row_utc"],
        "temporal_rule": "Do not use at or before the latest source outcome row in chronological replay.",
        "format_checked": True,
        "semantic_validation": "NOT_RUN",
        "held_out_validation": "NOT_RUN",
        "lesson": _candidate_text,
    }
    LLM_PILOT_CANDIDATE_PATH = LLM_MEMORY_DIR / "candidates" / (_candidate_id + ".json")
    _save_immutable_lesson_artifact(LLM_PILOT_CANDIDATE_PATH, LLM_PILOT_CANDIDATE)
    print("\nCANDIDATE MEMORY — PENDING VALIDATION")
    print(json.dumps(LLM_PILOT_CANDIDATE, indent=2, ensure_ascii=False))
    print("Saved candidate:", LLM_PILOT_CANDIDATE_PATH)
else:
    print("\nNO_CHANGE:", _result["data"]["reason"])
    print("Reflection saved. No candidate memory was created.")

LLM_CANDIDATE_PROPOSAL_READY = True
print("\nResponse reused:", _result["cache_hit"])
print("Request accounting:", json.dumps(llm_request_status(), indent=2))
_backup, _count = backup_llm_experiment()
print("Download updated backup:", _backup)
display(FileLink(os.path.relpath(_backup, Path.cwd())))
print("Block 16 complete. Model weights and active reviewer memory remain unchanged.")


RuntimeError: The identified claim is absent. Inspect the actual explanation first.

In [ ]:
# BLOCK 17 — Paired procedural check of one candidate memory.
# 12 SYNTHETIC account states x 2 conditions = 24 intended paid requests.
# Runs up to six requests concurrently, saves every result, and resumes unchanged.
# This is a development screen, not historical market validation or a trading run.

import copy
import json
import math
import os
import random
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from pathlib import Path
import pandas as pd
from IPython.display import display, FileLink

if not globals().get("LLM_CANDIDATE_PROPOSAL_READY", False):
    raise RuntimeError("Complete Block 16 first.")
if not globals().get("LLM_REQUEST_CLIENT_READY", False):
    raise RuntimeError("Run Block 13 first.")
if not globals().get("LLM_PILOT_CANDIDATE_PATH"):
    raise RuntimeError("Block 16 proposed no candidate; there is no lesson to screen.")

LLM_COST_SCREEN_READY = False
_screen_candidate = _llm_strict_json(LLM_PILOT_CANDIDATE_PATH.read_text())
if (_screen_candidate["status"] != "PENDING_VALIDATION"
        or _screen_candidate["active"] is not False
        or _screen_candidate["source_stage"] != "MEMORY_BUILD"
        or not re.fullmatch(r"mem_[0-9a-f]{16}", _screen_candidate["candidate_id"])):
    raise RuntimeError("Expected an inactive candidate from MEMORY_BUILD.")

_screen_id = "procedural_cost_screen_v1"
_screen_dir = (LLM_API_RUN_DIR / "evaluator_only" / _screen_id
               / _screen_candidate["candidate_id"])

# Labels describe coverage for us. Only neutral C01...C12 IDs reach the LLM.
_screen_specs = [
    ("flat_to_long",              80000,  0.00,  0.64, .0015),
    ("flat_to_short",            120000,  0.00, -0.72, .0015),
    ("near_long_proposal",        90000,  0.80,  0.90, .0015),
    ("near_short_proposal",      110000, -0.75, -0.80, .0015),
    ("already_at_half_long",     100000,  0.40,  0.80, .0015),
    ("already_at_half_short",    150000, -0.30, -0.60, .0015),
    ("long_to_short_reversal",    85000,  0.65, -0.70, .0015),
    ("short_to_long_reversal",   105000, -0.55,  0.60, .0015),
    ("tie_keep_halve",            95000,  0.60,  0.80, .0015),
    ("tie_halve_flat",           130000,  0.20,  0.80, .0015),
    ("zero_proposal_all_tie",     90000,  0.45,  0.00, .0009),
    ("marked_exposure_above_one", 70000,  1.05,  0.90, .0020),
]
_screen_cases = [{
    "case_id": f"C{index + 1:02d}", "coverage_label": label,
    "equity_usd": float(equity), "current_position": float(position),
    "ensemble_proposal": float(proposal), "fee_plus_slippage_frac": float(rate),
} for index, (label, equity, position, proposal, rate) in enumerate(_screen_specs)]

LLM_COST_SCREEN_PROMPT_V1 = """Audit entry transaction costs for a synthetic ETH account.
These are arithmetic questions, not investment decisions. Use only the supplied
account, protocol and any available memory. All exposure values are signed ETH
notional divided by pre-trade equity; positive is long and negative is short.
Equity stays fixed during this calculation. The stated fee plus slippage rate
applies to the absolute notional actually traded to reach the target position.

KEEP targets ensemble_proposal, HALVE targets half that proposal, and FLAT targets
zero exposure. KEEP does not mean leaving the current position unchanged.
Calculate entry costs only. There is no price movement or separate terminal
liquidation in this exercise. A current exposure above one can occur after marking
an existing position; still use the supplied current exposure in your calculation.

Return ONLY a JSON object with exactly these two keys:
{"entry_costs_usd":{"KEEP":0.0,"HALVE":0.0,"FLAT":0.0},
 "cheapest_actions":["KEEP"]}
Replace example numbers/actions with your calculations. Costs must be JSON numbers,
in dollars, rounded to cents or more precisely. Include ALL actions tied for the
smallest entry cost, and no duplicates. Cost ranking is not a trading recommendation.
"""


def _screen_expected(case):
    costs = {
        action: case["fee_plus_slippage_frac"] * case["equity_usd"]
                * abs(multiplier * case["ensemble_proposal"] - case["current_position"])
        for action, multiplier in {"KEEP": 1., "HALVE": .5, "FLAT": 0.}.items()
    }
    minimum = min(costs.values())
    cheapest = sorted(action for action, value in costs.items()
                      if math.isclose(value, minimum, rel_tol=0, abs_tol=1e-8))
    return costs, cheapest


def _score_cost_answer(case, answer):
    expected_costs, cheapest = _screen_expected(case)
    valid = isinstance(answer, dict) and set(answer) == {"entry_costs_usd", "cheapest_actions"}
    if valid:
        costs, actions = answer["entry_costs_usd"], answer["cheapest_actions"]
        valid = (isinstance(costs, dict) and set(costs) == set(expected_costs)
                 and all(type(n) in (int, float) and math.isfinite(n) and n >= 0
                         for n in costs.values())
                 and isinstance(actions, list) and 1 <= len(actions) <= 3
                 and all(isinstance(a, str) and a in expected_costs for a in actions)
                 and len(set(actions)) == len(actions))
    error = max(abs(answer["entry_costs_usd"][a] - expected_costs[a])
                for a in expected_costs) if valid else None
    costs_correct = valid and error <= .010000001
    rank_correct = valid and sorted(answer["cheapest_actions"]) == cheapest
    return {
        "format_valid": bool(valid), "all_costs_correct": bool(costs_correct),
        "cheapest_actions_correct": bool(rank_correct),
        "case_passed": bool(costs_correct and rank_correct),
        "max_abs_cost_error_usd": error,
        "expected_costs_usd": expected_costs, "expected_cheapest_actions": cheapest,
    }


def _cost_screen_messages(case, condition):
    # No coverage labels, answer key, returns, or prior test results in the prompt.
    account = {key: case[key] for key in [
        "case_id", "equity_usd", "current_position", "ensemble_proposal", "fee_plus_slippage_frac"
    ]}
    memories = [] if condition == "NO_MEMORY" else [{
        "id": _screen_candidate["candidate_id"],
        "instruction": _screen_candidate["lesson"]["instruction"],
    }]
    return [{"role": "system", "content": LLM_COST_SCREEN_PROMPT_V1},
            {"role": "user", "content": _llm_canonical({
                "synthetic_account": account, "available_memories": memories})}]


def _screen_purpose(case, condition):
    return "/".join([_screen_id, _screen_candidate["candidate_id"], condition, case["case_id"]])


def _screen_find_response(case, condition):
    messages, purpose = _cost_screen_messages(case, condition), _screen_purpose(case, condition)
    with LLM_DISK_LOCK:
        matches = [copy.deepcopy(record) for record in LLM_REQUEST_CLIENT.records.values()
                   if record["purpose"] == purpose and record["payload"]["messages"] == messages
                   and record["payload"]["max_tokens"] == 512]
    if len(matches) != 1 or matches[0]["attempts"][-1]["status"] != "response":
        return None
    return matches[0]


_screen_plan = {
    "screen_id": _screen_id, "candidate_hash": _llm_digest(_screen_candidate),
    "request_client_config_hash": LLM_REQUEST_CLIENT.config_hash,
    "prompt": LLM_COST_SCREEN_PROMPT_V1, "cases": _screen_cases,
    "conditions": ["NO_MEMORY", "WITH_CANDIDATE"], "max_tokens": 512,
    "scoring_version": "cost-and-all-ties-v1", "cost_tolerance_usd": .01,
    "passing_rule": "WITH_CANDIDATE must pass all 12 cases; report paired regressions and gains.",
    "scope": "Synthetic development screen designed after the pilot error; no market-generalization claim.",
    "promotion_rule": "Never activate memory based on this screen alone.",
}
_screen_plan_hash = _llm_digest(_screen_plan)


def _screen_job(case, condition, stop):
    if stop.is_set():
        raise RuntimeError("Screen paused before this queued request started.")
    messages = _cost_screen_messages(case, condition)
    try:
        result = llm_json_request(messages, purpose=_screen_purpose(case, condition), max_tokens=512)
        answer, request_id = result["data"], result["request_id"]
    except RuntimeError:
        # A received but malformed/truncated answer is a scored format failure.
        # An uncertain transport failure or exhausted HTTP retry pauses the run.
        cached = _screen_find_response(case, condition)
        if cached is None:
            raise
        try:
            LLM_REQUEST_CLIENT._result(cached, True)
        except RuntimeError:
            answer, request_id = None, cached["request_id"]
        else:
            raise
    record = {
        "plan_hash": _screen_plan_hash, "case_id": case["case_id"],
        "condition": condition, "request_id": request_id,
        "answer": answer, "score": _score_cost_answer(case, answer),
    }
    path = _screen_dir / "results" / (condition + "_" + case["case_id"] + ".json")
    _save_immutable_lesson_artifact(path, record)
    return record


def _run_cost_screen():
    _save_immutable_lesson_artifact(_screen_dir / "plan.json", _screen_plan)
    jobs = [(case, condition) for case in _screen_cases for condition in _screen_plan["conditions"]]
    random.Random(2026).shuffle(jobs)
    records, remaining = [], []
    for case, condition in jobs:
        path = _screen_dir / "results" / (condition + "_" + case["case_id"] + ".json")
        if not path.exists():
            remaining.append((case, condition))
            continue
        saved = _llm_strict_json(path.read_text())
        cached = _screen_find_response(case, condition)
        if (saved["plan_hash"] != _screen_plan_hash or saved["case_id"] != case["case_id"]
                or saved["condition"] != condition or not cached
                or saved["request_id"] != cached["request_id"]):
            raise RuntimeError("Saved screen result/cache does not match this plan. Restore the complete backup.")
        try:
            answer = LLM_REQUEST_CLIENT._result(cached, True)["data"]
        except RuntimeError:
            answer = None
        if saved["answer"] != answer or saved["score"] != _score_cost_answer(case, answer):
            raise RuntimeError("Saved screen answer or score differs from the cached response.")
        records.append(saved)

    print(f"Synthetic cost screen | restored {len(records)}/24 | pending {len(remaining)}", flush=True)
    before = llm_request_status()
    started, newly_finished = time.monotonic(), 0
    if remaining:
        stop = threading.Event()
        pool = ThreadPoolExecutor(max_workers=min(6, int(LLM_REQUEST_CLIENT.config["max_parallel_requests"])))
        futures = {pool.submit(_screen_job, case, condition, stop) for case, condition in remaining}
        try:
            while futures:
                completed, futures = wait(futures, timeout=5, return_when=FIRST_COMPLETED)
                if not completed:
                    print(f"[waiting] {len(records)}/24 saved | "
                          f"active={llm_request_status()['active_requests']} | "
                          f"elapsed={time.monotonic() - started:.0f}s", flush=True)
                for future in completed:
                    record = future.result()
                    records.append(record)
                    newly_finished += 1
                    elapsed = time.monotonic() - started
                    eta = elapsed / newly_finished * len(futures)
                    print(f"[screen {len(records)}/24 saved] {record['case_id']} "
                          f"{record['condition']} | passed={record['score']['case_passed']} | "
                          f"accounted API total=${llm_request_status()['accounted_usd']:.4f} | "
                          f"ETA~{eta:.0f}s", flush=True)
        except BaseException:
            stop.set()
            for future in futures:
                future.cancel()
            print("Screen paused. Completed results are saved; in-flight requests may finish before shutdown.", flush=True)
            raise
        finally:
            pool.shutdown(wait=True, cancel_futures=True)

    records.sort(key=lambda r: (r["case_id"], r["condition"]))
    rows = [{"case_id": r["case_id"], "condition": r["condition"], **r["score"]} for r in records]
    table = pd.DataFrame(rows)
    summary = table.groupby("condition").agg(
        cases=("case_passed", "size"), format_valid=("format_valid", "sum"),
        costs_correct=("all_costs_correct", "sum"),
        cost_ranking_correct=("cheapest_actions_correct", "sum"),
        cases_passed=("case_passed", "sum"),
    )
    paired = table.pivot(index="case_id", columns="condition", values="case_passed")
    gains = int((~paired["NO_MEMORY"] & paired["WITH_CANDIDATE"]).sum())
    regressions = int((paired["NO_MEMORY"] & ~paired["WITH_CANDIDATE"]).sum())
    passed = bool(paired["WITH_CANDIDATE"].all())
    report = {
        "plan_hash": _screen_plan_hash, "candidate_id": _screen_candidate["candidate_id"],
        "procedural_screen_passed": passed,
        "paired_gains": gains, "paired_regressions": regressions,
        "baseline_cases_passed": int(paired["NO_MEMORY"].sum()),
        "candidate_cases_passed": int(paired["WITH_CANDIDATE"].sum()),
        "total_cases_per_condition": 12,
        "interpretation": (
            "This measures answers to explicit cost questions on 12 synthetic development cases. "
            "It does not test spontaneous use during trading reviews or prove statistical improvement."
        ),
        "market_validation": "NOT_RUN", "active_memory_changed": False,
    }
    _save_immutable_lesson_artifact(_screen_dir / "report.json", report)
    print("\nPROCEDURAL COST SCREEN — SYNTHETIC STATES, NOT MARKET PERFORMANCE")
    display(summary)
    display(paired)
    print(json.dumps(report, indent=2))
    print("New HTTP attempts this execution:", llm_request_status()["attempts"] - before["attempts"])
    print("Results:", _screen_dir)
    return table, summary, report


if "LLM_COST_SCREEN_LOCK" not in globals():
    LLM_COST_SCREEN_LOCK = threading.Lock()
if not LLM_COST_SCREEN_LOCK.acquire(blocking=False):
    raise RuntimeError("This cost screen is already running in this kernel.")
try:
    LLM_COST_SCREEN_RESULTS, LLM_COST_SCREEN_SUMMARY, LLM_COST_SCREEN_REPORT = _run_cost_screen()
    LLM_COST_SCREEN_READY = True
finally:
    LLM_COST_SCREEN_LOCK.release()

_backup, _count = backup_llm_experiment()
print("\nDownload updated backup:", _backup)
display(FileLink(os.path.relpath(_backup, Path.cwd())))
print("Block 17 complete. Candidate remains inactive; MEMORY_GATE and FROZEN_EVAL were not used.")


In [23]:
# BLOCK 18 — Python supplies entry costs to the reviewer.
# Defines a VERSION 2 input builder and validator; previews a fresh account.
# No LLM requests, historical replay steps, or memory activation in this cell.
# Existing version 1 requests/results remain unchanged.

import copy
import json
import math
import os
from pathlib import Path
import pandas as pd
from IPython.display import display, FileLink

for _flag in ["LLM_REPLAY_ACCOUNTING_READY", "LLM_MEMORY_PERIODS_READY",
              "LLM_LOCAL_CHECKPOINT_HELPERS_READY", "LLM_REQUEST_CLIENT_READY"]:
    if not globals().get(_flag, False):
        raise RuntimeError("Missing setup: " + _flag)
if "LLM_REVIEW_PROMPT_V1" not in globals():
    raise RuntimeError("Run the earlier reviewer setup in Block 14 first.")

LLM_COST_AWARE_REVIEW_READY = False
LLM_REVIEW_COST_SPEC_V2 = {
    "version": "known-entry-costs-v2",
    "actions": {"KEEP": 1.0, "HALVE": 0.5, "FLAT": 0.0},
    "fee_rate": float(LLM_REPLAY_PROTOCOL["fee_rate"]),
    "slippage_rate": float(LLM_REPLAY_PROTOCOL["slippage_rate"]),
    "formula": "(fee_rate + slippage_rate) * abs(target_exposure * equity - current_position * equity)",
    "target_notional_basis": "pre-trade equity",
    "cost_ranking_tolerance_usd": 1e-8,
}


def compute_decision_costs_v2(state, *, terminal=False):
    """Pure calculation from decision-time state and fixed replay charges.

    Does not access prices, labels, future returns, or prior test results.
    """
    if type(terminal) is not bool:
        raise ValueError("terminal must be True or False.")
    equity = float(state["portfolio"]["equity"])
    position = float(state["portfolio"]["current_position"])
    proposal = float(state["model"]["ensemble_proposal"])
    fee = LLM_REVIEW_COST_SPEC_V2["fee_rate"]
    slippage = LLM_REVIEW_COST_SPEC_V2["slippage_rate"]
    if (not all(math.isfinite(x) for x in [equity, position, proposal, fee, slippage])
            or equity <= 0 or abs(proposal) > 1 + 1e-8 or fee < 0 or slippage < 0):
        raise ValueError("Invalid equity, position, proposal or cost settings.")
    # Current exposure may legitimately drift beyond [-1, 1]. Do not clip it.
    old_notional = position * equity
    options = {}
    for action, multiplier in LLM_REVIEW_COST_SPEC_V2["actions"].items():
        target = multiplier * proposal
        target_notional = target * equity
        turnover = abs(target_notional - old_notional)
        cost = (fee + slippage) * turnover
        if not all(math.isfinite(x) for x in [target_notional, turnover, cost]):
            raise ValueError("Non-finite dollar calculation.")
        options[action] = {
            "target_exposure": target,
            "target_notional_usd": target_notional,
            "traded_notional_usd": turnover,
            "entry_cost_usd": cost,
            "entry_cost_equity_frac": cost / equity,
        }
    minimum = min(option["entry_cost_usd"] for option in options.values())
    return {
        "source": "deterministic_python_calculation",
        "scope": "Entry charges under the replay's fixed fee/slippage model.",
        "fee_rate": fee, "slippage_rate": slippage,
        "actions": options,
        "lowest_entry_cost_actions": [action for action, option in options.items()
            if math.isclose(option["entry_cost_usd"], minimum, rel_tol=0,
                            abs_tol=LLM_REVIEW_COST_SPEC_V2["cost_ranking_tolerance_usd"])],
        "flatten_at_end_of_interval": terminal,
        "end_of_interval_closing_cost_usd": None if terminal else 0.0,
        "closing_cost_note": (
            "The evaluator will charge liquidation at the end of this interval. "
            "Its amount depends on the future marked notional and is unavailable now."
            if terminal else "No automatic end-of-interval liquidation on this step."
        ),
    }


LLM_REVIEW_PROMPT_V2 = LLM_REVIEW_PROMPT_V1 + """

COMPUTED ENTRY COSTS
The input also contains decision_costs, calculated by Python from the current
account and the fixed execution rules. Use the supplied target exposures, dollar
costs and lowest_entry_cost_actions directly. Do not calculate replacement costs,
invent a different fee rule, or treat KEEP as no trade. The table covers entry
charges only. Any unknown future liquidation cost is explicitly null.
If you justify a choice using costs, cite a relevant decision_costs field and keep
your explanation consistent with the supplied comparison. The cheapest entry
action need not be the best trading action: weigh the model proposal, market
features and portfolio state too. No expected future return is provided.
Memories are tentative procedural guidance. They cannot override supplied facts
or require unsupported claims about future price direction.
"""


def build_review_payload_v2(state, *, terminal=False, available_memories=None):
    if set(state) != {"source_timestamp_utc", "model", "market", "portfolio", "data_quality_notes"}:
        raise ValueError("Unexpected reviewer state fields; inspect before sending.")
    memories = copy.deepcopy([] if available_memories is None else available_memories)
    if not isinstance(memories, list):
        raise ValueError("available_memories must be a list of supplied id/instruction objects.")
    for memory in memories:
        if (not isinstance(memory, dict) or set(memory) != {"id", "instruction"}
                or not all(isinstance(memory[key], str) and memory[key].strip()
                           for key in ["id", "instruction"])):
            raise ValueError("Every supplied memory needs a nonempty id and instruction.")
    if len({memory["id"] for memory in memories}) != len(memories):
        raise ValueError("Duplicate memory IDs.")
    costs = compute_decision_costs_v2(state, terminal=terminal)
    fields = _pilot_evidence_fields(state)
    fields += [f"decision_costs.actions.{action}.{key}"
               for action, row in costs["actions"].items() for key in row]
    fields += ["decision_costs.lowest_entry_cost_actions",
               "decision_costs.fee_rate", "decision_costs.slippage_rate"]
    return {
        "state": copy.deepcopy(state), "available_memories": memories,
        "decision_costs": costs,
        "execution": {
            "fee_plus_slippage_frac": costs["fee_rate"] + costs["slippage_rate"],
            "cost_basis": "absolute traded notional",
            "target_notional_basis": "pre-trade equity",
            "holding_interval": "one daily replay interval",
            "flatten_at_end_of_this_interval": terminal,
        },
        "allowed_evidence_fields": sorted(fields),
    }


def build_review_messages_v2(state, *, terminal=False, available_memories=None):
    payload = build_review_payload_v2(state, terminal=terminal, available_memories=available_memories)
    return [{"role": "system", "content": LLM_REVIEW_PROMPT_V2},
            {"role": "user", "content": _llm_canonical(payload)}]


def validate_review_decision_v2(data, payload):
    if not isinstance(data, dict) or set(data) != {"action", "reason", "evidence_fields", "used_memory_ids"}:
        raise ValueError("Decision must have the four requested keys.")
    if not isinstance(data["action"], str) or data["action"] not in LLM_REVIEW_COST_SPEC_V2["actions"]:
        raise ValueError("Action must be KEEP, HALVE or FLAT.")
    if not isinstance(data["reason"], str) or not 1 <= len(data["reason"].strip()) <= 350:
        raise ValueError("Reason must contain 1-350 characters.")
    fields = data["evidence_fields"]
    if (not isinstance(fields, list) or not 1 <= len(fields) <= 4
            or not all(isinstance(x, str) for x in fields)
            or len(set(fields)) != len(fields)
            or not set(fields).issubset(payload["allowed_evidence_fields"])):
        raise ValueError("Cite 1-4 distinct fields supplied to this review.")
    ids = data["used_memory_ids"]
    if (not isinstance(ids, list) or not all(isinstance(x, str) for x in ids)
            or len(set(ids)) != len(ids)
            or not set(ids).issubset({m["id"] for m in payload["available_memories"]})):
        raise ValueError("A cited memory was not supplied to this review.")
    return copy.deepcopy(data)


LLM_REVIEW_INPUT_MANIFEST_V2 = {
    "harness_version": "known-entry-costs-v2", "origin": "human_authored_harness_change",
    "cost_spec": copy.deepcopy(LLM_REVIEW_COST_SPEC_V2), "system_prompt": LLM_REVIEW_PROMPT_V2,
    "condition_rule": "Every reviewer condition receives costs computed from its own current account.",
    "comparison_rule": "Compare memory conditions within version 2; do not attribute v1-to-v2 changes to memory.",
    "memory_rule": "This change activates no memory. Chronological memory eligibility is enforced by the replay runner.",
}
LLM_REVIEW_INPUT_MANIFEST_V2_PATH = Path(LLM_API_RUN_DIR) / "harnesses" / "known_entry_costs_v2.json"
with LLM_DISK_LOCK:
    if LLM_REVIEW_INPUT_MANIFEST_V2_PATH.exists():
        if _llm_strict_json(LLM_REVIEW_INPUT_MANIFEST_V2_PATH.read_text()) != LLM_REVIEW_INPUT_MANIFEST_V2:
            raise RuntimeError("Version 2 settings changed. Preserve this version and use a new version name.")
    else:
        llm_atomic_write_json(LLM_REVIEW_INPUT_MANIFEST_V2_PATH, LLM_REVIEW_INPUT_MANIFEST_V2)

# Fresh-account PREVIEW only. No action is selected or applied here.
_cost_preview_account = LLMDailyReplay(LLM_MEMORY_BUILD_DATES[:5])
_cost_preview_state = _cost_preview_account.get_state()
LLM_REVIEW_V2_EXAMPLE_PAYLOAD = build_review_payload_v2(_cost_preview_state, terminal=False)
print("PYTHON-COMPUTED ENTRY COSTS — FRESH ACCOUNT PREVIEW")
print("Source row:", _cost_preview_state["source_timestamp_utc"])
print("Current position:", _cost_preview_state["portfolio"]["current_position"])
display(pd.DataFrame.from_dict(LLM_REVIEW_V2_EXAMPLE_PAYLOAD["decision_costs"]["actions"], orient="index"))
print("Lowest entry cost:", LLM_REVIEW_V2_EXAMPLE_PAYLOAD["decision_costs"]["lowest_entry_cost_actions"])
print("Cost-input and prompt version saved:", LLM_REVIEW_INPUT_MANIFEST_V2_PATH)
LLM_COST_AWARE_REVIEW_READY = True
_cost_backup, _cost_count = backup_llm_experiment()
print("Download updated backup:", _cost_backup)
display(FileLink(os.path.relpath(_cost_backup, Path.cwd())))
print("Block 18 complete: version 2 input functions ready. No LLM calls or memory updates.")
print("Earlier pilot functions still use version 1. The next replay must explicitly use the new version 2 functions.")


PYTHON-COMPUTED ENTRY COSTS — FRESH ACCOUNT PREVIEW
Source row: 2023-10-01T00:00:00+00:00
Current position: 0.0


,target_exposure,target_notional_usd,traded_notional_usd,entry_cost_usd,entry_cost_equity_frac
KEEP,0.860508,86050.758407,86050.758407,129.076138,0.001291
HALVE,0.430254,43025.379204,43025.379204,64.538069,0.000645
FLAT,0.000000,0.000000,0.000000,0.000000,0.000000


Lowest entry cost: ['FLAT']
Cost-input and prompt version saved: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/harnesses/known_entry_costs_v2.json
Download updated backup: /kaggle/working/eth_llm_backup_20260929T115858_783601Z.zip


/kaggle/working/eth_llm_backup_20260929T115858_783601Z.zip

Block 18 complete: version 2 input functions ready. No LLM calls or memory updates.
Earlier pilot functions still use version 1. The next replay must explicitly use the new version 2 functions.


In [26]:
# BLOCK 18B — Treat the requested 350-character reason length as a style rule.
# Place AFTER Block 18 and BEFORE Blocks 19/20 for future notebook runs.
# Run this cell now, then rerun Block 20 unchanged.
# No API calls, action execution, prompt changes, response edits or truncation.

import copy
import json
import os
from pathlib import Path
import pandas as pd
from IPython.display import display, FileLink

for _flag in ["LLM_COST_AWARE_REVIEW_READY", "LLM_REQUEST_CLIENT_READY",
              "LLM_LOCAL_CHECKPOINT_HELPERS_READY"]:
    if not globals().get(_flag, False):
        raise RuntimeError("Missing setup: " + _flag)
if LLM_REQUEST_CLIENT.active:
    raise RuntimeError("Wait for active request workers to stop before applying this recovery.")
for _lock_name in ["LLM_COST_PILOT_LOCK", "LLM_BUILD_COLLECTION_LOCK"]:
    _lock = globals().get(_lock_name)
    if _lock is not None and _lock.locked():
        raise RuntimeError("Stop the active replay before changing validation: " + _lock_name)

LLM_REVIEW_VALIDATION_POLICY_V2 = {
    "policy_id": "v2-reason-length-warning-v1",
    "scope": "All version 2 reviewer conditions using validate_review_decision_v2.",
    "requested_reason_characters": 350,
    "hard_reason_requirement": "A nonempty string. Preserve the original text exactly.",
    "long_reason_handling": "Record a style violation; accept only if all structural checks pass.",
    "unchanged_checks": ["exact response keys", "allowed action", "valid supplied evidence references",
                         "valid supplied memory IDs", "completed JSON response enforced by request client"],
    "prompt_changed": False, "response_changed": False,
    "origin": "Human-authored validation amendment after a development-run formatting failure.",
    "evaluation_rule": "Apply equally to memory and no-memory conditions; retain style violations for reporting.",
}
_reason_policy_path = Path(LLM_API_RUN_DIR) / "harnesses" / "validation_policy_v2_reason_length_warning.json"
with LLM_DISK_LOCK:
    if _reason_policy_path.exists():
        if _llm_strict_json(_reason_policy_path.read_text()) != LLM_REVIEW_VALIDATION_POLICY_V2:
            raise RuntimeError("A different policy is already recorded at this path; preserve it and inspect.")
    else:
        llm_atomic_write_json(_reason_policy_path, LLM_REVIEW_VALIDATION_POLICY_V2)


def validate_review_decision_v2(data, payload):
    # Same structural/action/evidence checks as Block 18. Only the reason's
    # requested maximum becomes a recorded style warning, never a text rewrite.
    if not isinstance(data, dict) or set(data) != {"action", "reason", "evidence_fields", "used_memory_ids"}:
        raise ValueError("Decision must have the four requested keys.")
    if not isinstance(data["action"], str) or data["action"] not in LLM_REVIEW_COST_SPEC_V2["actions"]:
        raise ValueError("Action must be KEEP, HALVE or FLAT.")
    if not isinstance(data["reason"], str) or not data["reason"].strip():
        raise ValueError("Reason must be nonempty text. A missing explanation cannot be repaired by this length policy.")
    fields = data["evidence_fields"]
    if (not isinstance(fields, list) or not 1 <= len(fields) <= 4
            or not all(isinstance(x, str) for x in fields)
            or len(set(fields)) != len(fields)
            or not set(fields).issubset(payload["allowed_evidence_fields"])):
        raise ValueError("Cite 1-4 distinct fields supplied to this review.")
    ids = data["used_memory_ids"]
    if (not isinstance(ids, list) or not all(isinstance(x, str) for x in ids)
            or len(set(ids)) != len(ids)
            or not set(ids).issubset({m["id"] for m in payload["available_memories"]})):
        raise ValueError("A cited memory was not supplied to this review.")

    n_chars = len(data["reason"].strip())
    if n_chars > 350:
        observation = {
            "schema_version": 1, "policy_id": LLM_REVIEW_VALIDATION_POLICY_V2["policy_id"],
            "event_type": "reason_style_limit_exceeded",
            "source_timestamp_utc": payload["state"]["source_timestamp_utc"],
            "review_input_hash": _llm_digest(payload), "original_decision_hash": _llm_digest(data),
            "requested_max_characters": 350, "observed_characters": n_chars,
            "original_reason": data["reason"], "original_action": data["action"],
            "response_modified": False, "structural_validation": "PASSED",
            "meaning": "Formatting warning only; this does not certify explanation correctness or execute an action.",
        }
        path = (Path(LLM_API_RUN_DIR) / "events" / "reason_style_warnings"
                / (_llm_digest(observation) + ".json"))
        with LLM_DISK_LOCK:
            if path.exists():
                if _llm_strict_json(path.read_text()) != observation:
                    raise RuntimeError("Saved reason-format observation differs; preserve it and inspect.")
            else:
                llm_atomic_write_json(path, observation)
                print(f"[format warning saved] {observation['source_timestamp_utc'][:10]} | "
                      f"reason {n_chars} characters; requested 350. Original response retained.", flush=True)
    return copy.deepcopy(data)


print("Version 2 validator updated: long reasons are recorded as style warnings.")
print("Policy saved:", _reason_policy_path)

# Inspect the next uncommitted collection response, if one exists. This uses
# exact messages rebuilt from the SAVED account; it never generates a response.
_resume_root = Path(LLM_RUN_CHECKPOINT_DIR) / "memory_build_no_memory_costs_v2"
_progress_path, _plan_path = _resume_root / "progress.json", _resume_root / "plan.json"
if _progress_path.exists() and _plan_path.exists():
    _progress = _llm_strict_json(_progress_path.read_text())
    _plan = _llm_strict_json(_plan_path.read_text())
    if _progress["plan_hash"] != _llm_digest(_plan):
        raise RuntimeError("Saved collection plan/progress differ. Do not change or delete them.")
    _n = _progress["next_index"]
    if type(_n) is not int or not 0 <= _n <= len(_plan["dates"]):
        raise RuntimeError("Invalid collection progress index.")
    print(f"Saved collection progress: {_n}/{len(_plan['dates'])} days.")
    if _n < len(_plan["dates"]):
        _date = pd.Timestamp(_plan["dates"][_n])
        _state = build_reviewer_state(_date, portfolio_state=_progress["portfolio_after"])
        _messages = build_review_messages_v2(_state, terminal=_n == len(_plan["dates"]) - 1, available_memories=[])
        _purpose = _plan["collection_id"] + "/" + _date.isoformat() + "/review"
        with LLM_DISK_LOCK:
            _matches = [copy.deepcopy(r) for r in LLM_REQUEST_CLIENT.records.values()
                if r["purpose"] == _purpose and r["payload"]["messages"] == _messages
                and r["payload"]["max_tokens"] == _plan["max_tokens"]]
        if len(_matches) > 1:
            raise RuntimeError("Multiple responses match this request; inspect before resuming.")
        if _matches and _matches[0]["attempts"] and _matches[0]["attempts"][-1]["status"] == "response":
            _result = LLM_REQUEST_CLIENT._result(_matches[0], True)
            _payload = _llm_strict_json(_messages[1]["content"])
            _reason = _result["data"].get("reason")
            print("Paused response date:", _date.date())
            print("Reason characters:", len(_reason.strip()) if isinstance(_reason, str) else "not a string")
            print("Original reason:", _reason)
            validate_review_decision_v2(_result["data"], _payload)
            print("Saved response passes the updated structural checks. It is ready for cached reuse.")
        else:
            print("No completed response matches the next step's current inputs. Block 20 will verify resumption.")

_reason_backup, _reason_file_count = backup_llm_experiment()
print("Download updated backup:", _reason_backup)
display(FileLink(os.path.relpath(_reason_backup, Path.cwd())))
print("Block 18B complete. No API calls or actions executed. Rerun Block 20 unchanged.")
print("On future restarts, run this cell after Block 18 so the same validation policy is restored.")


Version 2 validator updated: long reasons are recorded as style warnings.
Policy saved: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/harnesses/validation_policy_v2_reason_length_warning.json
Saved collection progress: 5/273 days.
Paused response date: 2023-10-06
Reason characters: 352
Original reason: Model ensemble proposal is strongly long (0.857), with all PPO seeds at 1.0 and MLP direction long. Current position is short, so KEEP reverses to the model's long target. Recent 1d return is positive and RSI is neutral; no supplied evidence justifies halving or flattening. Costs are lowest for FLAT but that would forgo the model's exposure entirely.
[format warning saved] 2023-10-06 | reason 352 characters; requested 350. Original response retained.
Saved response passes the updated structural checks. It is ready for cached reuse.
Download updated backup: /kaggle/working/eth_llm_backup_20260929T122349_830164Z.zip


/kaggle/working/eth_llm_backup_20260929T122349_830164Z.zip

Block 18B complete. No API calls or actions executed. Rerun Block 20 unchanged.
On future restarts, run this cell after Block 18 so the same validation policy is restored.


In [24]:
# BLOCK 19 — Five real reviewer decisions with Python-computed entry costs.
# Paid API calls: five decisions when uncached; Block 13 controls any HTTP retries.
# Separate version 2 account/checkpoint, empty memory, MEMORY_BUILD dates only.
# Sequential within this account. Rerun unchanged to resume from saved responses.

import copy
import json
import math
import os
import threading
import time
from pathlib import Path
import pandas as pd
from IPython.display import display, FileLink

for _flag in ["LLM_COST_AWARE_REVIEW_READY", "LLM_REQUEST_CLIENT_READY",
              "LLM_MEMORY_PERIODS_READY", "LLM_REPLAY_ACCOUNTING_READY",
              "LLM_LOCAL_CHECKPOINT_HELPERS_READY"]:
    if not globals().get(_flag, False):
        raise RuntimeError("Missing setup: " + _flag + ". Run the preceding setup blocks.")

LLM_COST_PILOT_READY = False
LLM_COST_PILOT_ID = "pilot_no_memory_costs_v2"
LLM_COST_PILOT_DATES = pd.DatetimeIndex(LLM_MEMORY_BUILD_DATES[:5])
LLM_COST_PILOT_CHECKPOINT_PATH = Path(LLM_RUN_CHECKPOINT_DIR) / (LLM_COST_PILOT_ID + ".json")
if (len(LLM_COST_PILOT_DATES) != 5 or not LLM_MEMORY_ASSIGNMENT.loc[
        LLM_COST_PILOT_DATES].eq("MEMORY_BUILD").all()):
    raise RuntimeError("This pilot must use the first five MEMORY_BUILD dates.")


def _cost_pilot_check_harness():
    manifest = _llm_strict_json(LLM_REVIEW_INPUT_MANIFEST_V2_PATH.read_text())
    if (manifest != LLM_REVIEW_INPUT_MANIFEST_V2
            or manifest["system_prompt"] != LLM_REVIEW_PROMPT_V2
            or manifest["cost_spec"] != LLM_REVIEW_COST_SPEC_V2):
        raise RuntimeError("Cost-input settings differ from the version 2 manifest.")
    if (LLM_REVIEW_COST_SPEC_V2["actions"] != LLMDailyReplay.ACTIONS
            or any(LLM_REVIEW_COST_SPEC_V2[key] != float(LLM_REPLAY_PROTOCOL[key])
                   for key in ["fee_rate", "slippage_rate"])):
        raise RuntimeError("Reviewer costs and replay accounting use different settings.")
    return manifest


def _cost_pilot_inputs(state, index):
    messages = build_review_messages_v2(state, terminal=index == 4, available_memories=[])
    payload = _llm_strict_json(messages[1]["content"])
    if (messages[0] != {"role": "system", "content": LLM_REVIEW_PROMPT_V2}
            or payload["available_memories"] != [] or payload["state"] != state
            or payload != build_review_payload_v2(state, terminal=index == 4, available_memories=[])):
        raise RuntimeError("Version 2 prompt inputs do not match the current empty-memory account.")
    return messages, payload


def _cost_pilot_purpose(index):
    return LLM_COST_PILOT_ID + "/" + LLM_COST_PILOT_DATES[index].isoformat() + "/review"


def _cost_pilot_snapshot(replay, plan, steps):
    return {
        "schema_version": 1, "plan": plan, "plan_hash": _llm_digest(plan),
        "saved_utc": _llm_now(), "next_index": len(steps), "done": replay.done,
        "portfolio_after": copy.deepcopy(replay._portfolio), "steps": copy.deepcopy(steps),
        "scope": "Evaluator-only checkpoint. Never send this file to the reviewer.",
    }


def _cost_pilot_check_execution(payload, record):
    supplied = payload["decision_costs"]["actions"][record["action"]]
    for field, actual in [
        ("target_exposure", record["target_exposure"]),
        ("traded_notional_usd", record["traded_notional"]),
        ("entry_cost_usd", record["trade_cost"]),
    ]:
        if not math.isclose(supplied[field], actual, rel_tol=1e-10, abs_tol=1e-8):
            raise RuntimeError("Supplied cost/exposure differs from replay execution: " + field)


def _cost_pilot_evidence_value(payload, path):
    parts = path.split(".")
    value = payload if parts[0] == "decision_costs" else payload["state"]
    for part in parts:
        value = value[part]
    return value


def _run_cost_aware_pilot():
    manifest = _cost_pilot_check_harness()
    replay = LLMDailyReplay(LLM_COST_PILOT_DATES)
    initial_portfolio = copy.deepcopy(replay._portfolio)
    source_rows = []
    for index, date in enumerate(LLM_COST_PILOT_DATES):
        fresh_state = build_reviewer_state(date, portfolio_state=initial_portfolio)
        messages, _ = _cost_pilot_inputs(fresh_state, index)
        outcome = LLM_EVALUATOR_DAILY.loc[date]
        source_rows.append({
            "messages_with_fresh_account": messages,
            "outcome_price_row_utc": pd.Timestamp(outcome["exit_price_row_utc"]).isoformat(),
            "market_return_1d": float(outcome["market_return_1d"]),
        })
    # The outcome fingerprint is only for resumption checks; it never reaches the LLM.
    plan = {
        "pilot_id": LLM_COST_PILOT_ID, "dates": [d.isoformat() for d in LLM_COST_PILOT_DATES],
        "mode": LLM_EXPERIMENT_MODE, "memory_version": "EMPTY",
        "harness_manifest": manifest, "request_client_config_hash": LLM_REQUEST_CLIENT.config_hash,
        "replay_protocol": copy.deepcopy(LLM_REPLAY_PROTOCOL),
        "source_fingerprint": _llm_digest(source_rows),
        "decision_schema_version": "four-fields-cost-evidence-v2",
        "max_tokens": int(LLM_REQUEST_CLIENT.config["decision_max_tokens"]),
        "interpretation": "Development integration pilot; no memory-learning or performance claim.",
    }
    steps = []
    if LLM_COST_PILOT_CHECKPOINT_PATH.exists():
        saved = _llm_strict_json(LLM_COST_PILOT_CHECKPOINT_PATH.read_text())
        if saved["plan"] != plan or saved["plan_hash"] != _llm_digest(plan):
            raise RuntimeError("Version 2 pilot settings/data changed. Preserve this checkpoint and inspect.")
        steps = saved["steps"]
        if not isinstance(steps, list) or not 0 <= len(steps) <= 5 or saved["next_index"] != len(steps):
            raise RuntimeError("Pilot checkpoint has an invalid step count.")
        for index, entry in enumerate(steps):
            state = replay.get_state()
            messages, payload = _cost_pilot_inputs(state, index)
            decision = validate_review_decision_v2(entry["decision"], payload)
            with LLM_DISK_LOCK:
                cached = copy.deepcopy(LLM_REQUEST_CLIENT.records.get(entry["request_id"]))
            if not cached or not cached["attempts"] or cached["attempts"][-1]["status"] != "response":
                raise RuntimeError("Missing cached response for a saved step. Restore the complete backup.")
            if (cached["purpose"] != _cost_pilot_purpose(index)
                    or cached["payload"]["messages"] != messages
                    or cached["payload"]["max_tokens"] != plan["max_tokens"]
                    or LLM_REQUEST_CLIENT._result(cached, True)["data"] != decision
                    or entry["decision_costs"] != payload["decision_costs"]):
                raise RuntimeError("Saved response/costs do not match the reconstructed decision inputs.")
            reconstructed = replay.step(decision["action"])
            _cost_pilot_check_execution(payload, reconstructed)
            if _llm_digest(reconstructed) != _llm_digest(entry["replay_record"]):
                raise RuntimeError("Reconstructed version 2 accounting differs from the checkpoint.")
        if replay._portfolio != saved["portfolio_after"] or replay.done != saved["done"]:
            raise RuntimeError("Restored account does not match the saved checkpoint.")
    else:
        llm_atomic_write_json(LLM_COST_PILOT_CHECKPOINT_PATH, _cost_pilot_snapshot(replay, plan, steps))

    print(f"Version 2 pilot | {LLM_COST_PILOT_DATES[0].date()} to {LLM_COST_PILOT_DATES[-1].date()} | "
          f"restored {len(steps)}/5 | memory: empty | Python costs: supplied", flush=True)
    before = llm_request_status()
    started, completed_now = time.monotonic(), 0
    while not replay.done:
        index = len(steps)
        state = replay.get_state()
        messages, payload = _cost_pilot_inputs(state, index)
        stop_heartbeat = threading.Event()
        call_started = time.monotonic()

        def heartbeat():
            while not stop_heartbeat.wait(10):
                print(f"[waiting] day {index + 1}/5 | "
                      f"{time.monotonic() - call_started:.0f}s elapsed | {index}/5 saved", flush=True)

        reporter = threading.Thread(target=heartbeat, daemon=True)
        reporter.start()
        try:
            result = llm_json_request(messages, purpose=_cost_pilot_purpose(index), max_tokens=plan["max_tokens"])
        finally:
            stop_heartbeat.set()
            reporter.join(timeout=1)
        try:
            decision = validate_review_decision_v2(result["data"], payload)
        except ValueError as error:
            print("Saved response for inspection:", json.dumps(result["data"], indent=2))
            raise RuntimeError("Pilot paused before execution: " + str(error)) from None

        # The reviewer has chosen. Only the evaluator now applies the future return.
        record = replay.step(decision["action"])
        _cost_pilot_check_execution(payload, record)
        steps.append({
            "request_id": result["request_id"], "decision": decision,
            "decision_costs": copy.deepcopy(payload["decision_costs"]),
            "provider_model": result["provider_model"], "system_fingerprint": result["system_fingerprint"],
            "usage": result["usage"], "response_reused": result["cache_hit"], "replay_record": record,
        })
        llm_atomic_write_json(LLM_COST_PILOT_CHECKPOINT_PATH, _cost_pilot_snapshot(replay, plan, steps))
        completed_now += 1
        eta = (time.monotonic() - started) / completed_now * (5 - len(steps))
        print(f"[day {len(steps)}/5 saved] {pd.Timestamp(record['source_timestamp_utc']).date()} | "
              f"{decision['action']} | target={record['target_exposure']:+.3f} | "
              f"entry cost=${record['trade_cost']:.2f} | equity=${record['equity_after']:,.2f} | "
              f"accounted API total=${llm_request_status()['accounted_usd']:.4f} | ETA~{eta:.0f}s", flush=True)

    ledger = pd.DataFrame([{
        "date": pd.Timestamp(e["replay_record"]["source_timestamp_utc"]).date().isoformat(),
        "action": e["decision"]["action"], "base_proposal": e["replay_record"]["base_proposal"],
        "target_exposure": e["replay_record"]["target_exposure"],
        "entry_cost_usd": e["replay_record"]["trade_cost"],
        "terminal_closing_cost_usd": e["replay_record"]["closing_cost"],
        "net_return": e["replay_record"]["net_return"],
        "equity_after": e["replay_record"]["equity_after"],
        "drawdown_frac": e["replay_record"]["drawdown_frac"],
    } for e in steps])
    print("\nREAL REVIEWER DECISIONS — VERSION 2 DEVELOPMENT PILOT")
    display(ledger)
    print("\nFULL EXPLANATIONS AND INPUT COSTS")
    for index, entry in enumerate(steps):
        _, payload = _cost_pilot_inputs(entry["replay_record"]["state_before"], index)
        print(ledger.iloc[index]["date"], "|", entry["decision"]["action"])
        print(entry["decision"]["reason"])
        print("Supplied entry costs (USD):", json.dumps({
            a: round(values["entry_cost_usd"], 6) for a, values in entry["decision_costs"]["actions"].items()}))
        print("Cited input values:", json.dumps({
            field: _cost_pilot_evidence_value(payload, field) for field in entry["decision"]["evidence_fields"]}))
    print("\nAction counts:", ledger["action"].value_counts().to_dict())
    print("New HTTP attempts this execution:", llm_request_status()["attempts"] - before["attempts"])
    print("Checkpoint:", LLM_COST_PILOT_CHECKPOINT_PATH)
    print("Memory stayed empty. Explanation correctness still requires inspection.")
    print("This checks the new cost inputs; five days do not establish trading improvement.")
    return replay, ledger


if "LLM_COST_PILOT_LOCK" not in globals():
    LLM_COST_PILOT_LOCK = threading.Lock()
if not LLM_COST_PILOT_LOCK.acquire(blocking=False):
    raise RuntimeError("This version 2 pilot is already running in this kernel.")
try:
    LLM_COST_PILOT_REPLAY, LLM_COST_PILOT_LEDGER = _run_cost_aware_pilot()
    LLM_COST_PILOT_READY = True
finally:
    LLM_COST_PILOT_LOCK.release()

_v2_backup, _v2_file_count = backup_llm_experiment()
print("\nDownload updated backup:", _v2_backup)
display(FileLink(os.path.relpath(_v2_backup, Path.cwd())))
print("Block 19 complete. No reflections or memory updates. Unchanged reruns reuse saved responses.")


Version 2 pilot | 2023-10-01 to 2023-10-05 | restored 0/5 | memory: empty | Python costs: supplied
[request] pilot_no_memory_costs_v2/2023-10-01T00:00:00+00:00/review | attempt 1 | cb4b659d87d3
[saved] pilot_no_memory_costs_v2/2023-10-01T00:00:00+00:00/review | 1.4s | accounted total $0.0016
[day 1/5 saved] 2023-10-01 | KEEP | target=+0.861 | entry cost=$129.08 | equity=$96,387.68 | accounted API total=$0.0016 | ETA~6s
[request] pilot_no_memory_costs_v2/2023-10-02T00:00:00+00:00/review | attempt 1 | 990c09289c0e
[saved] pilot_no_memory_costs_v2/2023-10-02T00:00:00+00:00/review | 1.3s | accounted total $0.0020
[day 2/5 saved] 2023-10-02 | HALVE | target=-0.381 | entry cost=$178.91 | equity=$96,361.93 | accounted API total=$0.0020 | ETA~4s
[request] pilot_no_memory_costs_v2/2023-10-03T00:00:00+00:00/review | attempt 1 | e606939c5df4
[saved] pilot_no_memory_costs_v2/2023-10-03T00:00:00+00:00/review | 1.2s | accounted total $0.0024
[day 3/5 saved] 2023-10-03 | KEEP | target=-0.721 | entry 

,date,action,base_proposal,target_exposure,entry_cost_usd,terminal_closing_cost_usd,net_return,equity_after,drawdown_frac
0,2023-10-01,KEEP,0.860508,0.860508,129.076138,0.000000,-0.036123,96387.677737,0.036123
1,2023-10-02,HALVE,-0.761648,-0.380824,178.911380,0.000000,-0.000267,96361.934248,0.036381
2,2023-10-03,KEEP,-0.720801,-0.720801,49.356244,0.000000,0.003337,96683.517226,0.033165
3,2023-10-04,KEEP,-0.887716,-0.887716,25.111037,0.000000,0.019329,98552.301484,0.014477
4,2023-10-05,KEEP,-0.720801,-0.720801,19.345558,108.826476,-0.016667,96909.699556,0.030903



FULL EXPLANATIONS AND INPUT COSTS
2023-10-01 | KEEP
Model ensemble proposal is strongly long (0.86), PPO seeds all 1.0, and MLP direction is positive. Portfolio is flat with no drawdown, so following the model retains useful exposure; HALVE/FLAT would cut a supported signal. Entry cost is modest relative to equity.
Supplied entry costs (USD): {"FLAT": 0.0, "HALVE": 64.538069, "KEEP": 129.076138}
Cited input values: {"model.ensemble_proposal": 0.8605075840726089, "model.ppo_seed_proposals.ppo_seed0": 1.0, "portfolio.current_position": 0.0, "decision_costs.actions.KEEP.entry_cost_equity_frac": 0.0012907613761089132}
2023-10-02 | HALVE
Model ensemble is strongly short (-0.76) and current position is long (+0.86), so a reversal is warranted; HALVE cuts entry cost versus KEEP while still expressing the model's short signal, and FLAT would forgo the model's directional exposure.
Supplied entry costs (USD): {"FLAT": 123.851268, "HALVE": 178.91138, "KEEP": 233.971492}
Cited input values: {"mo

/kaggle/working/eth_llm_backup_20260929T115919_075472Z.zip

Block 19 complete. No reflections or memory updates. Unchanged reruns reuse saved responses.


In [27]:
# BLOCK 20 — Full MEMORY_BUILD experience collection, empty memory, version 2.
# One sequential account. Reuses the first four pilot responses only when exact
# request inputs match; all later days use the full-episode terminal rule.
# Each completed day is an atomic file. Backup ZIP every 25 days and at completion.
# No memory generation/activation, MEMORY_GATE replay, or FROZEN_EVAL replay here.

import copy
import json
import math
import os
import re
import threading
import time
from pathlib import Path
import pandas as pd
from IPython.display import display, FileLink

for _flag in ["LLM_COST_PILOT_READY", "LLM_COST_AWARE_REVIEW_READY", "LLM_REQUEST_CLIENT_READY",
              "LLM_MEMORY_PERIODS_READY", "LLM_LOCAL_CHECKPOINT_HELPERS_READY"]:
    if not globals().get(_flag, False):
        raise RuntimeError("Missing setup: " + _flag)

LLM_BUILD_COLLECTION_READY = False
LLM_BUILD_COLLECTION_ID = "memory_build_no_memory_costs_v2"
LLM_BUILD_COLLECTION_DATES = pd.DatetimeIndex(LLM_MEMORY_BUILD_DATES)
LLM_BUILD_COLLECTION_DIR = Path(LLM_RUN_CHECKPOINT_DIR) / LLM_BUILD_COLLECTION_ID
LLM_BUILD_FEEDBACK_PATH = Path(LLM_API_RUN_DIR) / "evaluator_only" / (LLM_BUILD_COLLECTION_ID + "_feedback.json")
LLM_BUILD_BACKUP_EVERY = 25
if (len(LLM_BUILD_COLLECTION_DATES) <= 5
        or not LLM_MEMORY_ASSIGNMENT.loc[LLM_BUILD_COLLECTION_DATES].eq("MEMORY_BUILD").all()):
    raise RuntimeError("Expected the full MEMORY_BUILD period, longer than the five-day pilot.")


def _build_collection_immutable(path, value):
    with LLM_DISK_LOCK:
        if path.exists():
            if _llm_strict_json(path.read_text()) != value:
                raise RuntimeError("Saved artifact differs: " + path.name + ". Preserve it and inspect.")
        else:
            llm_atomic_write_json(path, value)


def _build_collection_inputs(state, index):
    terminal = index == len(LLM_BUILD_COLLECTION_DATES) - 1
    messages = build_review_messages_v2(state, terminal=terminal, available_memories=[])
    payload = _llm_strict_json(messages[1]["content"])
    if (payload != build_review_payload_v2(state, terminal=terminal, available_memories=[])
            or payload["available_memories"] != []
            or messages[0] != {"role": "system", "content": LLM_REVIEW_PROMPT_V2}):
        raise RuntimeError("Unexpected version 2 review inputs.")
    return messages, payload


def _build_collection_purpose(index):
    return LLM_BUILD_COLLECTION_ID + "/" + LLM_BUILD_COLLECTION_DATES[index].isoformat() + "/review"


def _build_collection_cached(request_id):
    with LLM_DISK_LOCK:
        cached = copy.deepcopy(LLM_REQUEST_CLIENT.records.get(request_id))
    if not cached or not cached["attempts"] or cached["attempts"][-1]["status"] != "response":
        raise RuntimeError("A saved response is missing or incomplete. Restore the complete backup.")
    return cached


def _build_collection_reuse_pilot(index, messages, plan, pilot):
    if (index >= 4 or not pilot["done"] or len(pilot["steps"]) != 5
            or pilot["plan"]["request_client_config_hash"] != plan["request_client_config_hash"]
            or pilot["plan"]["dates"][index] != LLM_BUILD_COLLECTION_DATES[index].isoformat()):
        return None
    step = pilot["steps"][index]
    cached = _build_collection_cached(step["request_id"])
    if (cached["purpose"] != _cost_pilot_purpose(index)
            or cached["payload"]["messages"] != messages
            or cached["payload"]["max_tokens"] != plan["max_tokens"]):
        return None
    result = LLM_REQUEST_CLIENT._result(cached, True)
    if result["data"] != step["decision"]:
        raise RuntimeError("Pilot checkpoint decision differs from its cached response.")
    return result


def _build_collection_alternatives(replay):
    # Called only AFTER the current decision has been selected. Alternatives
    # share its actual pre-decision account and the same next market return.
    alternatives = {}
    for action in LLMDailyReplay.ACTIONS:
        branch = copy.copy(replay)
        branch._portfolio = copy.deepcopy(replay._portfolio)
        branch.records = []
        record = branch.step(action)
        alternatives[action] = {key: record[key] for key in [
            "target_exposure", "traded_notional", "trade_cost", "closing_cost",
            "total_cost", "gross_pnl", "net_return", "equity_after", "next_exposure",
            "drawdown_frac", "terminal",
        ]}
    return alternatives


def _build_collection_progress(replay, plan_hash, chain_hash):
    return {"plan_hash": plan_hash, "next_index": replay._i, "last_step_hash": chain_hash,
            "portfolio_after": copy.deepcopy(replay._portfolio), "done": replay.done}


def _build_collection_backup(label):
    backup, count = backup_llm_experiment()
    print(f"[backup] {label} | {count} files | {backup}", flush=True)
    display(FileLink(os.path.relpath(backup, Path.cwd())))
    return backup


def _run_build_collection():
    manifest = _cost_pilot_check_harness()
    dates = LLM_BUILD_COLLECTION_DATES
    replay = LLMDailyReplay(dates)
    initial_portfolio = copy.deepcopy(replay._portfolio)
    source_rows = []
    for index, date in enumerate(dates):
        fresh_state = build_reviewer_state(date, portfolio_state=initial_portfolio)
        messages, _ = _build_collection_inputs(fresh_state, index)
        outcome = LLM_EVALUATOR_DAILY.loc[date]
        source_rows.append({"messages_with_fresh_account": messages,
            "outcome_price_row_utc": pd.Timestamp(outcome["exit_price_row_utc"]).isoformat(),
            "market_return_1d": float(outcome["market_return_1d"])})
    plan = {
        "schema_version": 1, "collection_id": LLM_BUILD_COLLECTION_ID,
        "dates": [d.isoformat() for d in dates], "source_stage": "MEMORY_BUILD",
        "mode": LLM_EXPERIMENT_MODE, "memory_version": "EMPTY",
        "harness_manifest": manifest, "replay_protocol": copy.deepcopy(LLM_REPLAY_PROTOCOL),
        "request_client_config_hash": LLM_REQUEST_CLIENT.config_hash,
        "source_fingerprint": _llm_digest(source_rows),
        "max_tokens": int(LLM_REQUEST_CLIENT.config["decision_max_tokens"]),
        "pilot_reuse": "First four version 2 pilot responses only, with exact messages/settings.",
        "terminal_rule": "One episode across MEMORY_BUILD; automatic liquidation only at its final outcome.",
        "feedback_rule": "One-step counterfactuals are retrospective diagnostics, not correct-action labels.",
        "memory_rule": "Collect empty-memory experience now; generate candidate lessons separately afterward.",
        "step_schema_version": "empty-memory-costs-counterfactuals-v1",
    }
    plan_hash = _llm_digest(plan)
    _build_collection_immutable(LLM_BUILD_COLLECTION_DIR / "plan.json", plan)
    pilot = _llm_strict_json(LLM_COST_PILOT_CHECKPOINT_PATH.read_text())
    if (pilot["plan_hash"] != _llm_digest(pilot["plan"])
            or pilot["plan"]["pilot_id"] != LLM_COST_PILOT_ID):
        raise RuntimeError("Pilot checkpoint identity changed.")

    step_dir = LLM_BUILD_COLLECTION_DIR / "steps"
    files = sorted(step_dir.glob("*.json"))
    if (len(files) > len(dates)
            or [p.name for p in files] != [f"{i:06d}.json" for i in range(len(files))]):
        raise RuntimeError("Daily checkpoints have gaps or unexpected names. Restore the complete backup.")
    entries, chain_hash = [], plan_hash
    snapshots = [_build_collection_progress(replay, plan_hash, chain_hash)]
    for index, path in enumerate(files):
        entry = _llm_strict_json(path.read_text())
        state = replay.get_state()
        messages, payload = _build_collection_inputs(state, index)
        if (entry["index"] != index or entry["plan_hash"] != plan_hash
                or entry["previous_step_hash"] != chain_hash):
            raise RuntimeError("Daily checkpoint chain differs: " + path.name)
        decision = validate_review_decision_v2(entry["decision"], payload)
        cached = _build_collection_cached(entry["request_id"])
        expected_purpose = _build_collection_purpose(index)
        if entry["response_origin"] == "exact_input_pilot_reuse":
            if index >= 4:
                raise RuntimeError("Pilot reuse after day four is not permitted.")
            expected_purpose = _cost_pilot_purpose(index)
        elif entry["response_origin"] != "collection_request":
            raise RuntimeError("Unknown response origin.")
        if (cached["purpose"] != expected_purpose or cached["payload"]["messages"] != messages
                or cached["payload"]["max_tokens"] != plan["max_tokens"]
                or LLM_REQUEST_CLIENT._result(cached, True)["data"] != decision
                or entry["decision_costs"] != payload["decision_costs"]):
            raise RuntimeError("Daily checkpoint differs from the exact cached decision inputs.")
        alternatives = _build_collection_alternatives(replay)
        actual = replay.step(decision["action"])
        _cost_pilot_check_execution(payload, actual)
        if (entry["one_step_counterfactuals"] != alternatives
                or _llm_digest(entry["replay_record"]) != _llm_digest(actual)):
            raise RuntimeError("Restored daily accounting differs from saved feedback.")
        chain_hash = _llm_digest(entry)
        entries.append(entry)
        snapshots.append(_build_collection_progress(replay, plan_hash, chain_hash))

    progress_path = LLM_BUILD_COLLECTION_DIR / "progress.json"
    if progress_path.exists():
        progress = _llm_strict_json(progress_path.read_text())
        n = progress.get("next_index")
        if type(n) is not int or not 0 <= n <= len(entries) or progress != snapshots[n]:
            raise RuntimeError("Progress pointer is inconsistent with saved daily files.")
    # Daily files are authoritative if a stop happened before the pointer update.
    llm_atomic_write_json(progress_path, snapshots[-1])

    print(f"MEMORY_BUILD collection | {dates[0].date()} to {dates[-1].date()} | "
          f"restored {len(entries)}/{len(dates)} | memory: empty", flush=True)
    print("Sequential account; daily atomic saves; ZIP every 25 days. Download ZIPs to retain them outside Kaggle.")
    before = llm_request_status()
    started, completed_now = time.monotonic(), 0
    while not replay.done:
        index = len(entries)
        state = replay.get_state()
        messages, payload = _build_collection_inputs(state, index)
        result = _build_collection_reuse_pilot(index, messages, plan, pilot)
        origin = "exact_input_pilot_reuse" if result is not None else "collection_request"
        if result is not None:
            print(f"[reuse pilot] day {index + 1}/{len(dates)} | exact inputs match", flush=True)
        else:
            stop_heartbeat = threading.Event()
            call_started = time.monotonic()

            def heartbeat():
                while not stop_heartbeat.wait(10):
                    print(f"[waiting] day {index + 1}/{len(dates)} | {index} saved | "
                          f"{time.monotonic() - call_started:.0f}s elapsed", flush=True)

            reporter = threading.Thread(target=heartbeat, daemon=True)
            reporter.start()
            try:
                result = llm_json_request(messages, purpose=_build_collection_purpose(index), max_tokens=plan["max_tokens"])
            finally:
                stop_heartbeat.set()
                reporter.join(timeout=1)
        try:
            decision = validate_review_decision_v2(result["data"], payload)
        except ValueError as error:
            print("Saved response for inspection:", json.dumps(result["data"], indent=2))
            raise RuntimeError("Collection paused before execution: " + str(error)) from None

        alternatives = _build_collection_alternatives(replay)
        actual = replay.step(decision["action"])
        _cost_pilot_check_execution(payload, actual)
        if any(actual[key] != value for key, value in alternatives[decision["action"]].items()):
            raise RuntimeError("Chosen-action feedback differs from the actual account.")
        entry = {
            "index": index, "plan_hash": plan_hash, "previous_step_hash": chain_hash,
            "case_id": LLM_BUILD_COLLECTION_ID + "/" + actual["source_timestamp_utc"],
            "request_id": result["request_id"], "response_origin": origin,
            "decision": decision, "decision_costs": copy.deepcopy(payload["decision_costs"]),
            "provider_model": result["provider_model"], "system_fingerprint": result["system_fingerprint"],
            "usage": result["usage"], "replay_record": actual, "one_step_counterfactuals": alternatives,
        }
        _build_collection_immutable(step_dir / f"{index:06d}.json", entry)
        chain_hash = _llm_digest(entry)
        entries.append(entry)
        llm_atomic_write_json(progress_path, _build_collection_progress(replay, plan_hash, chain_hash))
        completed_now += 1
        eta = (time.monotonic() - started) / completed_now * (len(dates) - len(entries))
        print(f"[day {len(entries)}/{len(dates)} saved] {dates[index].date()} | {decision['action']} | "
              f"target={actual['target_exposure']:+.3f} | equity=${actual['equity_after']:,.2f} | "
              f"accounted API total=${llm_request_status()['accounted_usd']:.4f} | ETA~{eta / 60:.1f}m", flush=True)
        if len(entries) % LLM_BUILD_BACKUP_EVERY == 0 and not replay.done:
            _build_collection_backup(f"{len(entries)}/{len(dates)} days saved")

    cases = []
    for index, entry in enumerate(entries):
        record = entry["replay_record"]
        _, payload = _build_collection_inputs(record["state_before"], index)
        best_return = max(x["net_return"] for x in entry["one_step_counterfactuals"].values())
        cases.append({
            "case_id": entry["case_id"], "source_stage": "MEMORY_BUILD",
            "source_timestamp_utc": record["source_timestamp_utc"],
            "outcome_price_row_utc": record["outcome_price_row_utc"],
            "request_id": entry["request_id"], "state_at_decision": record["state_before"],
            "decision_costs_at_decision": entry["decision_costs"], "reviewer_decision": entry["decision"],
            "cited_evidence_values": {p: _cost_pilot_evidence_value(payload, p) for p in entry["decision"]["evidence_fields"]},
            "observed_market_return": record["market_return"],
            "chosen_outcome": entry["one_step_counterfactuals"][entry["decision"]["action"]],
            "one_step_counterfactuals": entry["one_step_counterfactuals"],
            "one_step_gap_bps": max(0.0, (best_return - record["net_return"]) * 10000),
            "verified_reasoning_error": None,
            "reasoning_error_status": "Not assessed. A loss or hindsight gap alone does not establish an error.",
        })
    feedback = {
        "schema_version": 1, "collection_id": LLM_BUILD_COLLECTION_ID,
        "plan_hash": plan_hash, "final_step_hash": chain_hash, "mode": LLM_EXPERIMENT_MODE,
        "source_stage": "MEMORY_BUILD", "decision_count": len(cases), "memory_during_collection": "EMPTY",
        "latest_outcome_price_row_utc": max(c["outcome_price_row_utc"] for c in cases),
        "interpretation": {
            "purpose": "Offline candidate-memory development; never include this feedback in a decision prompt.",
            "alternatives": "One interval from the same actual account, assuming price does not depend on action.",
            "error_rule": "Hindsight returns support hypotheses, not guaranteed correct-action labels.",
            "chronology": "Any lesson must postdate all source outcomes used to generate it.",
            "evaluation": "No MEMORY_GATE or FROZEN_EVAL outcomes are included. Base-model leakage remains in this prototype.",
        },
        "cases": cases,
    }
    _build_collection_immutable(LLM_BUILD_FEEDBACK_PATH, feedback)
    ledger = pd.DataFrame([{
        "date": e["replay_record"]["source_timestamp_utc"][:10], "action": e["decision"]["action"],
        "target_exposure": e["replay_record"]["target_exposure"], "net_return": e["replay_record"]["net_return"],
        "equity_after": e["replay_record"]["equity_after"], "drawdown_frac": e["replay_record"]["drawdown_frac"],
    } for e in entries])
    print("\nMEMORY_BUILD EXPERIENCE COLLECTED — DEVELOPMENT DATA")
    print("Decisions:", len(entries), "| Action counts:", ledger["action"].value_counts().to_dict())
    print("First four reused from pilot:", sum(e["response_origin"] == "exact_input_pilot_reuse" for e in entries))
    print("New HTTP attempts this execution:", llm_request_status()["attempts"] - before["attempts"])
    print("Daily checkpoints:", LLM_BUILD_COLLECTION_DIR)
    print("Evaluator-only feedback:", LLM_BUILD_FEEDBACK_PATH)
    print("Final five days:")
    display(ledger.tail(5))
    print("Request accounting:", json.dumps(llm_request_status(), indent=2))
    print("No candidate lessons generated or activated. This collection will feed the reflection step.")
    return replay, ledger, feedback


if "LLM_BUILD_COLLECTION_LOCK" not in globals():
    LLM_BUILD_COLLECTION_LOCK = threading.Lock()
if not LLM_BUILD_COLLECTION_LOCK.acquire(blocking=False):
    raise RuntimeError("This collection is already running in the current kernel.")
try:
    try:
        LLM_BUILD_COLLECTION_REPLAY, LLM_BUILD_COLLECTION_LEDGER, LLM_BUILD_FEEDBACK = _run_build_collection()
        LLM_BUILD_COLLECTION_READY = True
    except BaseException:
        print("Collection paused. Completed daily files and received API responses remain saved.", flush=True)
        try:
            _build_collection_backup("paused run")
        except Exception as backup_error:
            print("Could not create a new ZIP:", type(backup_error).__name__, "| daily checkpoints remain on disk.", flush=True)
        raise
finally:
    LLM_BUILD_COLLECTION_LOCK.release()

LLM_BUILD_LATEST_BACKUP = _build_collection_backup("completed collection")
print("Block 20 complete. Download the backup. Next: candidate reflections from saved MEMORY_BUILD feedback.")


MEMORY_BUILD collection | 2023-10-01 to 2024-06-29 | restored 5/273 | memory: empty
Sequential account; daily atomic saves; ZIP every 25 days. Download ZIPs to retain them outside Kaggle.
[cache] memory_build_no_memory_costs_v2/2023-10-06T00:00:00+00:00/review | dbc4f508d2d3
[day 6/273 saved] 2023-10-06 | KEEP | target=+0.857 | equity=$96,213.40 | accounted API total=$0.0039 | ETA~0.1m
[request] memory_build_no_memory_costs_v2/2023-10-07T00:00:00+00:00/review | attempt 1 | 5e08d459eeb1
[saved] memory_build_no_memory_costs_v2/2023-10-07T00:00:00+00:00/review | 1.3s | accounted total $0.0044
[day 7/273 saved] 2023-10-07 | KEEP | target=-0.868 | equity=$96,013.48 | accounted API total=$0.0044 | ETA~3.0m
[request] memory_build_no_memory_costs_v2/2023-10-08T00:00:00+00:00/review | attempt 1 | b593d849569e
[saved] memory_build_no_memory_costs_v2/2023-10-08T00:00:00+00:00/review | 1.2s | accounted total $0.0048
[day 8/273 saved] 2023-10-08 | KEEP | target=-0.871 | equity=$98,763.40 | accounte

/kaggle/working/eth_llm_backup_20260929T122422_401753Z.zip

[request] memory_build_no_memory_costs_v2/2023-10-26T00:00:00+00:00/review | attempt 1 | ebe5097e9fcc
[saved] memory_build_no_memory_costs_v2/2023-10-26T00:00:00+00:00/review | 1.4s | accounted total $0.0123
[day 26/273 saved] 2023-10-26 | KEEP | target=-0.883 | equity=$113,621.19 | accounted API total=$0.0123 | ETA~5.2m
[request] memory_build_no_memory_costs_v2/2023-10-27T00:00:00+00:00/review | attempt 1 | e600063b9d92
[saved] memory_build_no_memory_costs_v2/2023-10-27T00:00:00+00:00/review | 1.1s | accounted total $0.0127
[day 27/273 saved] 2023-10-27 | KEEP | target=-0.876 | equity=$113,810.72 | accounted API total=$0.0127 | ETA~5.2m
[request] memory_build_no_memory_costs_v2/2023-10-28T00:00:00+00:00/review | attempt 1 | 62d55dded8ca
[saved] memory_build_no_memory_costs_v2/2023-10-28T00:00:00+00:00/review | 1.1s | accounted total $0.0131
[day 28/273 saved] 2023-10-28 | KEEP | target=+0.872 | equity=$114,569.87 | accounted API total=$0.0131 | ETA~5.1m
[request] memory_build_no_memor

/kaggle/working/eth_llm_backup_20260929T122456_351577Z.zip

[request] memory_build_no_memory_costs_v2/2023-11-20T00:00:00+00:00/review | attempt 1 | 37904133aa7a
[saved] memory_build_no_memory_costs_v2/2023-11-20T00:00:00+00:00/review | 1.0s | accounted total $0.0226
[day 51/273 saved] 2023-11-20 | KEEP | target=-0.883 | equity=$132,870.98 | accounted API total=$0.0226 | ETA~4.8m
[request] memory_build_no_memory_costs_v2/2023-11-21T00:00:00+00:00/review | attempt 1 | 3d5ce1a42c9f
[saved] memory_build_no_memory_costs_v2/2023-11-21T00:00:00+00:00/review | 1.3s | accounted total $0.0230
[day 52/273 saved] 2023-11-21 | KEEP | target=+0.869 | equity=$140,127.66 | accounted API total=$0.0230 | ETA~4.8m
[request] memory_build_no_memory_costs_v2/2023-11-22T00:00:00+00:00/review | attempt 1 | 712d34a9e85b
[saved] memory_build_no_memory_costs_v2/2023-11-22T00:00:00+00:00/review | 1.0s | accounted total $0.0234
[day 53/273 saved] 2023-11-22 | KEEP | target=+0.861 | equity=$139,994.57 | accounted API total=$0.0234 | ETA~4.8m
[request] memory_build_no_memor

...

[request] memory_build_no_memory_costs_v2/2024-06-07T00:00:00+00:00/review | attempt 1 | 9e262aeb8777
[saved] memory_build_no_memory_costs_v2/2024-06-07T00:00:00+00:00/review | 1.2s | accounted total $0.1055
[day 251/273 saved] 2024-06-07 | KEEP | target=+0.442 | equity=$521,537.16 | accounted API total=$0.1055 | ETA~0.5m
[request] memory_build_no_memory_costs_v2/2024-06-08T00:00:00+00:00/review | attempt 1 | bad586e8f14c
[saved] memory_build_no_memory_costs_v2/2024-06-08T00:00:00+00:00/review | 1.4s | accounted total $0.1059
[day 252/273 saved] 2024-06-08 | KEEP | target=-0.018 | equity=$521,115.44 | accounted API total=$0.1059 | ETA~0.5m
[request] memory_build_no_memory_costs_v2/2024-06-09T00:00:00+00:00/review | attempt 1 | bca9f6632bc4
[saved] memory_build_no_memory_costs_v2/2024-06-09T00:00:00+00:00/review | 1.0s | accounted total $0.1063
[day 253/273 saved] 2024-06-09 | KEEP | target=-0.214 | equity=$522,143.11 | accounted API total=$0.1063 | ETA~0.4m
[request] memory_build_no_me

,date,action,target_exposure,net_return,equity_after,drawdown_frac
268,2024-06-25,HALVE,-0.336789,0.001817,519129.734721,0.019703
269,2024-06-26,KEEP,-0.572990,-0.013168,512293.848317,0.032612
270,2024-06-27,KEEP,-0.667214,0.013673,519298.576964,0.019384
271,2024-06-28,KEEP,-0.500001,-0.000118,519237.358258,0.019500
272,2024-06-29,KEEP,-0.075756,-0.002098,518148.110548,0.021557


Request accounting: {
  "active_requests": 0,
  "saved_responses": 279,
  "attempts": 279,
  "usage_based_usd": 0.114643764,
  "reserved_or_uncertain_usd": 0,
  "accounted_usd": 0.114643764,
  "budget_target_usd": 10.0
}
No candidate lessons generated or activated. This collection will feed the reflection step.
[backup] completed collection | 583 files | /kaggle/working/eth_llm_backup_20260929T122958_823531Z.zip


/kaggle/working/eth_llm_backup_20260929T122958_823531Z.zip

Block 20 complete. Download the backup. Next: candidate reflections from saved MEMORY_BUILD feedback.


In [28]:
# BLOCK 21 — Parallel candidate-memory proposals from saved MEMORY_BUILD feedback.
# 14 chronological cases per batch, at most ONE proposal per batch (or NO_CHANGE).
# For 273 days: 20 requests, up to six in parallel; bounded HTTP retries may add attempts.
# Every response/result is saved. No active memory, trading replay, or gate/eval access.

import copy
import json
import math
import os
import threading
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from pathlib import Path
import pandas as pd
from IPython.display import display, FileLink

for _flag in ["LLM_BUILD_COLLECTION_READY", "LLM_REQUEST_CLIENT_READY", "LLM_LOCAL_CHECKPOINT_HELPERS_READY"]:
    if not globals().get(_flag, False):
        raise RuntimeError("Missing setup: " + _flag + ". Complete Block 20 first.")

LLM_BUILD_REFLECTIONS_READY = False
LLM_BUILD_REFLECTION_ID = "build_reflections_costs_v2_pass1"
LLM_BUILD_REFLECTION_DIR = Path(LLM_MEMORY_DIR) / "proposal_batches" / LLM_BUILD_REFLECTION_ID
LLM_BUILD_CANDIDATE_DIR = Path(LLM_MEMORY_DIR) / "candidates" / "build_v2"

LLM_BUILD_REFLECTION_PROMPT_V1 = """You propose reusable memory lessons for a frozen ETH trading reviewer.
This is offline development from historical MEMORY_BUILD experience. The reviewer
chooses KEEP (full model proposal), HALVE (half the proposal) or FLAT (zero exposure).
KEEP does not mean keeping its existing portfolio position. Python already supplies
entry costs for all actions. Model weights stay fixed; a memory is a text instruction.

Each case separates information available BEFORE a decision from OBSERVED feedback
after the following interval. Read every case in this chronological batch. Propose
at most ONE concise, actionable lesson, or NO_CHANGE if support is insufficient.
Do not invent an error or a lesson merely to fill the output schema.

Two allowed lesson types:
PROCEDURE: a prospective improvement in interpreting/cross-checking supplied evidence,
handling disagreements or distinguishing exposure, turnover and uncertainty.
RISK_HYPOTHESIS: a tentative adjustment to review behavior suggested by repeated
patterns in this batch. Cite at least two distinct supporting cases for this type.
Consecutive cases are dependent observations, not independent statistical replication.

Use only supplied fields. Cite counterexamples within the batch when present.
State when the instruction applies and what the reviewer should check or consider.
It may affect action choice, but it must remain within KEEP/HALVE/FLAT. Costs alone
do not establish the best trade. The known cost table should be interpreted directly;
do not ask the LLM to replace Python arithmetic with its own calculations.
Do not equate a same-direction position with zero or minimal turnover. MLP class
probabilities are not calibrated probabilities of profit. Market volatility is daily
and fractional; 0.01 means 1%. Sentiment and news are not supplied.

Observed one-step alternatives all start from the SAME actual portfolio that day.
They are simulator diagnostics, not full alternative-policy returns. A loss, or a
hindsight-winning alternative, does not prove the original action was wrong. Never
use realized future returns as if they were available before deciding. Do not claim
statistical significance, causal improvement, validated thresholds, or future profit.
The underlying base model has known leakage; these are pipeline-prototype data.
Do not memorize case IDs, dates, exact historical returns or equity levels in the
instruction. No outside knowledge of historical markets/events. No new features,
tools, sentiment or retraining. Do not propose a universal always-KEEP/HALVE/FLAT rule.
Do not claim an explanation error is verified unless its contradiction with supplied
decision-time facts is explicit; all proposals still require separate semantic review
and held-out validation. No proposed lesson is active yet.

Return ONLY JSON with exactly status, reason and candidate:
{"status":"PROPOSE","reason":"Why this lesson is worth testing.","candidate":{
"title":"Short title","lesson_type":"PROCEDURE",
"instruction":"When the relevant condition holds, check or consider ...",
"supporting_case_ids":["D001"],"counterexample_case_ids":[],
"expected_benefit":"The behavior it aims to improve.",
"limitation":"What is uncertain or when it may fail."}}
Use the case IDs actually supplied in this batch. Supporting IDs: 1-4 unique IDs,
or 2-4 for RISK_HYPOTHESIS. Counterexample IDs: 0-4 unique IDs, disjoint from support.
For NO_CHANGE return {"status":"NO_CHANGE","reason":"Why there is insufficient support.","candidate":null}.
Aim for reason <=250 characters, title <=80, instruction <=600, benefit <=200 and
limitation <=250. These are style targets, except instruction has a hard 1200-character
storage limit. Preserve uncertainty. No additional keys.
"""


def _br_immutable(path, value):
    with LLM_DISK_LOCK:
        if path.exists():
            if _llm_strict_json(path.read_text()) != value:
                raise RuntimeError("Saved artifact differs: " + path.name + ". Preserve it and inspect.")
        else:
            llm_atomic_write_json(path, value)


def _br_load_source():
    feedback = _llm_strict_json(LLM_BUILD_FEEDBACK_PATH.read_text())
    root = Path(LLM_BUILD_COLLECTION_DIR)
    plan = _llm_strict_json((root / "plan.json").read_text())
    progress = _llm_strict_json((root / "progress.json").read_text())
    cases = feedback["cases"]
    dates = pd.DatetimeIndex(pd.to_datetime([c["source_timestamp_utc"] for c in cases], utc=True))
    if (not progress["done"] or progress["next_index"] != len(cases)
            or feedback["decision_count"] != len(cases)
            or feedback["source_stage"] != "MEMORY_BUILD" or feedback["memory_during_collection"] != "EMPTY"
            or not dates.equals(pd.DatetimeIndex(LLM_MEMORY_BUILD_DATES))
            or not LLM_MEMORY_ASSIGNMENT.loc[dates].eq("MEMORY_BUILD").all()
            or feedback["plan_hash"] != _llm_digest(plan) or progress["plan_hash"] != feedback["plan_hash"]):
        raise RuntimeError("Expected complete, unchanged MEMORY_BUILD feedback and checkpoints.")
    chain = feedback["plan_hash"]
    for index, case in enumerate(cases):
        step = _llm_strict_json((root / "steps" / f"{index:06d}.json").read_text())
        record = step["replay_record"]
        if (step["index"] != index or step["plan_hash"] != feedback["plan_hash"]
                or step["previous_step_hash"] != chain or step["case_id"] != case["case_id"]
                or step["request_id"] != case["request_id"]
                or step["decision"] != case["reviewer_decision"]
                or step["decision_costs"] != case["decision_costs_at_decision"]
                or record["state_before"] != case["state_at_decision"]
                or record["source_timestamp_utc"] != case["source_timestamp_utc"]
                or record["outcome_price_row_utc"] != case["outcome_price_row_utc"]
                or record["market_return"] != case["observed_market_return"]
                or step["one_step_counterfactuals"] != case["one_step_counterfactuals"]
                or step["one_step_counterfactuals"][step["decision"]["action"]] != case["chosen_outcome"]
                or pd.Timestamp(case["outcome_price_row_utc"]) != dates[index] + pd.Timedelta(days=1)):
            raise RuntimeError("Feedback differs from its saved daily checkpoint at index " + str(index))
        chain = _llm_digest(step)
    if (chain != feedback["final_step_hash"] or chain != progress["last_step_hash"]
            or feedback["latest_outcome_price_row_utc"] != max(c["outcome_price_row_utc"] for c in cases)):
        raise RuntimeError("Final checkpoint/feedback fingerprint differs.")
    return feedback


def _br_compact_case(case, number):
    state = case["state_at_decision"]
    equity = float(state["portfolio"]["equity"])
    if not math.isfinite(equity) or equity <= 0:
        raise RuntimeError("Invalid source equity.")
    return {
        "case_id": f"D{number:03d}",
        "available_at_decision": {
            "state": {key: copy.deepcopy(state[key]) for key in ["model", "market", "portfolio", "data_quality_notes"]},
            "decision_costs": copy.deepcopy(case["decision_costs_at_decision"]),
        },
        "reviewer_decision": copy.deepcopy(case["reviewer_decision"]),
        "observed_after_decision": {
            "market_return_frac": case["observed_market_return"],
            "chosen_net_return_frac": case["chosen_outcome"]["net_return"],
            "chosen_drawdown_after_frac": case["chosen_outcome"]["drawdown_frac"],
            "one_step_alternatives_from_same_account": {
                action: {"target_exposure": row["target_exposure"],
                         "net_return_frac": row["net_return"],
                         "total_cost_equity_frac": row["total_cost"] / equity,
                         "terminal_liquidation": row["terminal"]}
                for action, row in case["one_step_counterfactuals"].items()
            },
        },
        "reasoning_error_status": "Not automatically established by a loss or hindsight comparison.",
    }


def _br_validate(data, batch):
    warnings = []
    if not isinstance(data, dict) or set(data) != {"status", "reason", "candidate"}:
        raise ValueError("Expected exactly status, reason and candidate.")
    if not isinstance(data["reason"], str) or not data["reason"].strip():
        raise ValueError("Reflection reason must be nonempty text.")
    if len(data["reason"].strip()) > 250:
        warnings.append("reason_exceeds_250_char_style_target")
    if data["status"] == "NO_CHANGE":
        if data["candidate"] is not None:
            raise ValueError("NO_CHANGE requires candidate=null.")
        return None, warnings
    if data["status"] != "PROPOSE":
        raise ValueError("Status must be PROPOSE or NO_CHANGE.")
    candidate = data["candidate"]
    text_targets = {"title": 80, "instruction": 600, "expected_benefit": 200, "limitation": 250}
    keys = set(text_targets) | {"lesson_type", "supporting_case_ids", "counterexample_case_ids"}
    if not isinstance(candidate, dict) or set(candidate) != keys:
        raise ValueError("Candidate has missing or unexpected fields.")
    for key, target in text_targets.items():
        if not isinstance(candidate[key], str) or not candidate[key].strip():
            raise ValueError(key + " must be nonempty text.")
        if len(candidate[key].strip()) > target:
            warnings.append(key + "_exceeds_style_target")
    if len(candidate["instruction"].strip()) > 1200:
        raise ValueError("Instruction exceeds the declared 1200-character storage limit.")
    if candidate["lesson_type"] not in ("PROCEDURE", "RISK_HYPOTHESIS"):
        raise ValueError("Unknown lesson_type.")
    available = {c["case_id"] for c in batch["cases"]}
    for key, minimum in [("supporting_case_ids", 2 if candidate["lesson_type"] == "RISK_HYPOTHESIS" else 1),
                         ("counterexample_case_ids", 0)]:
        ids = candidate[key]
        if (not isinstance(ids, list) or not minimum <= len(ids) <= 4
                or not all(isinstance(x, str) for x in ids) or len(set(ids)) != len(ids)
                or not set(ids).issubset(available)):
            raise ValueError(key + " must reference the required number of distinct cases in this batch.")
    if set(candidate["supporting_case_ids"]) & set(candidate["counterexample_case_ids"]):
        raise ValueError("Supporting and counterexample cases must be disjoint.")
    return copy.deepcopy(candidate), warnings


def _br_messages(batch):
    payload = {"batch_id": batch["batch_id"], "source_stage": "MEMORY_BUILD",
               "case_order": "chronological", "available_memories": [],
               "other_proposals": "Not supplied; this batch is independent.", "cases": batch["cases"]}
    return [{"role": "system", "content": LLM_BUILD_REFLECTION_PROMPT_V1},
            {"role": "user", "content": _llm_canonical(payload)}]


def _br_purpose(batch):
    return LLM_BUILD_REFLECTION_ID + "/" + _BR_FEEDBACK_HASH + "/" + batch["batch_id"]


def _br_cached(batch):
    with LLM_DISK_LOCK:
        matches = [copy.deepcopy(r) for r in LLM_REQUEST_CLIENT.records.values()
            if r["purpose"] == _br_purpose(batch) and r["payload"]["messages"] == _br_messages(batch)
            and r["payload"]["max_tokens"] == _BR_PLAN["max_tokens"]]
    if len(matches) != 1 or not matches[0]["attempts"] or matches[0]["attempts"][-1]["status"] != "response":
        return None
    return matches[0]


def _br_record(batch, cached):
    response = cached["attempts"][-1]["response"]
    raw_data, candidate, warnings, errors = None, None, [], []
    try:
        raw_data = LLM_REQUEST_CLIENT._result(cached, True)["data"]
    except RuntimeError as error:
        errors.append(str(error))
    if not errors:
        try:
            candidate, warnings = _br_validate(raw_data, batch)
        except ValueError as error:
            errors.append(str(error))
    status = "QUARANTINED" if errors else "PENDING_VALIDATION" if candidate is not None else "NO_CHANGE"
    artifact = None
    if status == "PENDING_VALIDATION":
        candidate_id = "mem_" + _llm_digest({"source": _BR_FEEDBACK_HASH,
            "batch_id": batch["batch_id"], "candidate": candidate})[:16]
        mapped_lesson = copy.deepcopy(candidate)
        for field in ["supporting_case_ids", "counterexample_case_ids"]:
            mapped_lesson[field] = [batch["source_case_map"][short_id]["case_id"] for short_id in candidate[field]]
        artifact = {
            "schema_version": 2, "candidate_id": candidate_id, "status": "PENDING_VALIDATION", "active": False,
            "origin": "independent_batch_reflection_v2", "source_stage": "MEMORY_BUILD",
            "source_feedback_hash": _BR_FEEDBACK_HASH, "proposal_plan_hash": _BR_PLAN_HASH,
            "batch_id": batch["batch_id"], "proposer_request_id": cached["request_id"],
            "latest_source_outcome_price_row_utc": batch["latest_outcome_price_row_utc"],
            "bundle_development_end_outcome_utc": _BR_FEEDBACK["latest_outcome_price_row_utc"],
            "temporal_rule": "No application to MEMORY_BUILD. This bundle is for subsequent validation/evaluation only.",
            "format_checked": True, "case_references_checked": "ID membership only; evidential support not verified.",
            "semantic_validation": "NOT_RUN", "held_out_validation": "NOT_RUN", "verified_error": False,
            "style_warnings": warnings, "lesson": mapped_lesson,
        }
    return {
        "schema_version": 1, "plan_hash": _BR_PLAN_HASH, "batch_id": batch["batch_id"],
        "request_id": cached["request_id"], "status": status, "validation_errors": errors,
        "style_warnings": warnings, "raw_proposal": raw_data, "candidate_artifact": artifact,
        "provider_model": response.get("model") if isinstance(response, dict) else None,
        "system_fingerprint": response.get("system_fingerprint") if isinstance(response, dict) else None,
    }


def _br_save(record):
    if record["candidate_artifact"] is not None:
        artifact = record["candidate_artifact"]
        _br_immutable(LLM_BUILD_CANDIDATE_DIR / (artifact["candidate_id"] + ".json"), artifact)
    _br_immutable(LLM_BUILD_REFLECTION_DIR / "results" / (record["batch_id"] + ".json"), record)


def _br_job(batch, stop):
    if stop.is_set():
        raise RuntimeError("Queued reflection paused before dispatch.")
    try:
        llm_json_request(_br_messages(batch), purpose=_br_purpose(batch), max_tokens=_BR_PLAN["max_tokens"])
    except RuntimeError:
        # A received invalid/truncated JSON response is preserved and quarantined.
        # A transport/budget/retry failure pauses dispatch instead of silently retrying.
        cached = _br_cached(batch)
        if cached is None:
            raise
        try:
            LLM_REQUEST_CLIENT._result(cached, True)
        except RuntimeError:
            pass
        else:
            raise
    cached = _br_cached(batch)
    if cached is None:
        raise RuntimeError("Completed reflection response is missing from the saved cache.")
    record = _br_record(batch, cached)
    _br_save(record)
    return record


def _run_build_reflections():
    _br_immutable(LLM_BUILD_REFLECTION_DIR / "plan.json", _BR_PLAN)
    records, pending = [], []
    for batch in _BR_BATCHES:
        path = LLM_BUILD_REFLECTION_DIR / "results" / (batch["batch_id"] + ".json")
        if not path.exists():
            pending.append(batch)
            continue
        cached = _br_cached(batch)
        if cached is None:
            raise RuntimeError("A saved reflection has no cached response. Restore the complete backup.")
        expected = _br_record(batch, cached)
        if _llm_strict_json(path.read_text()) != expected:
            raise RuntimeError("Saved reflection result differs from its plan/response.")
        _br_save(expected)
        records.append(expected)
    print(f"Candidate reflections | {len(_BR_BATCHES)} batches | restored {len(records)} | pending {len(pending)}", flush=True)
    print("No outcomes from MEMORY_GATE/FROZEN_EVAL are read. All candidate memories remain inactive.")
    before, started, new_done = llm_request_status(), time.monotonic(), 0
    if pending:
        stop = threading.Event()
        pool = ThreadPoolExecutor(max_workers=min(6, int(LLM_REQUEST_CLIENT.config["max_parallel_requests"])))
        futures = {pool.submit(_br_job, batch, stop) for batch in pending}
        try:
            while futures:
                finished, futures = wait(futures, timeout=5, return_when=FIRST_COMPLETED)
                if not finished:
                    print(f"[waiting] {len(records)}/{len(_BR_BATCHES)} saved | "
                          f"active={llm_request_status()['active_requests']} | elapsed={time.monotonic() - started:.0f}s", flush=True)
                for future in finished:
                    record = future.result()
                    records.append(record)
                    new_done += 1
                    eta = (time.monotonic() - started) / new_done * len(futures)
                    print(f"[batch {len(records)}/{len(_BR_BATCHES)} saved] {record['batch_id']} | "
                          f"{record['status']} | accounted API total=${llm_request_status()['accounted_usd']:.4f} | "
                          f"ETA~{eta:.0f}s", flush=True)
        except BaseException:
            stop.set()
            for future in futures:
                future.cancel()
            print("Reflection run paused. Saved results remain; in-flight requests may finish before shutdown.", flush=True)
            raise
        finally:
            pool.shutdown(wait=True, cancel_futures=True)
    records.sort(key=lambda r: r["batch_id"])
    candidates = [r["candidate_artifact"] for r in records if r["candidate_artifact"] is not None]
    instruction_groups = {}
    for candidate in candidates:
        key = _llm_digest(" ".join(candidate["lesson"]["instruction"].split()))
        instruction_groups.setdefault(key, []).append(candidate["candidate_id"])
    catalog = {
        "schema_version": 1, "plan_hash": _BR_PLAN_HASH, "source_feedback_hash": _BR_FEEDBACK_HASH,
        "status_counts": dict(Counter(r["status"] for r in records)),
        "candidate_ids": [c["candidate_id"] for c in candidates], "active_memory_changed": False,
        "exact_duplicate_instruction_groups": [ids for ids in instruction_groups.values() if len(ids) > 1],
        "semantic_review": "NOT_RUN", "held_out_validation": "NOT_RUN",
        "selection_rule": "Catalog contains this generation pass only; exclude retired pilot memories.",
    }
    _br_immutable(LLM_BUILD_REFLECTION_DIR / "catalog.json", catalog)
    table = pd.DataFrame([{
        "batch": r["batch_id"], "status": r["status"],
        "candidate_id": r["candidate_artifact"]["candidate_id"] if r["candidate_artifact"] else None,
        "type": r["candidate_artifact"]["lesson"]["lesson_type"] if r["candidate_artifact"] else None,
        "title": r["candidate_artifact"]["lesson"]["title"] if r["candidate_artifact"] else None,
        "style_warnings": len(r["style_warnings"]), "validation_errors": "; ".join(r["validation_errors"]),
    } for r in records])
    print("\nCANDIDATE PROPOSALS — INACTIVE, NOT VALIDATED")
    with pd.option_context("display.max_rows", 30, "display.max_colwidth", 100):
        display(table)
    print("Status counts:", catalog["status_counts"])
    print("Exact duplicate groups:", catalog["exact_duplicate_instruction_groups"])
    for candidate in candidates:
        lesson = candidate["lesson"]
        print("\n" + candidate["candidate_id"] + " | " + lesson["title"])
        print("Instruction:", lesson["instruction"])
        print("Support:", lesson["supporting_case_ids"])
        print("Counterexamples:", lesson["counterexample_case_ids"])
        print("Limitation:", lesson["limitation"])
    print("\nNew HTTP attempts this execution:", llm_request_status()["attempts"] - before["attempts"])
    print("Candidate folder:", LLM_BUILD_CANDIDATE_DIR)
    print("Catalog:", LLM_BUILD_REFLECTION_DIR / "catalog.json")
    print("Request accounting:", json.dumps(llm_request_status(), indent=2))
    return candidates, table, catalog


# Freeze batching and the proposal budget before dispatch. No adaptive retries
# to obtain a preferred lesson; a received invalid proposal stays quarantined.
_BR_FEEDBACK = _br_load_source()
_BR_FEEDBACK_HASH = _llm_digest(_BR_FEEDBACK)
_BR_BATCHES = []
for start in range(0, len(_BR_FEEDBACK["cases"]), 14):
    source = _BR_FEEDBACK["cases"][start:start + 14]
    _BR_BATCHES.append({
        "batch_id": f"B{len(_BR_BATCHES) + 1:03d}",
        "cases": [_br_compact_case(case, start + j + 1) for j, case in enumerate(source)],
        "source_case_map": {f"D{start + j + 1:03d}": {
            "case_id": case["case_id"], "source_timestamp_utc": case["source_timestamp_utc"],
            "outcome_price_row_utc": case["outcome_price_row_utc"]} for j, case in enumerate(source)},
        "latest_outcome_price_row_utc": max(c["outcome_price_row_utc"] for c in source),
    })
_BR_PLAN = {
    "schema_version": 1, "generation_id": LLM_BUILD_REFLECTION_ID, "source_feedback_hash": _BR_FEEDBACK_HASH,
    "source_collection_id": _BR_FEEDBACK["collection_id"], "batch_size": 14,
    "maximum_candidates": len(_BR_BATCHES), "batches": _BR_BATCHES,
    "system_prompt": LLM_BUILD_REFLECTION_PROMPT_V1,
    "request_client_config_hash": LLM_REQUEST_CLIENT.config_hash, "max_tokens": 1024,
    "format_policy": "Nonempty required text; style targets are warnings; instruction hard limit 1200 characters.",
    "invalid_response_policy": "Save and quarantine without regenerating; operational failures pause.",
    "validation_scope": "Structural checks and case-ID membership only. Claims remain hypotheses.",
    "time_policy": "Anonymized case IDs in prompts. No retrospective use of generated memories during MEMORY_BUILD.",
    "gate_policy": "No gate outcomes here. Freeze candidate text and declare selection rule before gate evaluation.",
}
_BR_PLAN_HASH = _llm_digest(_BR_PLAN)

if "LLM_BUILD_REFLECTION_LOCK" not in globals():
    LLM_BUILD_REFLECTION_LOCK = threading.Lock()
if not LLM_BUILD_REFLECTION_LOCK.acquire(blocking=False):
    raise RuntimeError("This reflection pass is already running in the current kernel.")
try:
    try:
        LLM_BUILD_CANDIDATES, LLM_BUILD_REFLECTION_TABLE, LLM_BUILD_REFLECTION_CATALOG = _run_build_reflections()
        LLM_BUILD_REFLECTIONS_READY = True
    except BaseException:
        try:
            _br_backup, _br_count = backup_llm_experiment()
            print("Paused-run backup:", _br_backup)
            display(FileLink(os.path.relpath(_br_backup, Path.cwd())))
        except Exception as error:
            print("Could not create a new ZIP:", type(error).__name__, "| saved files remain on disk.")
        raise
finally:
    LLM_BUILD_REFLECTION_LOCK.release()

_br_backup, _br_count = backup_llm_experiment()
print("\nDownload updated backup:", _br_backup)
display(FileLink(os.path.relpath(_br_backup, Path.cwd())))
print("Block 21 complete. Candidate lessons remain inactive. Next: inspect lessons and freeze the validation protocol.")


Candidate reflections | 20 batches | restored 0 | pending 20
No outcomes from MEMORY_GATE/FROZEN_EVAL are read. All candidate memories remain inactive.
[request] build_reflections_costs_v2_pass1/0a1cc43376e94284f789f8c1776046c3e2721ce818351ee65a03b7d2141ebe79/B001 | attempt 1 | c44e552f7fa1
[request] build_reflections_costs_v2_pass1/0a1cc43376e94284f789f8c1776046c3e2721ce818351ee65a03b7d2141ebe79/B004 | attempt 1 | cb647947e2ea
[request] build_reflections_costs_v2_pass1/0a1cc43376e94284f789f8c1776046c3e2721ce818351ee65a03b7d2141ebe79/B003 | attempt 1 | 2028df0d2f0b
[request] build_reflections_costs_v2_pass1/0a1cc43376e94284f789f8c1776046c3e2721ce818351ee65a03b7d2141ebe79/B005 | attempt 1 | 141b8de08923
[request] build_reflections_costs_v2_pass1/0a1cc43376e94284f789f8c1776046c3e2721ce818351ee65a03b7d2141ebe79/B002 | attempt 1 | df5c65862baf
[request] build_reflections_costs_v2_pass1/0a1cc43376e94284f789f8c1776046c3e2721ce818351ee65a03b7d2141ebe79/B006 | attempt 1 | aee83c98561c
[saved] 

,batch,status,candidate_id,type,title,style_warnings,validation_errors
0,B001,PENDING_VALIDATION,mem_49d00286d1d72bc1,PROCEDURE,Check proposal-position alignment before cutting exposure,2,
1,B002,PENDING_VALIDATION,mem_87f10ee6773977d3,PROCEDURE,"Check traded notional, not just entry cost, when citing KEEP",1,
2,B003,PENDING_VALIDATION,mem_704e3bfa02b7b72e,PROCEDURE,Distinguish reversal vs incremental adjustment before choosing action,2,
3,B004,PENDING_VALIDATION,mem_dfba94df3b89b4e5,PROCEDURE,Check position-to-target gap before accepting highest-cost KEEP,2,
4,B005,PENDING_VALIDATION,mem_067a3a71d9d18ea9,PROCEDURE,"Check traded notional and exposure change, not just cost rank",0,
5,B006,PENDING_VALIDATION,mem_6caaf8e5411fe1e9,PROCEDURE,"Treat lowest entry cost as tie-breaker, not directional evidence",0,
6,B007,PENDING_VALIDATION,mem_15a70a13839a729d,PROCEDURE,Cross-check PPO seed dispersion against MLP direction before sizing,1,
7,B008,PENDING_VALIDATION,mem_2a98387b0d0d769b,PROCEDURE,Check MLP confidence and cost when model is strongly directional,1,
8,B009,PENDING_VALIDATION,mem_d36819b66a2eaeee,PROCEDURE,Distinguish aligned KEEP from reversal KEEP before citing low cost,2,
9,B010,PENDING_VALIDATION,mem_264bc70bb72fbe2d,PROCEDURE,Consistent treatment of extreme RSI when model signal is strong,3,


Status counts: {'PENDING_VALIDATION': 20}
Exact duplicate groups: []

mem_49d00286d1d72bc1 | Check proposal-position alignment before cutting exposure
Instruction: When the model ensemble proposal is close to the current portfolio position (same sign and similar magnitude), consider that KEEP may already be the lowest-turnover way to express the model view, and check decision_costs.actions.KEEP.entry_cost_equity_frac and traded_notional_usd against HALVE/FLAT before reducing exposure. If the proposal is near the current position, cutting to HALVE or FLAT typically requires trading away from the model target and can incur higher entry cost. This does not override a clear disagreement between model components or a materially different proposal; it only flags that alignment plus low KEEP cost is evidence against unnecessary turnover.
Support: ['memory_build_no_memory_costs_v2/2023-10-08T00:00:00+00:00', 'memory_build_no_memory_costs_v2/2023-10-09T00:00:00+00:00', 'memory_build_no_memory_c

/kaggle/working/eth_llm_backup_20260929T124438_812472Z.zip

Block 21 complete. Candidate lessons remain inactive. Next: inspect lessons and freeze the validation protocol.


In [30]:
# BLOCK 22 — Inspect the evidence behind candidate memories.
# No LLM calls. Reads MEMORY_BUILD only; creates an evaluator-only review packet.
# Candidates are neither edited nor activated. This is NOT the held-out gate.

import copy
import json
import math
import os
from pathlib import Path
import pandas as pd
from IPython.display import display, FileLink

for _flag in ["LLM_BUILD_REFLECTIONS_READY", "LLM_LOCAL_CHECKPOINT_HELPERS_READY"]:
    if not globals().get(_flag, False):
        raise RuntimeError("Complete Block 21 first: missing " + _flag)
if llm_request_status()["active_requests"]:
    raise RuntimeError("Wait for the current requests to finish before creating this audit.")

LLM_CANDIDATE_EVIDENCE_READY = False
_b22_before = copy.deepcopy(llm_request_status())
_b22_feedback = _br_load_source()  # Checks Block 20's complete daily checkpoint chain.
_b22_plan = _llm_strict_json((LLM_BUILD_REFLECTION_DIR / "plan.json").read_text())
_b22_catalog = _llm_strict_json((LLM_BUILD_REFLECTION_DIR / "catalog.json").read_text())
_b22_plan_hash = _llm_digest(_b22_plan)
if (_b22_plan_hash != _BR_PLAN_HASH or _b22_plan_hash != _b22_catalog["plan_hash"]
        or _llm_digest(_b22_feedback) != _b22_plan["source_feedback_hash"]
        or _b22_catalog["source_feedback_hash"] != _b22_plan["source_feedback_hash"]):
    raise RuntimeError("The collection and proposal files do not belong to the same saved run.")

_b22_manifest = _llm_strict_json(LLM_REVIEW_INPUT_MANIFEST_V2_PATH.read_text())
_b22_spec = _b22_manifest["cost_spec"]
_b22_rate = float(_b22_spec["fee_rate"]) + float(_b22_spec["slippage_rate"])
if (not math.isfinite(_b22_rate) or _b22_rate <= 0
        or _b22_spec["actions"] != {"KEEP": 1.0, "HALVE": 0.5, "FLAT": 0.0}):
    raise RuntimeError("This audit requires the saved positive, common fee/slippage model.")
_b22_tolerance = float(_b22_spec["cost_ranking_tolerance_usd"])


def _b22_close(actual, expected, label):
    actual, expected = float(actual), float(expected)
    if (not math.isfinite(actual) or not math.isfinite(expected)
            or not math.isclose(actual, expected, rel_tol=1e-10, abs_tol=1e-8)):
        raise RuntimeError("Saved cost arithmetic differs: " + label)


def _b22_case_facts(case):
    state, costs = case["state_at_decision"], case["decision_costs_at_decision"]
    portfolio, model = state["portfolio"], state["model"]
    equity = float(portfolio["equity"])
    position, proposal = float(portfolio["current_position"]), float(model["ensemble_proposal"])
    if not all(math.isfinite(x) for x in [equity, position, proposal]) or equity <= 0:
        raise RuntimeError("Invalid decision-time account in " + case["case_id"])
    _b22_close(costs["fee_rate"], _b22_spec["fee_rate"], "fee rate")
    _b22_close(costs["slippage_rate"], _b22_spec["slippage_rate"], "slippage rate")
    for action, multiplier in _b22_spec["actions"].items():
        row = costs["actions"][action]
        target = proposal * multiplier
        traded = abs(target * equity - position * equity)
        expected = {"target_exposure": target, "target_notional_usd": target * equity,
                    "traded_notional_usd": traded, "entry_cost_usd": _b22_rate * traded,
                    "entry_cost_equity_frac": _b22_rate * traded / equity}
        for key, value in expected.items():
            _b22_close(row[key], value, case["case_id"] + "/" + action + "/" + key)
    actions = costs["actions"]
    minimum = min(row["entry_cost_usd"] for row in actions.values())
    cheapest = sorted(action for action, row in actions.items()
                      if math.isclose(row["entry_cost_usd"], minimum, rel_tol=0, abs_tol=_b22_tolerance))
    if sorted(costs["lowest_entry_cost_actions"]) != cheapest:
        raise RuntimeError("Saved cost ranking differs in " + case["case_id"])
    seeds = [float(x) for x in model["ppo_seed_proposals"].values()]
    if not seeds or not all(math.isfinite(x) for x in seeds):
        raise RuntimeError("Missing or invalid PPO seed outputs.")
    mlp_direction = int(model["mlp_direction"])
    return {
        "case_id": case["case_id"], "date": case["source_timestamp_utc"][:10],
        "chosen_action": case["reviewer_decision"]["action"],
        "current_position": position, "ensemble_proposal": proposal,
        "reversal": position * proposal < 0,
        "MLP_direction": mlp_direction, "MLP_max_probability": float(model["mlp_max_probability"]),
        "MLP_neutral": mlp_direction == 0,
        "MLP_opposes_ensemble": mlp_direction * proposal < 0,
        "PPO_mixed_signs": min(seeds) < 0 < max(seeds),
        "PPO_seed_min": min(seeds), "PPO_seed_max": max(seeds),
        "RSI_14": state["market"]["rsi_14"],
        "daily_volatility_8d_frac": state["market"]["daily_volatility_8d_frac"],
        "lowest_entry_cost_actions": cheapest,
        "KEEP_entry_cost_usd": float(actions["KEEP"]["entry_cost_usd"]),
        "HALVE_entry_cost_usd": float(actions["HALVE"]["entry_cost_usd"]),
        "FLAT_entry_cost_usd": float(actions["FLAT"]["entry_cost_usd"]),
        "HALVE_more_costly_than_KEEP": (
            actions["HALVE"]["entry_cost_usd"] > actions["KEEP"]["entry_cost_usd"] + _b22_tolerance),
        "cost_arithmetic_checked": True,
    }


_b22_cases = {case["case_id"]: copy.deepcopy(case) for case in _b22_feedback["cases"]}
if len(_b22_cases) != len(_b22_feedback["cases"]):
    raise RuntimeError("Duplicate case IDs in the source collection.")
_b22_facts = {key: _b22_case_facts(case) for key, case in _b22_cases.items()}
_b22_candidates, _b22_summary = [], []
for _b22_batch in _b22_plan["batches"]:
    _b22_result_path = LLM_BUILD_REFLECTION_DIR / "results" / (_b22_batch["batch_id"] + ".json")
    _b22_result = _llm_strict_json(_b22_result_path.read_text())
    _b22_cached_response = _br_cached(_b22_batch)
    if (_b22_cached_response is None
            or _b22_result != _br_record(_b22_batch, _b22_cached_response)):
        raise RuntimeError("Saved proposal differs from its cached response: " + _b22_batch["batch_id"])
    _b22_candidate = _b22_result["candidate_artifact"]
    if _b22_candidate is None:
        continue
    _b22_id = _b22_candidate["candidate_id"]
    _b22_on_disk = _llm_strict_json((LLM_BUILD_CANDIDATE_DIR / (_b22_id + ".json")).read_text())
    if (_b22_candidate != _b22_on_disk or _b22_candidate["active"] is not False
            or _b22_candidate["status"] != "PENDING_VALIDATION"):
        raise RuntimeError("Expected the unchanged inactive candidate: " + _b22_id)
    _b22_lesson = _b22_candidate["lesson"]
    _b22_support = _b22_lesson["supporting_case_ids"]
    _b22_counter = _b22_lesson["counterexample_case_ids"]
    _b22_entire_batch = [x["case_id"] for x in _b22_batch["source_case_map"].values()]
    if (not set(_b22_support + _b22_counter).issubset(_b22_entire_batch)
            or not set(_b22_entire_batch).issubset(_b22_cases)):
        raise RuntimeError("Candidate references do not belong to its source batch.")
    _b22_candidates.append({
        "candidate": copy.deepcopy(_b22_candidate),
        "proposer_reason": _b22_result["raw_proposal"]["reason"],
        "semantic_review_status": "NEEDS_REVIEW", "gate_admission": "NOT_DECIDED",
        "support_facts": [_b22_facts[x] for x in _b22_support],
        "counterexample_facts": [_b22_facts[x] for x in _b22_counter],
        "all_batch_case_ids": _b22_entire_batch,
        "uncited_batch_case_ids": [x for x in _b22_entire_batch if x not in _b22_support + _b22_counter],
    })
    _b22_summary.append({
        "batch": _b22_batch["batch_id"], "candidate_id": _b22_id,
        "type": _b22_lesson["lesson_type"], "title": _b22_lesson["title"],
        "support_cases": len(_b22_support), "counterexamples": len(_b22_counter),
        "support_reversals": sum(_b22_facts[x]["reversal"] for x in _b22_support),
        "support_HALVE_more_costly": sum(_b22_facts[x]["HALVE_more_costly_than_KEEP"] for x in _b22_support),
        "support_PPO_mixed": sum(_b22_facts[x]["PPO_mixed_signs"] for x in _b22_support),
        "support_MLP_neutral": sum(_b22_facts[x]["MLP_neutral"] for x in _b22_support),
    })
if [x["candidate"]["candidate_id"] for x in _b22_candidates] != _b22_catalog["candidate_ids"]:
    raise RuntimeError("Candidate list differs from the saved catalog.")

LLM_CANDIDATE_EVIDENCE_PACKET = {
    "schema_version": 1, "audit_id": "candidate_evidence_audit_v1",
    "experiment_mode": "PIPELINE_PROTOTYPE", "source_stage": "MEMORY_BUILD",
    "source_feedback_hash": _llm_digest(_b22_feedback), "proposal_plan_hash": _b22_plan_hash,
    "scope": "Mechanical cost checks and an evidence packet for human semantic review. No automatic acceptance.",
    "active_memory_changed": False, "held_out_validation": "NOT_RUN",
    "cost_spec": _b22_spec,
    "verified_protocol_facts": [
        "Entry cost equals a positive common rate times traded notional; their rankings are identical.",
        "Smallest among three entry costs does not mean zero trading or a small absolute trade.",
        "For nonzero opposite-sign position/proposal, traded notional is FLAT < HALVE < KEEP.",
        "HALVE halves the model target, not the existing account; it need not minimize turnover.",
    ],
    "semantic_review_questions": [
        "Do the cited decision-time states actually meet the instruction's conditions? Check uncited batch cases too.",
        "Are cost/turnover claims consistent with the fixed protocol and supplied action table?",
        "Does the lesson separate MLP direction, MLP class confidence and PPO seed agreement?",
        "Are numerical thresholds tentative hypotheses, rather than purported validated cutoffs?",
        "Does it introduce a useful procedure beyond instructions already in the reviewer/proposer prompts?",
        "Does it repeat or conflict with another candidate? Exact-string uniqueness does not resolve this.",
        "Are any claims supported only by hindsight returns? A loss alone does not prove a review error.",
    ],
    "reviewer_harness_at_collection": _b22_manifest,
    "proposer_prompt": _b22_plan["system_prompt"],
    "proposal_catalog": _b22_catalog, "summary": _b22_summary, "candidates": _b22_candidates,
    # All original build cases are retained so reviewers can inspect uncited evidence.
    # This packet contains observed outcomes; never insert it into a decision prompt.
    "evaluator_only_source_cases": _b22_cases,
    "decision_time_facts": _b22_facts,
}
LLM_CANDIDATE_EVIDENCE_PATH = (Path(LLM_API_RUN_DIR) / "evaluator_only"
                               / "candidate_evidence_audit_v1.json")
_br_immutable(LLM_CANDIDATE_EVIDENCE_PATH, LLM_CANDIDATE_EVIDENCE_PACKET)
LLM_CANDIDATE_EVIDENCE_TABLE = pd.DataFrame(_b22_summary)
LLM_CANDIDATE_EVIDENCE_READY = True

print("CANDIDATE EVIDENCE AUDIT — SEMANTIC REVIEW STILL REQUIRED")
print("Build cases checked:", len(_b22_cases), "| Inactive candidates:", len(_b22_candidates))
print("All saved entry-cost arithmetic and rankings match the fixed replay protocol.")
print("The support columns describe cited cases; they do not automatically confirm or reject a lesson.")
with pd.option_context("display.max_rows", 30, "display.max_columns", None, "display.max_colwidth", 75):
    display(LLM_CANDIDATE_EVIDENCE_TABLE)
print("\nImportant: HALVE means half the model proposal, not half your existing position.")
print("Example: current +0.80, proposal +1.00: KEEP trades 0.20 of equity; HALVE trades 0.30.")
print("Lowest entry cost and lowest traded notional have the same ranking under this protocol.")
print("No candidate text was edited; no candidate was admitted to the gate or activated.")
print("New HTTP attempts:", llm_request_status()["attempts"] - _b22_before["attempts"])
print("\nDownload and share this evidence packet:", LLM_CANDIDATE_EVIDENCE_PATH)
display(FileLink(os.path.relpath(LLM_CANDIDATE_EVIDENCE_PATH, Path.cwd())))
_b22_backup, _b22_count = backup_llm_experiment()
print("Download updated backup:", _b22_backup)
display(FileLink(os.path.relpath(_b22_backup, Path.cwd())))
print("Block 22 complete. Next: review candidate claims, then freeze the gate selection protocol.")


CANDIDATE EVIDENCE AUDIT — SEMANTIC REVIEW STILL REQUIRED
Build cases checked: 273 | Inactive candidates: 20
All saved entry-cost arithmetic and rankings match the fixed replay protocol.
The support columns describe cited cases; they do not automatically confirm or reject a lesson.


,batch,candidate_id,type,title,support_cases,counterexamples,support_reversals,support_HALVE_more_costly,support_PPO_mixed,support_MLP_neutral
0,B001,mem_49d00286d1d72bc1,PROCEDURE,Check proposal-position alignment before cutting exposure,4,2,0,4,0,0
1,B002,mem_87f10ee6773977d3,PROCEDURE,"Check traded notional, not just entry cost, when citing KEEP",4,2,4,0,0,0
2,B003,mem_704e3bfa02b7b72e,PROCEDURE,Distinguish reversal vs incremental adjustment before choosing action,4,1,3,0,0,0
3,B004,mem_dfba94df3b89b4e5,PROCEDURE,Check position-to-target gap before accepting highest-cost KEEP,4,2,0,4,0,0
4,B005,mem_067a3a71d9d18ea9,PROCEDURE,"Check traded notional and exposure change, not just cost rank",4,2,4,0,3,1
5,B006,mem_6caaf8e5411fe1e9,PROCEDURE,"Treat lowest entry cost as tie-breaker, not directional evidence",4,1,2,2,4,4
6,B007,mem_15a70a13839a729d,PROCEDURE,Cross-check PPO seed dispersion against MLP direction before sizing,4,2,4,0,3,3
7,B008,mem_2a98387b0d0d769b,PROCEDURE,Check MLP confidence and cost when model is strongly directional,4,2,3,0,0,0
8,B009,mem_d36819b66a2eaeee,PROCEDURE,Distinguish aligned KEEP from reversal KEEP before citing low cost,3,2,2,1,0,0
9,B010,mem_264bc70bb72fbe2d,PROCEDURE,Consistent treatment of extreme RSI when model signal is strong,2,0,2,0,0,0



Important: HALVE means half the model proposal, not half your existing position.
Example: current +0.80, proposal +1.00: KEEP trades 0.20 of equity; HALVE trades 0.30.
Lowest entry cost and lowest traded notional have the same ranking under this protocol.
No candidate text was edited; no candidate was admitted to the gate or activated.
New HTTP attempts: 0

Download and share this evidence packet: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/evaluator_only/candidate_evidence_audit_v1.json


/kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/evaluator_only/candidate_evidence_audit_v1.json

Download updated backup: /kaggle/working/eth_llm_backup_20260929T130332_407850Z.zip


/kaggle/working/eth_llm_backup_20260929T130332_407850Z.zip

Block 22 complete. Next: review candidate claims, then freeze the gate selection protocol.


In [37]:
# BLOCK 23 — Record the assisted semantic screen and freeze the gate protocol.
# No API calls, replay, gate outcomes, final-evaluation outcomes, or memory activation.
# This screen was prepared from the Block 22 packet uploaded for review.
# It is an assisted development decision, NOT an autonomous learned validator.

import copy
import json
import math
import os
from collections import Counter
from pathlib import Path
import pandas as pd
from IPython.display import display, FileLink

for _flag in ["LLM_CANDIDATE_EVIDENCE_READY", "LLM_MEMORY_PERIODS_READY",
              "LLM_REQUEST_CLIENT_READY", "LLM_LOCAL_CHECKPOINT_HELPERS_READY"]:
    if not globals().get(_flag, False):
        raise RuntimeError("Missing setup: " + _flag + ". Complete Block 22 first.")
if llm_request_status()["active_requests"]:
    raise RuntimeError("Wait until current API work finishes.")
LLM_GATE_PROTOCOL_READY = False
_p23_packet = _llm_strict_json(LLM_CANDIDATE_EVIDENCE_PATH.read_text())
_p23_expected_packet_hash = "1a834da501c036f4a330d986739f785a624452639f503647ce0dc0526b437167"
if _llm_digest(_p23_packet) != _p23_expected_packet_hash:
    raise RuntimeError("This is not the exact evidence packet reviewed. Preserve it and share the changed packet.")
if _p23_packet["source_stage"] != "MEMORY_BUILD" or _p23_packet["held_out_validation"] != "NOT_RUN":
    raise RuntimeError("Expected the development-only packet before held-out validation.")

# These annotations are separate from the original generated instructions.
# ADMIT means eligible to be tested, not demonstrated useful or active.
_p23_notes = {
    "B001": ("ADMIT_TO_GATE", "PROCEDURE", "position_target_costs",
        "All four cited alignment cases have KEEP as the cheapest action. The instruction explicitly checks the supplied table. "
        "Retain the earliest candidate in this overlapping cost-procedure family. This largely elaborates existing harness guidance; novelty and utility are unproved."),
    "B002": ("EXCLUDE_SEMANTIC", "PROCEDURE", "position_target_costs",
        "The proposer misdescribes its evidence: every cited support is a reversal with KEEP most expensive. "
        "On 2023-10-17 the reviewer chose HALVE and identified FLAT as cheapest; on 2023-10-26 it explicitly called KEEP highest-cost. "
        "The supposed lowest-cost reversal rationale is not supported by those cases."),
    "B003": ("EXCLUDE_REDUNDANT", "PROCEDURE", "position_target_costs",
        "Overlaps the B001 instruction to compare current position, model target and supplied costs. "
        "The cited 2023-11-07 state is a same-direction resize rather than a reversal. Do not add another overlapping cost instruction in this first gate."),
    "B004": ("EXCLUDE_SEMANTIC", "PROCEDURE", "position_target_costs",
        "The instruction says halving/flattening near an aligned target adds cost without materially changing exposure. "
        "For cited proposals near +/-0.86, HALVE changes target magnitude by about 0.43 and FLAT removes it. The exposure claim is misleading."),
    "B005": ("EXCLUDE_SEMANTIC", "PROCEDURE", "position_target_costs",
        "Its opposite-sign/cheapest-KEEP scenario conflicts with the fixed proportional-cost protocol. "
        "All four cited reversals have FLAT < HALVE < KEEP costs; the 2023-11-28 and 2023-12-06 reviewers explicitly identify KEEP as highest-cost."),
    "B006": ("DEFER_SUPPORT_OR_AMBIGUITY", "PROCEDURE", "cost_vs_signal",
        "Separating cost from directional evidence is sensible, but the cited explanations do not clearly establish the claimed conflation. "
        "On 2023-12-22 the reviewer explicitly calls KEEP highest-cost. On 2023-12-23 the observable error is reversed cost ordering. "
        "A general instruction already present in the harness is not by itself an experience-derived correction."),
    "B007": ("EXCLUDE_SEMANTIC", "RISK_HYPOTHESIS", "component_disagreement",
        "The instruction generalizes that HALVE reduces traded notional and entry cost without conditioning on current position. "
        "That is not true in this protocol. The 2024-01-03 support also has unanimous short PPO seeds, not seed disagreement. "
        "B013 preserves the disagreement hypothesis without that unconditional cost claim."),
    "B008": ("ADMIT_TO_GATE", "RISK_HYPOTHESIS", "confidence_volatility_costs",
        "The listed states support testing strong proposals with moderate class confidence, volatility and action costs. "
        "Only 2024-01-18 among the four support cases is already near the HALVE target; the other three are reversals. "
        "Keep the instruction's conditional cost check. Its approximate 0.85/0.5 levels and 'elevated' volatility are unvalidated, and its benefit remains a hypothesis."),
    "B009": ("DEFER_SUPPORT_OR_AMBIGUITY", "PROCEDURE", "position_target_costs",
        "The instruction mixes absolute trade size with relative cost rank and groups reversals with same-direction resizing. "
        "KEEP can be cheapest while requiring a material same-direction resize, but cannot be cheapest in a nonzero reversal here. "
        "The cited reversal cases are highest-cost KEEP. Exclude this ambiguous overlapping version without rewriting it."),
    "B010": ("EXCLUDE_SEMANTIC", "PROCEDURE", "rsi_symmetry",
        "Both cited states have high RSI (90.4 and 89.7), with opposite proposed directions and different account states. "
        "They are not a matched overbought/oversold symmetry comparison. Different decisions alone do not establish inconsistent reasoning, "
        "and the instruction's prohibition on using RSI differently is unsupported."),
    "B011": ("EXCLUDE_REDUNDANT", "PROCEDURE", "position_target_costs",
        "The cited cost comparisons are consistent with the protocol, but this substantially repeats B001's position-target alignment check. "
        "Keep B001 as the earliest representative; preserve this original candidate for the raw-memory control."),
    "B012": ("DEFER_SUPPORT_OR_AMBIGUITY", "RISK_HYPOTHESIS", "reversal_sizing",
        "Only 2024-03-12 among its four supports is a reversal; March 13-15 are increases to an existing short. "
        "A sign change alone does not guarantee a large trade without specifying magnitudes. Defer the wording/support mismatch rather than silently repair it."),
    "B013": ("ADMIT_TO_GATE", "RISK_HYPOTHESIS", "component_disagreement",
        "The three cited cases show mixed PPO signs or MLP direction opposing PPO/ensemble direction. "
        "The rule proposes checking disagreement and reducing exposure without asserting HALVE always has lower turnover. "
        "A class score near 0.5 is not a calibrated risk threshold; whether disagreement warrants less exposure must be tested."),
    "B014": ("EXCLUDE_SEMANTIC", "RISK_HYPOTHESIS", "component_disagreement",
        "It treats a class probability near 0.5 as directional incoherence. Confidence and direction agreement are different. "
        "The 2024-04-13 support has ensemble magnitude only 0.043, inconsistent with the stated large-proposal premise. "
        "Do not turn that conflation into persistent guidance."),
    "B015": ("EXCLUDE_SEMANTIC", "RISK_HYPOTHESIS", "component_disagreement",
        "The parenthetical probability-near-0.5 condition is incorrectly described as disagreement. "
        "On 2024-04-24 MLP and every PPO seed are short; on 2024-04-18 all are long. "
        "Only the April 20 support has actual opposing MLP direction. B013 is the cleaner candidate for this theme."),
    "B016": ("EXCLUDE_SEMANTIC", "RISK_HYPOTHESIS", "extended_returns",
        "Three of four cited MLP maxima are 0.561, 0.577 and 0.680, above its proposed below-about-0.55 condition. "
        "It also generalizes that HALVE reduces turnover without the necessary position condition. "
        "The late/extended interpretation and numeric cutoff are not established by those citations."),
    "B017": ("EXCLUDE_REDUNDANT", "PROCEDURE", "position_target_costs",
        "The cost interpretation is broadly consistent, but repeats the position-target/cost comparison family represented by B001. "
        "Its May 13 counterexample is also a reversal with KEEP highest-cost, so it does not refute the factual cost relation. No separate gate trial for this near-duplicate."),
    "B018": ("ADMIT_TO_GATE", "RISK_HYPOTHESIS", "rsi_class_confidence",
        "The cited states support a tentative high-RSI/moderate-class-confidence sizing hypothesis. "
        "The exact 0.53-0.64 and 75-81 ranges appear tailored to a small batch and are not validated thresholds. "
        "The May 30 counterexample meets similar conditions but chose KEEP. Retain unchanged solely for an exploratory gate test; do not call it a verified procedure."),
    "B019": ("ADMIT_TO_GATE", "RISK_HYPOTHESIS", "neutral_mlp_exposure_change",
        "All three supports have neutral MLP and same-sign short PPO seeds. June 14/15 increase short exposure and June 18 reverses a long; "
        "HALVE is cheaper than KEEP in those states. June 19/20 show near-target positions with KEEP cheaper. "
        "Unanimity here means direction, not identical numeric actions. The conditional sizing/risk benefit remains unvalidated."),
    "B020": ("DEFER_SUPPORT_OR_AMBIGUITY", "PROCEDURE", "absolute_vs_relative_turnover",
        "June 29 illustrates a material resize even though KEEP is relatively cheapest. But the other cited support, June 25, "
        "chose HALVE and has KEEP highest-cost, contrary to the proposer reason. "
        "Defer the mixed support rather than treat the whole proposed explanation as verified."),
}
_p23_expected_ids = {
    "B001": "mem_49d00286d1d72bc1", "B002": "mem_87f10ee6773977d3",
    "B003": "mem_704e3bfa02b7b72e", "B004": "mem_dfba94df3b89b4e5",
    "B005": "mem_067a3a71d9d18ea9", "B006": "mem_6caaf8e5411fe1e9",
    "B007": "mem_15a70a13839a729d", "B008": "mem_2a98387b0d0d769b",
    "B009": "mem_d36819b66a2eaeee", "B010": "mem_264bc70bb72fbe2d",
    "B011": "mem_a7daf8db37b7e85a", "B012": "mem_e86a63b31e4d0b73",
    "B013": "mem_2f1deec24a38ffb4", "B014": "mem_a481872b16f045d6",
    "B015": "mem_9f8a15460d89f951", "B016": "mem_aeadf7cb6e1c219e",
    "B017": "mem_f92af694036a6224", "B018": "mem_3cd12e50ca1fbe48",
    "B019": "mem_48ba400c9ec40214", "B020": "mem_4409f8811601e767",
}
_p23_originals, _p23_reviews = {}, []
for _p23_item in _p23_packet["candidates"]:
    _p23_c = _p23_item["candidate"]
    _p23_b, _p23_id = _p23_c["batch_id"], _p23_c["candidate_id"]
    if _p23_expected_ids.get(_p23_b) != _p23_id:
        raise RuntimeError("An annotation would apply to a different candidate.")
    _p23_saved = _llm_strict_json((LLM_BUILD_CANDIDATE_DIR / (_p23_id + ".json")).read_text())
    if _p23_saved != _p23_c or _p23_c["active"] is not False:
        raise RuntimeError("Candidate changed or was already activated: " + _p23_id)
    _p23_originals[_p23_b] = copy.deepcopy(_p23_c)
    _p23_status, _p23_kind, _p23_family, _p23_note = _p23_notes[_p23_b]
    _p23_reviews.append({
        "batch_id": _p23_b, "candidate_id": _p23_id,
        "original_candidate_hash": _llm_digest(_p23_c),
        "original_lesson_type": _p23_c["lesson"]["lesson_type"],
        "review_classification": _p23_kind, "family": _p23_family,
        "screen_decision": _p23_status, "rationale": _p23_note,
        "instruction_edited": False, "held_out_utility": "NOT_TESTED",
    })
if set(_p23_originals) != set(_p23_notes):
    raise RuntimeError("The reviewed candidate list is incomplete.")

_p23_harness = _llm_strict_json(LLM_REVIEW_INPUT_MANIFEST_V2_PATH.read_text())
if (_p23_harness != _p23_packet["reviewer_harness_at_collection"]
        or LLM_REVIEW_PROMPT_V2 != _p23_harness["system_prompt"]):
    raise RuntimeError("Reviewer harness differs from the development run.")
_p23_validation_path = Path(LLM_API_RUN_DIR) / "harnesses" / "validation_policy_v2_reason_length_warning.json"
if not _p23_validation_path.is_file():
    raise RuntimeError("Run the saved Block 18B reason-length policy before freezing the gate.")
_p23_validation = _llm_strict_json(_p23_validation_path.read_text())
if (_p23_validation.get("policy_id") != "v2-reason-length-warning-v1"
        or globals().get("LLM_REVIEW_VALIDATION_POLICY_V2") != _p23_validation):
    raise RuntimeError("The common response-validation policy differs.")

_p23_gate_dates = pd.DatetimeIndex(LLM_MEMORY_GATE_DATES)
_p23_eval_dates = pd.DatetimeIndex(LLM_FROZEN_EVAL_DATES)
if (not _p23_gate_dates.equals(pd.date_range("2024-07-01", "2024-12-30", tz="UTC"))
        or not _p23_eval_dates.equals(pd.date_range("2025-01-01", "2026-09-24", tz="UTC"))
        or not LLM_MEMORY_ASSIGNMENT.loc[_p23_gate_dates].eq("MEMORY_GATE").all()
        or not LLM_MEMORY_ASSIGNMENT.loc[_p23_eval_dates].eq("FROZEN_EVAL").all()):
    raise RuntimeError("The periods differ from the experiment whose packet was reviewed.")
if pd.Timestamp(_p23_packet["evaluator_only_source_cases"][
        max(_p23_packet["evaluator_only_source_cases"])
    ]["outcome_price_row_utc"]) >= _p23_gate_dates.min():
    raise RuntimeError("Development feedback overlaps the gate start.")

_p23_ann = int(LLM_SETUP["ANN_DAYS"])
_p23_rf = float(LLM_SETUP["RF_ANNUAL"])
if _p23_ann != 365 or not math.isfinite(_p23_rf) or _p23_rf <= -1:
    raise RuntimeError("Unexpected annualization/risk-free configuration.")

_p23_admitted = [r["batch_id"] for r in _p23_reviews if r["screen_decision"] == "ADMIT_TO_GATE"]


def _p23_memories(batches):
    return [{"id": _p23_originals[b]["candidate_id"],
             "instruction": _p23_originals[b]["lesson"]["instruction"]} for b in batches]


_p23_conditions = [
    {"condition_id": "model_keep", "kind": "SCRIPTED", "action": "KEEP", "memories": [], "selectable": False},
    {"condition_id": "fixed_halve", "kind": "SCRIPTED", "action": "HALVE", "memories": [], "selectable": False},
    {"condition_id": "llm_no_memory", "kind": "LLM", "memories": [], "selectable": False},
    {"condition_id": "llm_raw_memory", "kind": "LLM", "memories": _p23_memories(sorted(_p23_originals)), "selectable": False},
    {"condition_id": "llm_screen_only", "kind": "LLM", "memories": _p23_memories(_p23_admitted), "selectable": False},
]
for _p23_b in _p23_admitted:
    _p23_conditions.append({"condition_id": "llm_single_" + _p23_b.lower(), "kind": "LLM",
                            "memories": _p23_memories([_p23_b]), "selectable": True})

LLM_MEMORY_SEMANTIC_SCREEN_V1 = {
    "schema_version": 1, "screen_id": "assisted_semantic_screen_v1",
    "reviewed_packet_hash": _p23_expected_packet_hash,
    "source_feedback_hash": _p23_packet["source_feedback_hash"],
    "origin": "Codex-assisted development review of candidate wording and MEMORY_BUILD evidence.",
    "scope": "Not an autonomous learned validator, proof of error-free text, or evidence of better trading.",
    "screen_basis": "Protocol contradictions, cited-state mismatches, wording ambiguity and overlapping instructions; no gate scores used.",
    "originals_preserved": True, "candidate_text_edited": False,
    "reviews": _p23_reviews, "admitted_batches": _p23_admitted,
    "admission_meaning": "Eligible for an exploratory replay test only. Generated original files remain inactive.",
}
LLM_GATE_PROTOCOL_V1 = {
    "schema_version": 1, "protocol_id": "memory_gate_v1", "mode": "PIPELINE_PROTOTYPE",
    "semantic_screen_hash": _llm_digest(LLM_MEMORY_SEMANTIC_SCREEN_V1),
    "reviewed_packet_hash": _p23_expected_packet_hash,
    "reviewer_harness_hash": _llm_digest(_p23_harness),
    "reviewer_system_prompt": _p23_harness["system_prompt"],
    "response_validation_policy": _p23_validation,
    "request_client_config_hash": LLM_REQUEST_CLIENT.config_hash,
    "api_config": copy.deepcopy(LLM_REQUEST_CLIENT.config),
    "replay_protocol": copy.deepcopy(LLM_REPLAY_PROTOCOL),
    "period_protocol": copy.deepcopy(LLM_MEMORY_PERIOD_PROTOCOL),
    "gate_source_dates": [d.isoformat() for d in _p23_gate_dates],
    "frozen_eval_source_dates": [d.isoformat() for d in _p23_eval_dates],
    "conditions": _p23_conditions,
    "memory_presentation": "All listed id/instruction pairs, in this frozen order, at every decision; no retrieval, rewrites or additions.",
    "account_rule": "Fresh identical initial cash per condition and stage; own sequential portfolio, own computed costs, terminal liquidation on the last date.",
    "base_proposals": "The same fixed original model proposal tape for every condition; not recomputed from the reviewer's account.",
    "response_rule": "Require a valid completed decision. Long reasons get the existing style warning. Other invalid/uncertain responses pause; no fallback or skipped dates.",
    "parallelism": "Up to six independent accounts in parallel; decisions sequential within an account; shared request budget guard.",
    "gate_metrics": {
        "annualization_days": _p23_ann, "risk_free_annual": _p23_rf,
        "risk_free_daily": (1 + _p23_rf) ** (1 / _p23_ann) - 1,
        "sharpe": "sqrt(365) * mean(net_daily_return - risk_free_daily) / sample_std(net_daily_return, ddof=1).",
        "undefined_sharpe": "If sample standard deviation <= 1e-12 or any value nonfinite, Sharpe is undefined and the condition is not eligible for selection.",
        "max_drawdown": "Maximum of 1 - equity/running_peak, including initial equity and terminal closing costs.",
        "total_log_growth": "log(final_equity / initial_equity).",
        "secondary": ["total_net_return", "annualized_volatility", "turnover_as_traded_notional_over_pretrade_equity",
                      "entry_plus_terminal_costs", "action_counts", "mean_absolute_target_exposure", "API_cost_and_tokens"],
        "cash_assumption": "No cash interest is credited by this simulator; RF is used only to calculate excess-return Sharpe.",
    },
    "selection": {
        "reference_condition": "llm_no_memory", "candidate_conditions": [c["condition_id"] for c in _p23_conditions if c["selectable"]],
        "required_complete_decisions": len(_p23_gate_dates),
        "minimum_sharpe_improvement": 0.10,
        "maximum_drawdown_increase_frac": 0.01,
        "minimum_positive_reference_log_growth_fraction": 0.90,
        "growth_rule": "If reference total_log_growth > 0, require candidate growth >= 0.90 * reference growth; otherwise require candidate growth >= reference growth.",
        "all_requirements_must_pass": True,
        "ranking": "Among eligible single-memory conditions: highest Sharpe, then lower max drawdown for an exact tie, then lexicographic condition_id.",
        "selected_memory_limit": 1,
        "no_pass_rule": "Keep empty memory; report that no candidate passed. If reference Sharpe is undefined, do not choose a memory by a substitute metric.",
        "incomplete_run_rule": "No selection until every scheduled condition finishes; do not drop failed candidates based on partial performance.",
        "interpretation": "Exploratory engineering selection thresholds chosen before gate scores; not statistical significance or a guarantee of generalization.",
    },
    "final_evaluation": {
        "conditions": ["model_keep", "fixed_halve", "llm_no_memory", "llm_raw_memory", "llm_screen_only", "llm_gate_selected"],
        "selected_condition": "One winning unchanged instruction, or empty memory if none passes. Never combine individually tested memories after seeing scores.",
        "null_memory_case": "If none passes, llm_gate_selected is the identical empty-memory policy; alias the no-memory trajectory rather than claim a new independent replicate.",
        "frozen": "No memory generation, prompt tuning, threshold changes, retrieval changes or selection on the 632 evaluation decisions.",
        "comparison_scope": "Raw versus screened distinguishes the assisted semantic screen; selected versus screen-only helps assess the additional gate and memory-size change. Report all conditions, including negative results.",
        "attribution_limit": "Screening is assisted and memory lengths differ. This design does not isolate an autonomous validator or a pure length-controlled validation effect.",
    },
    "scope_limitations": [
        "Known upstream future-label leakage and idealized execution timing remain; not an investable/OOS performance claim.",
        "The gate is selection data, not the final holdout. Five candidate comparisons create selection bias even with fixed thresholds.",
        "The proposer labeled all candidates PROCEDURE; this screen records four admitted sizing ideas as unvalidated RISK_HYPOTHESIS without rewriting their text.",
        "Many cost lessons elaborate instructions already in the prompt. Do not claim novel financial knowledge merely because text was persisted.",
    ],
}


def _p23_save_immutable(path, value):
    with LLM_DISK_LOCK:
        if path.exists():
            # Block 11 date ranges are tuples in RAM and lists in saved JSON.
            # Compare canonical JSON hashes without rewriting the frozen file.
            if _llm_digest(_llm_strict_json(path.read_text())) != _llm_digest(value):
                raise RuntimeError("Saved protocol differs. Preserve it; do not change a gate after inspecting results: " + path.name)
        else:
            llm_atomic_write_json(path, value)


LLM_MEMORY_SEMANTIC_SCREEN_PATH = Path(LLM_API_RUN_DIR) / "protocols" / "assisted_semantic_screen_v1.json"
LLM_GATE_PROTOCOL_PATH = Path(LLM_API_RUN_DIR) / "protocols" / "memory_gate_v1.json"
_p23_save_immutable(LLM_MEMORY_SEMANTIC_SCREEN_PATH, LLM_MEMORY_SEMANTIC_SCREEN_V1)
_p23_save_immutable(LLM_GATE_PROTOCOL_PATH, LLM_GATE_PROTOCOL_V1)
LLM_GATE_PROTOCOL_HASH = _llm_digest(LLM_GATE_PROTOCOL_V1)
LLM_GATE_PROTOCOL_READY = True
print("ASSISTED SEMANTIC SCREEN — UTILITY NOT YET TESTED")
print("Screen decisions:", dict(Counter(r["screen_decision"] for r in _p23_reviews)))
display(pd.DataFrame([{k: r[k] for k in ["batch_id", "candidate_id", "review_classification", "screen_decision"]}
                      for r in _p23_reviews]))
print("\nADMITTED FOR TESTING")
for _p23_b in _p23_admitted:
    print(_p23_b, "|", _p23_originals[_p23_b]["lesson"]["title"])
print("\nFROZEN GATE PLAN")
print("Dates:", _p23_gate_dates.min().date(), "to", _p23_gate_dates.max().date(), "|", len(_p23_gate_dates), "decisions/account")
print("Accounts:", len(_p23_conditions), "| LLM accounts:", sum(c["kind"] == "LLM" for c in _p23_conditions))
print("Planned LLM gate decisions:", len(_p23_gate_dates) * sum(c["kind"] == "LLM" for c in _p23_conditions))
print("Choose at most ONE of five candidates after a complete gate run.")
print("Requires Sharpe >= no-memory + 0.10, max drawdown <= no-memory + 1 percentage point,")
print("and >=90% of positive no-memory log growth (or no worse log growth if reference growth is nonpositive).")
print("No passing candidate means empty memory. Thresholds are exploratory, not significance tests.")
print("Screen-only and raw-memory controls will be retained in final evaluation.")
print("Protocol:", LLM_GATE_PROTOCOL_PATH)
print("Protocol hash:", LLM_GATE_PROTOCOL_HASH)
print("No API calls or gate/evaluation returns read. Candidate files remain unchanged and inactive.")
_p23_backup, _p23_count = backup_llm_experiment()
print("Download updated backup:", _p23_backup)
display(FileLink(os.path.relpath(_p23_backup, Path.cwd())))
print("Block 23 complete. Next: a resumable parallel MEMORY_GATE runner using this frozen protocol.")


ASSISTED SEMANTIC SCREEN — UTILITY NOT YET TESTED
Screen decisions: {'ADMIT_TO_GATE': 5, 'EXCLUDE_SEMANTIC': 8, 'EXCLUDE_REDUNDANT': 3, 'DEFER_SUPPORT_OR_AMBIGUITY': 4}


,batch_id,candidate_id,review_classification,screen_decision
0,B001,mem_49d00286d1d72bc1,PROCEDURE,ADMIT_TO_GATE
1,B002,mem_87f10ee6773977d3,PROCEDURE,EXCLUDE_SEMANTIC
2,B003,mem_704e3bfa02b7b72e,PROCEDURE,EXCLUDE_REDUNDANT
3,B004,mem_dfba94df3b89b4e5,PROCEDURE,EXCLUDE_SEMANTIC
4,B005,mem_067a3a71d9d18ea9,PROCEDURE,EXCLUDE_SEMANTIC
5,B006,mem_6caaf8e5411fe1e9,PROCEDURE,DEFER_SUPPORT_OR_AMBIGUITY
6,B007,mem_15a70a13839a729d,RISK_HYPOTHESIS,EXCLUDE_SEMANTIC
7,B008,mem_2a98387b0d0d769b,RISK_HYPOTHESIS,ADMIT_TO_GATE
8,B009,mem_d36819b66a2eaeee,PROCEDURE,DEFER_SUPPORT_OR_AMBIGUITY
9,B010,mem_264bc70bb72fbe2d,PROCEDURE,EXCLUDE_SEMANTIC



ADMITTED FOR TESTING
B001 | Check proposal-position alignment before cutting exposure
B008 | Check MLP confidence and cost when model is strongly directional
B013 | Cross-check model agreement before full exposure
B018 | Cross-check MLP conviction and RSI stretch before sizing strong proposals
B019 | Check proposal magnitude against recent position and volatility before full KEEP

FROZEN GATE PLAN
Dates: 2024-07-01 to 2024-12-30 | 183 decisions/account
Accounts: 10 | LLM accounts: 8
Planned LLM gate decisions: 1464
Choose at most ONE of five candidates after a complete gate run.
Requires Sharpe >= no-memory + 0.10, max drawdown <= no-memory + 1 percentage point,
and >=90% of positive no-memory log growth (or no worse log growth if reference growth is nonpositive).
No passing candidate means empty memory. Thresholds are exploratory, not significance tests.
Screen-only and raw-memory controls will be retained in final evaluation.
Protocol: /kaggle/working/llm_memory_experiment/llm_runs/

/kaggle/working/eth_llm_backup_20260929T135226_834806Z.zip

Block 23 complete. Next: a resumable parallel MEMORY_GATE runner using this frozen protocol.


In [40]:
# BLOCK 24 — Run the frozen MEMORY_GATE experiment with daily checkpoints.
# PAID API CALLS: 8 LLM accounts x 183 dates = 1,464 decisions before cache reuse/retries.
# Two scripted controls need no LLM. Up to six accounts run in parallel.
# ZIP backups at barriers every 25 dates; no requests are in flight at a barrier.
# No FROZEN_EVAL replay, candidate edits, reflections, selection or memory activation.
# Rerun this entire cell unchanged to verify checkpoints and resume.
# Serialization fix: compare frozen JSON hashes, allowing tuple/list round trips.
# Compact output: one updating progress line; request details go to a local log.
# Recorded amendment: >4 valid, distinct citations get a style warning.

import copy
import json
import math
import os
import re
import threading
import time
import types
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, FileLink

for _flag in ["LLM_GATE_PROTOCOL_READY", "LLM_COST_AWARE_REVIEW_READY", "LLM_REQUEST_CLIENT_READY",
              "LLM_REPLAY_ACCOUNTING_READY", "LLM_LOCAL_CHECKPOINT_HELPERS_READY"]:
    if not globals().get(_flag, False):
        raise RuntimeError("Missing setup: " + _flag + ". Complete Block 23 first.")
if "LLM_GATE_RUN_LOCK" not in globals():
    LLM_GATE_RUN_LOCK = threading.Lock()
if LLM_GATE_RUN_LOCK.locked() or llm_request_status()["active_requests"]:
    raise RuntimeError("Wait for the existing run to finish or stop before rerunning this cell.")

LLM_GATE_RUN_READY = False
LLM_GATE_RUN_DIR = Path(LLM_RUN_CHECKPOINT_DIR) / "memory_gate_v1"
LLM_GATE_RESULTS_PATH = Path(LLM_API_RUN_DIR) / "evaluator_only" / "memory_gate_v1_results.json"
LLM_GATE_BACKUP_INTERVAL = 25
LLM_GATE_PROGRESS_SECONDS = 10
LLM_GATE_DETAIL_LOG = Path(LLM_API_RUN_DIR) / "logs" / "memory_gate_v1.log"


def _g24_detail_print(*values, sep=" ", end="\n", **kwargs):
    # Existing request messages contain IDs/status, not API keys. Keep them out
    # of notebook output; the authoritative responses/checkpoints stay intact.
    line = sep.join(str(v) for v in values) + end
    with LLM_DISK_LOCK:
        LLM_GATE_DETAIL_LOG.parent.mkdir(parents=True, exist_ok=True)
        with LLM_GATE_DETAIL_LOG.open("a", encoding="utf-8") as stream:
            stream.write(time.strftime("[%Y-%m-%dT%H:%M:%SZ] ", time.gmtime()) + line)


def _g24_logged_copy(function):
    # Reuse the exact function bytecode with a LOCAL print sink. Do not change
    # builtins.print, sys.stdout, the shared client or other notebook cells.
    owner = getattr(function, "__self__", None)
    source = getattr(function, "__func__", function)
    if not isinstance(source, types.FunctionType):
        raise RuntimeError("Expected the existing Python request/validation function for compact logging.")
    namespace = dict(source.__globals__)
    namespace["print"] = _g24_detail_print
    local = types.FunctionType(source.__code__, namespace, source.__name__,
                               source.__defaults__, source.__closure__)
    local.__kwdefaults__ = copy.copy(source.__kwdefaults__)
    return types.MethodType(local, owner) if owner is not None else local


class _G24Progress:
    def __init__(self):
        self.handle = None
        self.started = False
        self.last_update = float("-inf")

    def show(self, line, *, force=False):
        now = time.monotonic()
        interval = LLM_GATE_PROGRESS_SECONDS if self.handle is not None else 30
        if not force and now - self.last_update < interval:
            return
        self.last_update = now
        if not self.started:
            # Kaggle updates this single display instead of appending lines.
            # Plain consoles fall back to a summary at most every 30 seconds.
            self.handle = display({"text/plain": line}, raw=True, display_id=True)
            self.started = True
        elif self.handle is not None:
            self.handle.update({"text/plain": line}, raw=True)
        else:
            print(line, flush=True)


def _g24_immutable(path, value):
    with LLM_DISK_LOCK:
        if path.exists():
            if _llm_digest(_llm_strict_json(path.read_text())) != _llm_digest(value):
                raise RuntimeError("Saved gate artifact differs: " + str(path) + ". Preserve it and inspect.")
        else:
            llm_atomic_write_json(path, value)


def _g24_check_protocol():
    protocol = _llm_strict_json(LLM_GATE_PROTOCOL_PATH.read_text())
    # JSON converts Block 11's date-range tuples to lists. Both representations
    # must match the existing frozen hash; never recompute that reference here.
    saved_hash = _llm_digest(protocol)
    current_hash = _llm_digest(LLM_GATE_PROTOCOL_V1)
    if (saved_hash != LLM_GATE_PROTOCOL_HASH or current_hash != LLM_GATE_PROTOCOL_HASH
            or protocol.get("protocol_id") != "memory_gate_v1"):
        raise RuntimeError(
            "The saved frozen gate protocol differs from the current setup. "
            f"Saved hash: {saved_hash}; in-memory hash: {current_hash}; "
            f"frozen hash: {LLM_GATE_PROTOCOL_HASH}. Preserve the files and inspect."
        )
    if (protocol["request_client_config_hash"] != LLM_REQUEST_CLIENT.config_hash
            or protocol["api_config"] != LLM_REQUEST_CLIENT.config
            or protocol["replay_protocol"] != LLM_REPLAY_PROTOCOL):
        raise RuntimeError("API or accounting settings changed after the protocol was frozen.")
    manifest = _cost_pilot_check_harness()
    if (_llm_digest(manifest) != protocol["reviewer_harness_hash"]
            or LLM_REVIEW_PROMPT_V2 != protocol["reviewer_system_prompt"]
            or LLM_REVIEW_VALIDATION_POLICY_V2 != protocol["response_validation_policy"]):
        raise RuntimeError("The reviewer prompt, costs or response-validation policy changed.")
    policy_path = Path(LLM_API_RUN_DIR) / "harnesses" / "validation_policy_v2_reason_length_warning.json"
    if _llm_strict_json(policy_path.read_text()) != protocol["response_validation_policy"]:
        raise RuntimeError("The saved response policy changed.")
    screen = _llm_strict_json(LLM_MEMORY_SEMANTIC_SCREEN_PATH.read_text())
    if _llm_digest(screen) != protocol["semantic_screen_hash"]:
        raise RuntimeError("The assisted screen differs from the frozen protocol.")
    dates = pd.DatetimeIndex(pd.to_datetime(protocol["gate_source_dates"], utc=True))
    if (not dates.equals(pd.DatetimeIndex(LLM_MEMORY_GATE_DATES))
            or len(dates) != protocol["selection"]["required_complete_decisions"]
            or not LLM_MEMORY_ASSIGNMENT.loc[dates].eq("MEMORY_GATE").all()):
        raise RuntimeError("Expected exactly the frozen MEMORY_GATE dates.")
    ids = [c["condition_id"] for c in protocol["conditions"]]
    if len(ids) != len(set(ids)) or any(not re.fullmatch(r"[a-z0-9_]+", x) for x in ids):
        raise RuntimeError("Invalid condition IDs.")
    for condition in protocol["conditions"]:
        if condition["kind"] not in {"SCRIPTED", "LLM"}:
            raise RuntimeError("Unknown gate condition kind.")
        if condition["kind"] == "SCRIPTED" and (condition["action"] not in LLMDailyReplay.ACTIONS or condition["memories"]):
            raise RuntimeError("Invalid scripted control.")
        for memory in condition["memories"]:
            candidate = _llm_strict_json((LLM_BUILD_CANDIDATE_DIR / (memory["id"] + ".json")).read_text())
            review = next((r for r in screen["reviews"] if r["candidate_id"] == memory["id"]), None)
            if (review is None or _llm_digest(candidate) != review["original_candidate_hash"]
                    or candidate["active"] is not False
                    or candidate["lesson"]["instruction"] != memory["instruction"]
                    or pd.Timestamp(candidate["bundle_development_end_outcome_utc"]) >= dates.min()):
                raise RuntimeError("Memory text, provenance or eligibility changed: " + memory["id"])
    return protocol, dates


def _g24_prepare_citation_amendment(protocol):
    # Preserve the original frozen protocol. Record the post-freeze change as
    # a separate immutable addendum, before resuming any decisions or scores.
    amendment = {
        "schema_version": 1,
        "amendment_id": "supplied-citation-count-style-v1",
        "base_protocol_hash": _llm_digest(protocol),
        "base_validation_policy_hash": _llm_digest(protocol["response_validation_policy"]),
        "origin": "Assistant-authored amendment after a gate response cited five valid supplied fields.",
        "timing": "Introduced after the gate began; report as a post-freeze validation amendment.",
        "trigger_request_id": "969b520e50f89144b2889dbd2586e10d52d98cc8bf7d1f57b1eece1e443c5acf",
        "requested_evidence_count": "1–4, unchanged in the prompt.",
        "hard_evidence_rule": "Nonempty list of distinct strings; every name must be in the request's allowed_evidence_fields.",
        "excess_count_rule": "More than four otherwise valid citations are accepted unchanged and recorded as a style warning.",
        "other_checks": "Exact four keys; KEEP/HALVE/FLAT; nonempty reason; distinct supplied memory IDs; completed JSON enforced by request client.",
        "application": "All gate conditions, including restored decisions. Carry the identical policy into every final-evaluation condition.",
        "preservation": "No prompt, response, action, memory, selection threshold, original protocol or cache identity edits; no truncation or regeneration.",
        "reporting": "Report excess-citation counts per condition. This is a harness implementation amendment, not learned memory or proof of rationale correctness.",
    }
    path = Path(LLM_API_RUN_DIR) / "protocols" / "memory_gate_v1_citation_count_amendment_v1.json"
    if LLM_GATE_RESULTS_PATH.exists() and not path.exists():
        raise RuntimeError("Gate results already exist under the original validation policy. Inspect before amending a completed run.")
    _g24_immutable(path, amendment)
    info = {
        "base_protocol_hash": _llm_digest(protocol),
        "amendment_id": amendment["amendment_id"],
        "amendment_hash": _llm_digest(amendment),
        "amendment_relative_path": "protocols/" + path.name,
    }
    info["effective_protocol_hash"] = _llm_digest({
        "base_protocol_hash": info["base_protocol_hash"],
        "validation_amendment_hash": info["amendment_hash"],
    })
    return info


def _g24_validate_with_citation_warning(data, payload):
    # Apply the original checks to ordinary responses, preserving existing
    # reason-length warnings. For excess citations, verify ALL fields and ALL
    # other requirements explicitly; never truncate the response to pass.
    if not isinstance(data, dict) or set(data) != {"action", "reason", "evidence_fields", "used_memory_ids"}:
        raise ValueError("Decision must have the four requested keys.")
    fields = data["evidence_fields"]
    if (not isinstance(fields, list) or not fields
            or not all(isinstance(x, str) for x in fields)
            or len(set(fields)) != len(fields)
            or not set(fields).issubset(payload["allowed_evidence_fields"])):
        raise ValueError("Cite distinct fields supplied to this review; empty, duplicate or unsupported citations are invalid.")
    if len(fields) <= 4:
        return _g24_base_validate(data, payload)
    if not isinstance(data["action"], str) or data["action"] not in LLM_REVIEW_COST_SPEC_V2["actions"]:
        raise ValueError("Action must be KEEP, HALVE or FLAT.")
    if not isinstance(data["reason"], str) or not data["reason"].strip():
        raise ValueError("Reason must be nonempty text.")
    ids = data["used_memory_ids"]
    if (not isinstance(ids, list) or not all(isinstance(x, str) for x in ids)
            or len(set(ids)) != len(ids)
            or not set(ids).issubset({m["id"] for m in payload["available_memories"]})):
        raise ValueError("A cited memory was not supplied to this review.")
    event = {
        "schema_version": 1,
        "event_type": "supplied_evidence_count_style_limit_exceeded",
        "validation_amendment_hash": LLM_GATE_VALIDATION_AMENDMENT["amendment_hash"],
        "source_timestamp_utc": payload["state"]["source_timestamp_utc"],
        "review_input_hash": _llm_digest(payload),
        "original_decision_hash": _llm_digest(data),
        "requested_max_fields": 4, "observed_fields": len(fields),
        "reason_characters": len(data["reason"].strip()),
        "reason_style_limit_also_exceeded": len(data["reason"].strip()) > 350,
        "original_response": copy.deepcopy(data), "response_modified": False,
        "structural_validation": "PASSED_UNDER_RECORDED_AMENDMENT",
        "meaning": "Formatting warning; all citations are supplied and distinct. No claim about trading utility or explanation truth.",
    }
    path = Path(LLM_API_RUN_DIR) / "events" / "citation_count_style_warnings" / (_llm_digest(event) + ".json")
    _g24_immutable(path, event)
    return copy.deepcopy(data)


def _g24_inputs(state, index, condition, dates, protocol):
    terminal = index == len(dates) - 1
    messages = build_review_messages_v2(state, terminal=terminal, available_memories=condition["memories"])
    payload = _llm_strict_json(messages[1]["content"])
    if (messages[0] != {"role": "system", "content": protocol["reviewer_system_prompt"]}
            or payload != build_review_payload_v2(state, terminal=terminal, available_memories=condition["memories"])
            or payload["available_memories"] != condition["memories"]):
        raise RuntimeError("Decision inputs differ from the frozen condition.")
    return messages, payload


def _g24_purpose(condition, date):
    return "memory_gate_v1/" + condition["condition_id"] + "/" + date.isoformat() + "/review"


def _g24_request_payload(messages, protocol):
    config = protocol["api_config"]
    return {"model": config["model"], "messages": copy.deepcopy(messages),
            "thinking": copy.deepcopy(config["thinking"]), "temperature": config["temperature"],
            "response_format": {"type": "json_object"},
            "max_tokens": int(config["decision_max_tokens"]), "stream": False}


def _g24_cached(messages, purpose, protocol):
    payload = _g24_request_payload(messages, protocol)
    key = _llm_digest({"config_hash": LLM_REQUEST_CLIENT.config_hash, "purpose": purpose, "payload": payload})
    with LLM_DISK_LOCK:
        record = copy.deepcopy(LLM_REQUEST_CLIENT.records.get(key))
    if record is None:
        return None, None
    if record["payload"] != payload or record["purpose"] != purpose or record["request_id"] != key:
        raise RuntimeError("Cached request identity differs.")
    if not record["attempts"]:
        return record, None
    status = record["attempts"][-1]["status"]
    if status == "response":
        return record, LLM_REQUEST_CLIENT._result(record, True)
    if status in {"started", "transport_unknown"}:
        raise RuntimeError("Uncertain delivery/billing for " + purpose + ". Reservation is retained; do not blindly resend.")
    if status != "retryable_http":
        raise RuntimeError("Previous HTTP failure for " + purpose + ". Inspect the saved request.")
    return record, None  # The existing client enforces Retry-After and attempt limits.


def _g24_scripted(condition):
    return {"action": condition["action"], "reason": "Fixed scripted control.",
            "evidence_fields": [], "used_memory_ids": []}


def _g24_snapshot(context, chain=None):
    replay = context["replay"]
    return {"plan_hash": context["plan_hash"], "next_index": replay._i,
            "last_step_hash": context["chain"] if chain is None else chain,
            "portfolio_after": copy.deepcopy(replay._portfolio), "done": replay.done}


def _g24_make_entry(context, decision, payload, result, record):
    return {"schema_version": 1, "index": len(context["entries"]),
            "plan_hash": context["plan_hash"], "previous_step_hash": context["chain"],
            "condition_id": context["condition"]["condition_id"],
            "decision_source": context["condition"]["kind"],
            "decision": copy.deepcopy(decision), "decision_costs": copy.deepcopy(payload["decision_costs"]),
            "memory_ids_supplied": [m["id"] for m in context["condition"]["memories"]],
            "request_id": result["request_id"] if result else None,
            "provider_model": result["provider_model"] if result else None,
            "system_fingerprint": result["system_fingerprint"] if result else None,
            "usage": copy.deepcopy(result["usage"]) if result else {},
            "replay_record": record}


def _g24_prepare_account(condition, protocol, dates, source_hash):
    root = LLM_GATE_RUN_DIR / condition["condition_id"]
    plan = {"schema_version": 1, "runner_version": "parallel-gate-v1", "stage": "MEMORY_GATE",
            "protocol_hash": _llm_digest(protocol), "source_fingerprint": source_hash,
            "condition": copy.deepcopy(condition), "dates": [d.isoformat() for d in dates]}
    _g24_immutable(root / "plan.json", plan)
    context = {"condition": condition, "root": root, "plan_hash": _llm_digest(plan),
               "replay": LLMDailyReplay(dates), "entries": [], "chain": _llm_digest(plan)}
    replay, entries = context["replay"], context["entries"]
    snapshots = [_g24_snapshot(context)]
    files = sorted((root / "steps").glob("*.json"))
    if len(files) > len(dates) or [p.name for p in files] != [f"{i:06d}.json" for i in range(len(files))]:
        raise RuntimeError("Gate daily files have gaps or unexpected names: " + condition["condition_id"])
    for index, path in enumerate(files):
        saved = _llm_strict_json(path.read_text())
        messages, payload = _g24_inputs(replay.get_state(), index, condition, dates, protocol)
        result = None
        if condition["kind"] == "LLM":
            _, result = _g24_cached(messages, _g24_purpose(condition, dates[index]), protocol)
            if result is None:
                raise RuntimeError("Completed gate step lacks its response cache. Restore the full backup.")
            decision = _g24_validate(result["data"], payload)
        else:
            decision = _g24_scripted(condition)
        record = replay.step(decision["action"])
        _cost_pilot_check_execution(payload, record)
        expected = _g24_make_entry(context, decision, payload, result, record)
        if saved != expected:
            raise RuntimeError("Restored gate step or accounting differs: " + condition["condition_id"] + "/" + path.name)
        context["chain"] = _llm_digest(saved)
        entries.append(saved)
        snapshots.append(_g24_snapshot(context))
    progress_path = root / "progress.json"
    if progress_path.exists():
        progress = _llm_strict_json(progress_path.read_text())
        n = progress.get("next_index")
        if type(n) is not int or not 0 <= n < len(snapshots) or progress != snapshots[n]:
            raise RuntimeError("Gate progress pointer differs from its daily files: " + condition["condition_id"])
    # A crash between the daily file and pointer write is repaired from the daily file.
    llm_atomic_write_json(progress_path, snapshots[-1])
    # Inspect a paid response received just before an earlier interruption before any
    # accounts dispatch more requests. Never automatically redo an invalid response.
    if not replay.done and condition["kind"] == "LLM":
        messages, payload = _g24_inputs(replay.get_state(), replay._i, condition, dates, protocol)
        _, pending = _g24_cached(messages, _g24_purpose(condition, dates[replay._i]), protocol)
        if pending is not None:
            _g24_validate(pending["data"], payload)
    _g24_detail_print(f"[restore] {condition['condition_id']} | {len(entries)}/{len(dates)} saved")
    return context


def _g24_advance(context, protocol, dates, end_index, stop):
    replay, condition = context["replay"], context["condition"]
    try:
        while replay._i < end_index and not stop.is_set():
            index = replay._i
            messages, payload = _g24_inputs(replay.get_state(), index, condition, dates, protocol)
            result = None
            if condition["kind"] == "LLM":
                result = _g24_request(messages, purpose=_g24_purpose(condition, dates[index]),
                                          max_tokens=int(protocol["api_config"]["decision_max_tokens"]))
                try:
                    decision = _g24_validate(result["data"], payload)
                except ValueError as error:
                    raise RuntimeError("Gate paused before execution: " + condition["condition_id"] + "/"
                                       + dates[index].isoformat() + " | " + str(error)
                                       + " | Received response remains cached: " + result["request_id"]) from None
            else:
                decision = _g24_scripted(condition)
            record = replay.step(decision["action"])
            _cost_pilot_check_execution(payload, record)
            entry = _g24_make_entry(context, decision, payload, result, record)
            chain = _llm_digest(entry)
            # Both writes use the shared disk lock; daily files remain authoritative
            # if a process/storage interruption occurs before the pointer update.
            with LLM_DISK_LOCK:
                _g24_immutable(context["root"] / "steps" / f"{index:06d}.json", entry)
                llm_atomic_write_json(context["root"] / "progress.json", _g24_snapshot(context, chain))
            context["chain"] = chain
            context["entries"].append(entry)
            if (index + 1) % 5 == 0 or index + 1 == end_index:
                _g24_detail_print(f"[saved] {condition['condition_id']} | {index + 1}/{len(dates)} | "
                                  f"{dates[index].date()} | {decision['action']}")
        return condition["condition_id"]
    except BaseException:
        stop.set()
        raise


def _g24_backup(label):
    global LLM_GATE_LATEST_BACKUP
    if llm_request_status()["active_requests"]:
        raise RuntimeError("A consistent gate backup requires all request workers to finish first.")
    backup, count = backup_llm_experiment()
    LLM_GATE_LATEST_BACKUP = backup
    print(f"[backup] {label} | {count} files | {backup}", flush=True)
    display(FileLink(os.path.relpath(backup, Path.cwd())))


def _g24_metrics(context, protocol):
    entries, replay = context["entries"], context["replay"]
    if not replay.done or len(entries) != len(replay.dates):
        raise RuntimeError("Do not score a partial account.")
    records = [e["replay_record"] for e in entries]
    returns = np.array([r["net_return"] for r in records], dtype=float)
    equity = np.array([records[0]["equity_before"]] + [r["equity_after"] for r in records], dtype=float)
    if (not np.isfinite(returns).all() or not np.isfinite(equity).all() or (equity <= 0).any()
            or not np.allclose(returns, equity[1:] / equity[:-1] - 1, rtol=1e-10, atol=1e-12)):
        raise RuntimeError("Invalid completed equity/return path.")
    cfg = protocol["gate_metrics"]
    std = float(returns.std(ddof=1)) if len(returns) > 1 else 0.0
    sharpe = (math.sqrt(cfg["annualization_days"]) * float((returns - cfg["risk_free_daily"]).mean()) / std
              if std > 1e-12 else None)
    if sharpe is not None and not math.isfinite(sharpe):
        sharpe = None
    mdd = float((1 - equity / np.maximum.accumulate(equity)).max())
    if not math.isclose(mdd, max(r["drawdown_frac"] for r in records), rel_tol=1e-10, abs_tol=1e-12):
        raise RuntimeError("Recorded drawdowns differ from the full equity path.")
    rate = float(protocol["replay_protocol"]["fee_rate"]) + float(protocol["replay_protocol"]["slippage_rate"])
    with LLM_DISK_LOCK:
        requests = [copy.deepcopy(LLM_REQUEST_CLIENT.records[e["request_id"]]) for e in entries if e["request_id"]]
    usage = [e["usage"] for e in entries if e["request_id"]]
    def token_total(key):
        values = [u.get(key) for u in usage]
        return sum(values) if all(type(v) is int and v >= 0 for v in values) else None
    return {
        "condition_id": context["condition"]["condition_id"], "kind": context["condition"]["kind"],
        "selectable": context["condition"]["selectable"], "decisions": len(records),
        "memory_count": len(context["condition"]["memories"]),
        "memory_ids": [m["id"] for m in context["condition"]["memories"]],
        "initial_equity": float(equity[0]), "final_equity": float(equity[-1]),
        "total_net_return": float(equity[-1] / equity[0] - 1),
        "total_log_growth": math.log(float(equity[-1] / equity[0])),
        "sharpe": sharpe, "max_drawdown_frac": mdd,
        "annualized_volatility": std * math.sqrt(cfg["annualization_days"]),
        "entry_cost_usd": sum(r["trade_cost"] for r in records),
        "terminal_cost_usd": sum(r["closing_cost"] for r in records),
        "total_cost_usd": sum(r["total_cost"] for r in records),
        "entry_turnover_sum_frac": sum(r["traded_notional"] / r["equity_before"] for r in records),
        "terminal_turnover_sum_frac": sum(r["closing_cost"] / rate / r["equity_before"] for r in records),
        "mean_absolute_target_exposure": float(np.mean([abs(r["target_exposure"]) for r in records])),
        "action_counts": {a: sum(r["action"] == a for r in records) for a in LLMDailyReplay.ACTIONS},
        "evidence_count_style_warnings": sum(len(e["decision"].get("evidence_fields", [])) > 4 for e in entries),
        "reason_length_style_warnings": sum(len(e["decision"].get("reason", "").strip()) > 350 for e in entries),
        "prompt_tokens": token_total("prompt_tokens"), "completion_tokens": token_total("completion_tokens"),
        "api_attempts": sum(len(r["attempts"]) for r in requests),
        "api_accounted_usd": sum(a["accounted_usd"] for r in requests for a in r["attempts"]),
        "api_cost_note": "Recorded request-client accounting, including retry/uncertain reservations where applicable; not a provider invoice.",
        "provider_models": sorted({str(e["provider_model"]) for e in entries if e["request_id"]}),
        "system_fingerprints": sorted({str(e["system_fingerprint"]) for e in entries if e["request_id"]}),
        "plan_hash": context["plan_hash"], "final_step_hash": context["chain"],
    }


def _run_memory_gate_v1():
    global _g24_request, _g24_validate, _g24_base_validate, LLM_GATE_VALIDATION_AMENDMENT
    protocol, dates = _g24_check_protocol()
    LLM_GATE_VALIDATION_AMENDMENT = _g24_prepare_citation_amendment(protocol)
    _g24_request = _g24_logged_copy(LLM_REQUEST_CLIENT.request)
    _g24_base_validate = _g24_logged_copy(validate_review_decision_v2)
    _g24_validate = _g24_logged_copy(_g24_validate_with_citation_warning)
    _g24_detail_print("[run] start/resume | frozen protocol " + LLM_GATE_PROTOCOL_HASH)
    # Private reproducibility fingerprint: reads ONLY gate rows. These outcome
    # values and this fingerprint are never included in any reviewer message.
    fresh = LLMDailyReplay(dates)
    rows = []
    for date in dates:
        outcome = LLM_EVALUATOR_DAILY.loc[date]
        exit_date = pd.Timestamp(outcome["exit_price_row_utc"])
        if not bool(outcome["eligible"]) or exit_date != date + pd.Timedelta(days=1):
            raise RuntimeError("Gate requires exact one-calendar-day outcomes.")
        rows.append({"state_with_fresh_account": build_reviewer_state(date, portfolio_state=fresh._portfolio),
                     "outcome_price_row_utc": exit_date.isoformat(), "market_return_1d": float(outcome["market_return_1d"])})
    source_hash = _llm_digest(rows)
    _g24_immutable(LLM_GATE_RUN_DIR / "run_manifest.json", {
        "runner_version": "parallel-gate-v1", "protocol_hash": _llm_digest(protocol),
        "source_fingerprint": source_hash, "source_stage": "MEMORY_GATE",
        "backup_barrier_dates": LLM_GATE_BACKUP_INTERVAL,
    })
    # Verify ALL saved accounts before any paid requests in this execution.
    contexts = [_g24_prepare_account(c, protocol, dates, source_hash) for c in protocol["conditions"]]
    total = len(contexts) * len(dates)
    restored = sum(len(c["entries"]) for c in contexts)
    before, started = llm_request_status(), time.monotonic()
    print(f"\nMEMORY_GATE | {len(contexts)} accounts | {restored}/{total} decisions restored", flush=True)
    print("Up to six independent accounts; daily saves; consistent ZIP at each 25-date barrier.")
    print("Ctrl+C pauses dispatch; in-flight requests finish/save before the paused-run backup.")
    print("No gate scores are displayed until all scheduled accounts finish.")
    print("Recorded validation amendment: >4 valid distinct citations are style warnings for every condition.")
    print("Compact progress; full request log:", LLM_GATE_DETAIL_LOG)
    progress = _G24Progress()
    progress.show(f"[progress] restored={restored}/{total} | preparing accounts", force=True)
    workers = min(6, int(protocol["api_config"]["max_parallel_requests"]))
    while any(not c["replay"].done for c in contexts):
        earliest = min(c["replay"]._i for c in contexts if not c["replay"].done)
        end_index = min((earliest // LLM_GATE_BACKUP_INTERVAL + 1) * LLM_GATE_BACKUP_INTERVAL, len(dates))
        jobs = [c for c in contexts if c["replay"]._i < end_index]
        stop = threading.Event()
        pool = ThreadPoolExecutor(max_workers=workers)
        pending = set()
        errors = []
        try:
            pending = {pool.submit(_g24_advance, c, protocol, dates, end_index, stop) for c in jobs}
            while pending:
                finished, pending = wait(pending, timeout=5, return_when=FIRST_COMPLETED)
                for future in finished:
                    try:
                        future.result()
                    except Exception as error:
                        stop.set()
                        errors.append(error)
                saved = sum(len(c["entries"]) for c in contexts)
                new = saved - restored
                elapsed = time.monotonic() - started
                eta = f"{elapsed / new * (total - saved) / 60:.1f}m (rough)" if new else "unknown"
                status = llm_request_status()
                progress.show(
                    f"[progress] saved={saved}/{total} ({100 * saved / total:.1f}%) | "
                    f"accounts complete={sum(c['replay'].done for c in contexts)}/{len(contexts)} | "
                    f"active={status['active_requests']} | "
                    f"accounted=${status['accounted_usd']:.4f}/${status['budget_target_usd']:.2f} | ETA~{eta}",
                    force=not pending,
                )
            if errors:
                raise RuntimeError("Gate paused. " + " | ".join(str(e) for e in errors)) from errors[0]
        except BaseException:
            stop.set()
            for future in pending:
                future.cancel()
            raise
        finally:
            pool.shutdown(wait=True, cancel_futures=True)
        if end_index < len(dates):
            _g24_backup("all accounts saved through " + str(dates[end_index - 1].date()))

    metrics = [_g24_metrics(context, protocol) for context in contexts]
    for context, metric in zip(contexts, metrics):
        _g24_immutable(context["root"] / "summary.json", metric)
    results = {
        "schema_version": 1, "protocol_id": "memory_gate_v1", "protocol_hash": _llm_digest(protocol),
        "source_stage": "MEMORY_GATE", "source_fingerprint": source_hash, "complete": True,
        "dates": [d.isoformat() for d in dates], "metrics": metrics,
        "validation_amendment": copy.deepcopy(LLM_GATE_VALIDATION_AMENDMENT),
        "selection_status": "NOT_APPLIED", "active_memory_changed": False,
        "interpretation": "Development selection data only. Do not tune lessons or thresholds from this table. Apply the already frozen rule next.",
    }
    _g24_immutable(LLM_GATE_RESULTS_PATH, results)
    table = pd.DataFrame([{
        "condition": m["condition_id"], "memories": m["memory_count"], "days": m["decisions"],
        "Sharpe": m["sharpe"], "net_return_pct": 100 * m["total_net_return"],
        "max_drawdown_pct": 100 * m["max_drawdown_frac"],
        "mean_abs_target": m["mean_absolute_target_exposure"],
        "KEEP": m["action_counts"]["KEEP"], "HALVE": m["action_counts"]["HALVE"], "FLAT": m["action_counts"]["FLAT"],
        "API_accounted_usd": m["api_accounted_usd"],
        "citation_count_warnings": m["evidence_count_style_warnings"],
    } for m in metrics])
    print("\nCOMPLETE MEMORY_GATE REPLAYS — SELECTION DATA, PROTOTYPE")
    with pd.option_context("display.max_columns", None, "display.max_rows", 20):
        display(table)
    print("New HTTP attempts this execution:", llm_request_status()["attempts"] - before["attempts"])
    print("Checkpoint folder:", LLM_GATE_RUN_DIR)
    print("Results:", LLM_GATE_RESULTS_PATH)
    display(FileLink(os.path.relpath(LLM_GATE_RESULTS_PATH, Path.cwd())))
    print("Request accounting:", json.dumps(llm_request_status(), indent=2))
    return contexts, results, table


if not LLM_GATE_RUN_LOCK.acquire(blocking=False):
    raise RuntimeError("The gate runner is already active.")
try:
    try:
        LLM_GATE_ACCOUNT_CONTEXTS, LLM_GATE_RESULTS, LLM_GATE_METRICS_TABLE = _run_memory_gate_v1()
        LLM_GATE_RUN_READY = True
    except BaseException:
        print("Gate paused. Received responses and completed daily files remain saved.", flush=True)
        try:
            _g24_backup("paused gate")
        except Exception as error:
            print("Could not create a new ZIP:", type(error).__name__, "| existing saved files remain on disk.")
        raise
finally:
    LLM_GATE_RUN_LOCK.release()

_g24_backup("complete gate")
print("Block 24 complete. No memory selected or activated. Next: apply the frozen selection rule, then freeze final evaluation.")



MEMORY_GATE | 10 accounts | 1577/1830 decisions restored
Up to six independent accounts; daily saves; consistent ZIP at each 25-date barrier.
Ctrl+C pauses dispatch; in-flight requests finish/save before the paused-run backup.
No gate scores are displayed until all scheduled accounts finish.
Recorded validation amendment: >4 valid distinct citations are style warnings for every condition.
Compact progress; full request log: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/logs/memory_gate_v1.log


[progress] saved=1830/1830 (100.0%) | accounts complete=10/10 | active=0 | accounted=$0.8719/$10.00 | ETA~0.0m (rough)

[backup] all accounts saved through 2024-12-22 | 4011 files | /kaggle/working/eth_llm_backup_20260929T140619_867899Z.zip


/kaggle/working/eth_llm_backup_20260929T140619_867899Z.zip


COMPLETE MEMORY_GATE REPLAYS — SELECTION DATA, PROTOTYPE


...

New HTTP attempts this execution: 236
Checkpoint folder: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/checkpoints/memory_gate_v1
Results: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/evaluator_only/memory_gate_v1_results.json


/kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/evaluator_only/memory_gate_v1_results.json

Request accounting: {
  "active_requests": 0,
  "saved_responses": 1763,
  "attempts": 1763,
  "usage_based_usd": 0.87186042,
  "reserved_or_uncertain_usd": 0,
  "accounted_usd": 0.87186042,
  "budget_target_usd": 10.0
}
[backup] complete gate | 4179 files | /kaggle/working/eth_llm_backup_20260929T140646_093591Z.zip


/kaggle/working/eth_llm_backup_20260929T140646_093591Z.zip

Block 24 complete. No memory selected or activated. Next: apply the frozen selection rule, then freeze final evaluation.


In [41]:
# BLOCK 25 — Apply the frozen gate rule and freeze the final evaluation.
# No API calls, new lessons, candidate edits, or final-evaluation prices/returns.
# Uses full-precision saved gate metrics. Reruns preserve identical artifacts.

import copy
import json
import math
import os
from pathlib import Path
import pandas as pd
from IPython.display import display, FileLink

for _flag in ["LLM_GATE_RUN_READY", "LLM_GATE_PROTOCOL_READY", "LLM_REQUEST_CLIENT_READY",
              "LLM_LOCAL_CHECKPOINT_HELPERS_READY"]:
    if not globals().get(_flag, False):
        raise RuntimeError("Missing setup: " + _flag + ". Finish or restore Block 24 first.")
if llm_request_status()["active_requests"] or LLM_GATE_RUN_LOCK.locked():
    raise RuntimeError("Wait for Block 24 and its requests to finish.")
LLM_GATE_SELECTION_READY = False
LLM_FROZEN_EVAL_PROTOCOL_READY = False


def _g25_select(protocol, metrics):
    conditions = {c["condition_id"]: c for c in protocol["conditions"]}
    indexed = {m["condition_id"]: m for m in metrics}
    if len(indexed) != len(metrics) or set(indexed) != set(conditions):
        raise RuntimeError("All frozen gate conditions must be present exactly once.")
    rule = protocol["selection"]
    for cid, metric in indexed.items():
        if metric["decisions"] != rule["required_complete_decisions"]:
            raise RuntimeError("Incomplete gate condition: " + cid)
        for field in ["max_drawdown_frac", "total_log_growth"]:
            if type(metric[field]) not in (int, float) or not math.isfinite(metric[field]):
                raise RuntimeError("Invalid gate metric: " + cid + "/" + field)
        sharpe = metric["sharpe"]
        if sharpe is not None and (type(sharpe) not in (int, float) or not math.isfinite(sharpe)):
            raise RuntimeError("Nonfinite Sharpe must be stored as null: " + cid)
    expected = [c["condition_id"] for c in protocol["conditions"] if c["selectable"]]
    if (rule["candidate_conditions"] != expected or rule["selected_memory_limit"] != 1
            or rule["all_requirements_must_pass"] is not True):
        raise RuntimeError("Unexpected selection semantics; inspect the frozen rule.")
    reference = indexed[rule["reference_condition"]]
    min_sharpe = (reference["sharpe"] + rule["minimum_sharpe_improvement"]
                  if reference["sharpe"] is not None else None)
    max_dd = reference["max_drawdown_frac"] + rule["maximum_drawdown_increase_frac"]
    ref_growth = reference["total_log_growth"]
    min_growth = (rule["minimum_positive_reference_log_growth_fraction"] * ref_growth
                  if ref_growth > 0 else ref_growth)
    assessments = []
    for cid in expected:
        if conditions[cid]["kind"] != "LLM" or len(conditions[cid]["memories"]) != 1:
            raise RuntimeError("Only the frozen single-memory conditions may be selected.")
        m = indexed[cid]
        checks = {
            "sharpe_pass": min_sharpe is not None and m["sharpe"] is not None and m["sharpe"] >= min_sharpe,
            "drawdown_pass": m["max_drawdown_frac"] <= max_dd,
            "growth_pass": m["total_log_growth"] >= min_growth,
        }
        assessments.append({
            "condition_id": cid, "memory_id": conditions[cid]["memories"][0]["id"],
            "sharpe": m["sharpe"], "max_drawdown_frac": m["max_drawdown_frac"],
            "total_log_growth": m["total_log_growth"], **checks,
            "eligible": all(checks.values()),
            "failed_checks": [name for name, passed in checks.items() if not passed],
        })
    eligible = sorted([a for a in assessments if a["eligible"]],
                      key=lambda a: (-a["sharpe"], a["max_drawdown_frac"], a["condition_id"]))
    winner = eligible[0]["condition_id"] if eligible else None
    return {
        "reference_condition": rule["reference_condition"],
        "reference_metrics": {k: reference[k] for k in ["sharpe", "max_drawdown_frac", "total_log_growth"]},
        "thresholds": {"minimum_sharpe": min_sharpe, "maximum_drawdown_frac": max_dd,
                       "minimum_total_log_growth": min_growth},
        "candidate_assessments": assessments,
        "eligible_conditions_ranked": [a["condition_id"] for a in eligible],
        "selected_gate_condition": winner,
        "selected_memories": copy.deepcopy(conditions[winner]["memories"]) if winner else [],
        "outcome": "ONE_MEMORY_SELECTED" if winner else "NO_MEMORY_PASSED",
    }


def _g25_verify_saved_gate(protocol, dates):
    results = _llm_strict_json(LLM_GATE_RESULTS_PATH.read_text())
    if (_llm_digest(results) != _llm_digest(LLM_GATE_RESULTS)
            or results["protocol_hash"] != _llm_digest(protocol)
            or results.get("complete") is not True or results["source_stage"] != "MEMORY_GATE"
            or results["dates"] != [d.isoformat() for d in dates]):
        raise RuntimeError("Saved gate results do not match the completed run.")
    info = results["validation_amendment"]
    amendment_path = Path(LLM_API_RUN_DIR) / "protocols" / "memory_gate_v1_citation_count_amendment_v1.json"
    amendment = _llm_strict_json(amendment_path.read_text())
    if (info != LLM_GATE_VALIDATION_AMENDMENT or _llm_digest(amendment) != info["amendment_hash"]
            or amendment["base_protocol_hash"] != results["protocol_hash"]
            or info["base_protocol_hash"] != results["protocol_hash"]
            or amendment["base_validation_policy_hash"] != _llm_digest(protocol["response_validation_policy"])
            or info["effective_protocol_hash"] != _llm_digest({
                "base_protocol_hash": results["protocol_hash"], "validation_amendment_hash": info["amendment_hash"]})):
        raise RuntimeError("The recorded validation amendment changed or is missing.")
    contexts = {c["condition"]["condition_id"]: c for c in LLM_GATE_ACCOUNT_CONTEXTS}
    if len(contexts) != len(LLM_GATE_ACCOUNT_CONTEXTS) or set(contexts) != {c["condition_id"] for c in protocol["conditions"]}:
        raise RuntimeError("Completed account contexts are missing or duplicated. Restore Block 24.")
    metrics = {m["condition_id"]: m for m in results["metrics"]}
    if len(metrics) != len(results["metrics"]) or set(metrics) != set(contexts):
        raise RuntimeError("All ten frozen accounts must have metrics.")
    for condition in protocol["conditions"]:
        cid = condition["condition_id"]
        context = contexts[cid]
        root = Path(LLM_GATE_RUN_DIR) / cid
        plan = _llm_strict_json((root / "plan.json").read_text())
        if (context["condition"] != condition or plan["condition"] != condition
                or _llm_digest(plan) != context["plan_hash"]
                or plan["protocol_hash"] != results["protocol_hash"]
                or plan["source_fingerprint"] != results["source_fingerprint"]
                or plan["dates"] != results["dates"] or not context["replay"].done
                or len(context["entries"]) != len(dates)):
            raise RuntimeError("Incomplete or changed account: " + cid)
        files = sorted((root / "steps").glob("*.json"))
        if [p.name for p in files] != [f"{i:06d}.json" for i in range(len(dates))]:
            raise RuntimeError("Missing or extra daily checkpoints: " + cid)
        chain = context["plan_hash"]
        for index, (path, entry) in enumerate(zip(files, context["entries"])):
            saved = _llm_strict_json(path.read_text())
            if (saved != entry or saved["previous_step_hash"] != chain
                    or saved["index"] != index or saved["plan_hash"] != context["plan_hash"]
                    or saved["replay_record"]["source_timestamp_utc"] != dates[index].isoformat()):
                raise RuntimeError("Changed daily checkpoint: " + cid + "/" + path.name)
            chain = _llm_digest(saved)
        progress = _llm_strict_json((root / "progress.json").read_text())
        if chain != context["chain"] or progress != _g24_snapshot(context):
            raise RuntimeError("Completed progress differs: " + cid)
        summary = _llm_strict_json((root / "summary.json").read_text())
        if summary != metrics[cid] or summary != _g24_metrics(context, protocol):
            raise RuntimeError("Metrics differ from the completed gate trajectory: " + cid)
    return results, amendment


def _g25_build_final_protocol(protocol, results, amendment, selection, dates):
    conditions = {c["condition_id"]: c for c in protocol["conditions"]}
    final_conditions = []
    for cid in protocol["final_evaluation"]["conditions"]:
        if cid == "llm_gate_selected":
            if selection["selected_memories"]:
                item = {"condition_id": cid, "kind": "LLM", "memories": copy.deepcopy(selection["selected_memories"])}
            else:
                item = {"condition_id": cid, "kind": "ALIAS", "alias_of": "llm_no_memory", "memories": []}
        else:
            item = copy.deepcopy(conditions[cid])
        item["selectable"] = False
        final_conditions.append(item)
    return {
        "schema_version": 1, "protocol_id": "frozen_eval_v1", "source_stage": "FROZEN_EVAL",
        "mode": "PIPELINE_PROTOTYPE", "gate_protocol_hash": _llm_digest(protocol),
        "gate_results_hash": _llm_digest(results), "selection_hash": _llm_digest(selection),
        "gate_effective_protocol_hash": results["validation_amendment"]["effective_protocol_hash"],
        "validation_amendment": copy.deepcopy(results["validation_amendment"]),
        "validation_amendment_document": copy.deepcopy(amendment),
        "base_response_validation_policy": copy.deepcopy(protocol["response_validation_policy"]),
        "reviewer_harness_hash": protocol["reviewer_harness_hash"],
        "reviewer_system_prompt": protocol["reviewer_system_prompt"],
        "request_client_config_hash": protocol["request_client_config_hash"],
        "api_config": copy.deepcopy(protocol["api_config"]),
        "replay_protocol": copy.deepcopy(protocol["replay_protocol"]),
        "source_dates": [d.isoformat() for d in dates],
        "conditions": final_conditions,
        "selected_gate_condition": selection["selected_gate_condition"],
        "memory_presentation": protocol["memory_presentation"],
        "account_rule": protocol["account_rule"], "base_proposals": protocol["base_proposals"],
        "metrics": copy.deepcopy(protocol["gate_metrics"]),
        "additional_reporting": ["evidence_count_style_warnings", "reason_length_style_warnings"],
        "controls_and_comparisons": copy.deepcopy(protocol["final_evaluation"]),
        "policy_rule": "Use the same recorded citation-count amendment and reason-length policy in every LLM condition; keep all original responses.",
        "freeze_rule": "No reflections, memory updates, selection, prompt tuning or threshold changes during final evaluation. Portfolio state evolves normally.",
        "completion_rule": "Report every planned condition after complete runs; no dropping conditions or choosing a replacement memory from final results.",
        "alias_rule": "If selected memory is empty, reuse the no-memory trajectory for its alias without new calls or treating it as an independent replicate.",
        "scope_limitations": copy.deepcopy(protocol["scope_limitations"]),
    }


_g25_protocol, _g25_dates = _g24_check_protocol()
_g25_results, _g25_amendment = _g25_verify_saved_gate(_g25_protocol, _g25_dates)
_g25_selection = _g25_select(_g25_protocol, _g25_results["metrics"])
LLM_GATE_SELECTION = {
    "schema_version": 1, "selection_id": "memory_gate_v1_selection",
    "status": "FROZEN_FOR_FINAL_EVALUATION", "source_stage": "MEMORY_GATE",
    "gate_protocol_hash": _llm_digest(_g25_protocol), "gate_results_hash": _llm_digest(_g25_results),
    "gate_effective_protocol_hash": _g25_results["validation_amendment"]["effective_protocol_hash"],
    "frozen_selection_rule": copy.deepcopy(_g25_protocol["selection"]), **_g25_selection,
    "candidate_files_modified": False,
    "interpretation": "A gate-selected hypothesis for final evaluation; not evidence of final-period improvement.",
}
_g25_final_dates = pd.DatetimeIndex(pd.to_datetime(_g25_protocol["frozen_eval_source_dates"], utc=True))
if (not _g25_final_dates.equals(pd.DatetimeIndex(LLM_FROZEN_EVAL_DATES))
        or not _g25_final_dates.is_unique or not _g25_final_dates.is_monotonic_increasing
        or not LLM_MEMORY_ASSIGNMENT.loc[_g25_final_dates].eq("FROZEN_EVAL").all()
        or _g25_dates.max() + pd.Timedelta(days=1) >= _g25_final_dates.min()):
    raise RuntimeError("Final evaluation dates or the gate boundary differ from the frozen plan.")
LLM_FROZEN_EVAL_PROTOCOL = _g25_build_final_protocol(
    _g25_protocol, _g25_results, _g25_amendment, LLM_GATE_SELECTION, _g25_final_dates)
LLM_GATE_SELECTION_PATH = Path(LLM_API_RUN_DIR) / "memory" / "selections" / "memory_gate_v1_selection.json"
LLM_FROZEN_EVAL_PROTOCOL_PATH = Path(LLM_API_RUN_DIR) / "protocols" / "frozen_eval_v1.json"
# Check both existing files before adding either, so a changed setup cannot
# partially overwrite a previously frozen selection/evaluation pair.
_g25_artifacts = [(LLM_GATE_SELECTION_PATH, LLM_GATE_SELECTION),
                  (LLM_FROZEN_EVAL_PROTOCOL_PATH, LLM_FROZEN_EVAL_PROTOCOL)]
with LLM_DISK_LOCK:
    for _g25_path, _g25_value in _g25_artifacts:
        if _g25_path.exists() and _llm_digest(_llm_strict_json(_g25_path.read_text())) != _llm_digest(_g25_value):
            raise RuntimeError("Saved selection/evaluation plan differs. Preserve it and inspect: " + _g25_path.name)
    for _g25_path, _g25_value in _g25_artifacts:
        if not _g25_path.exists():
            llm_atomic_write_json(_g25_path, _g25_value)
LLM_FROZEN_EVAL_PROTOCOL_HASH = _llm_digest(LLM_FROZEN_EVAL_PROTOCOL)
LLM_GATE_SELECTION_READY = True
LLM_FROZEN_EVAL_PROTOCOL_READY = True

print("FROZEN GATE RULE APPLIED — FULL-PRECISION SAVED METRICS")
print("Thresholds:", json.dumps(LLM_GATE_SELECTION["thresholds"], indent=2))
display(pd.DataFrame([{
    "condition": a["condition_id"], "Sharpe": a["sharpe"],
    "max_drawdown_pct": 100 * a["max_drawdown_frac"], "log_growth": a["total_log_growth"],
    "Sharpe_pass": a["sharpe_pass"], "drawdown_pass": a["drawdown_pass"],
    "growth_pass": a["growth_pass"], "eligible": a["eligible"],
    "selected": a["condition_id"] == LLM_GATE_SELECTION["selected_gate_condition"],
} for a in LLM_GATE_SELECTION["candidate_assessments"]]))
print("Selected gate condition:", LLM_GATE_SELECTION["selected_gate_condition"] or "NONE — empty memory")
for _g25_memory in LLM_GATE_SELECTION["selected_memories"]:
    print("Selected memory:", _g25_memory["id"])
    print("Unchanged instruction:", _g25_memory["instruction"])
print("Selected memory saved:", LLM_GATE_SELECTION_PATH)
print("\nFINAL EVALUATION FROZEN")
print("Dates:", _g25_final_dates.min().date(), "to", _g25_final_dates.max().date(), "|", len(_g25_final_dates), "decisions/account")
display(pd.DataFrame([{"condition": c["condition_id"], "kind": c["kind"], "memories": len(c["memories"]),
                       "alias_of": c.get("alias_of", "")} for c in LLM_FROZEN_EVAL_PROTOCOL["conditions"]]))
print("Planned LLM decisions:", len(_g25_final_dates) * sum(c["kind"] == "LLM" for c in LLM_FROZEN_EVAL_PROTOCOL["conditions"]))
print("Protocol:", LLM_FROZEN_EVAL_PROTOCOL_PATH)
print("Protocol hash:", LLM_FROZEN_EVAL_PROTOCOL_HASH)
print("No API calls or final-period prices, returns or scores read. No candidate text edited.")
_g25_backup, _g25_count = backup_llm_experiment()
print("Download updated backup:", _g25_backup)
display(FileLink(os.path.relpath(_g25_backup, Path.cwd())))
print("Block 25 complete. Next: a resumable parallel FROZEN_EVAL runner using this exact plan.")


FROZEN GATE RULE APPLIED — FULL-PRECISION SAVED METRICS
Thresholds: {
  "minimum_sharpe": 6.775762547351676,
  "maximum_drawdown_frac": 0.09531820837977133,
  "minimum_total_log_growth": 1.3312928298066338
}


,condition,Sharpe,max_drawdown_pct,log_growth,Sharpe_pass,drawdown_pass,growth_pass,eligible,selected
0,llm_single_b001,7.079111,10.091752,1.671450,True,False,True,False,False
1,llm_single_b008,6.735188,6.754621,0.862389,False,True,False,False,False
2,llm_single_b013,7.107191,9.175697,1.675782,True,True,True,True,True
3,llm_single_b018,6.788088,9.141039,1.567001,True,True,True,True,False
4,llm_single_b019,6.901442,5.465362,0.872605,True,True,False,False,False


Selected gate condition: llm_single_b013
Selected memory: mem_2f1deec24a38ffb4
Unchanged instruction: When the ensemble proposal is modest in magnitude or the MLP direction conflicts with the PPO seeds, check whether the model components actually agree before choosing KEEP. If MLP max probability is near 0.5 and PPO seeds are mixed or opposite to the MLP direction, consider that the proposal may be weakly supported and that HALVE or FLAT could reduce exposure to an uncertain signal. When the proposal is large and PPO seeds and MLP direction are aligned, this concern is weaker. Use the supplied cost table directly rather than recomputing costs.
Selected memory saved: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/memory/selections/memory_gate_v1_selection.json

FINAL EVALUATION FROZEN
Dates: 2025-01-01 to 2026-09-24 | 632 decisions/account


,condition,kind,memories,alias_of
0,model_keep,SCRIPTED,0,
1,fixed_halve,SCRIPTED,0,
2,llm_no_memory,LLM,0,
3,llm_raw_memory,LLM,20,
4,llm_screen_only,LLM,5,
5,llm_gate_selected,LLM,1,


Planned LLM decisions: 2528
Protocol: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/protocols/frozen_eval_v1.json
Protocol hash: 34803b17d49f80e20ffc636ee01871813030ab2fae22de3057e6a20dd7e4bea3
No API calls or final-period prices, returns or scores read. No candidate text edited.
Download updated backup: /kaggle/working/eth_llm_backup_20260929T141443_448553Z.zip


/kaggle/working/eth_llm_backup_20260929T141443_448553Z.zip

Block 25 complete. Next: a resumable parallel FROZEN_EVAL runner using this exact plan.


In [50]:
# BLOCK 26 — Run the frozen final evaluation, with compact output and resumption.
# PAID CALLS: four LLM conditions x 632 dates = 2,528 before cache reuse/retries.
# Two scripted controls need no LLM. Every account starts with fresh initial cash.
# Daily checkpoints; consistent ZIP backup every 50 dates, on pause and at completion.
# No learning, memory selection, prompt changes or repairs based on final outcomes.
# Requires Blocks 24 and 25 in the current kernel; do not change their frozen files.
# Replacement: records a post-start validation amendment for the supplied PPO
# group citation. Original prompts, responses, actions and checkpoints stay intact.

import copy
import json
import math
import os
import threading
import time
import types
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, FileLink

for _flag in ["LLM_GATE_SELECTION_READY", "LLM_FROZEN_EVAL_PROTOCOL_READY", "LLM_REQUEST_CLIENT_READY",
              "LLM_REPLAY_ACCOUNTING_READY", "LLM_LOCAL_CHECKPOINT_HELPERS_READY"]:
    if not globals().get(_flag, False):
        raise RuntimeError("Missing setup: " + _flag + ". Complete or restore Block 25 first.")
if "LLM_FROZEN_EVAL_RUN_LOCK" not in globals():
    LLM_FROZEN_EVAL_RUN_LOCK = threading.Lock()
if (LLM_FROZEN_EVAL_RUN_LOCK.locked() or LLM_GATE_RUN_LOCK.locked()
        or llm_request_status()["active_requests"]):
    raise RuntimeError("Wait for existing replays and requests to stop before running this cell.")
LLM_FROZEN_EVAL_RUN_READY = False
LLM_FROZEN_EVAL_RUN_DIR = Path(LLM_RUN_CHECKPOINT_DIR) / "frozen_eval_v1"
LLM_FROZEN_EVAL_RESULTS_PATH = Path(LLM_API_RUN_DIR) / "evaluator_only" / "frozen_eval_v1_results.json"
LLM_FROZEN_EVAL_DETAIL_LOG = Path(LLM_API_RUN_DIR) / "logs" / "frozen_eval_v1.log"
LLM_FROZEN_EVAL_BACKUP_INTERVAL = 50
LLM_FROZEN_EVAL_PROGRESS_SECONDS = 10
_e26_backup_display = None


def _e26_detail_print(*values, sep=" ", end="\n", **kwargs):
    # Existing request messages contain IDs/status, not API keys. Keep them out
    # of notebook output; the authoritative responses/checkpoints stay intact.
    line = sep.join(str(v) for v in values) + end
    with LLM_DISK_LOCK:
        LLM_FROZEN_EVAL_DETAIL_LOG.parent.mkdir(parents=True, exist_ok=True)
        with LLM_FROZEN_EVAL_DETAIL_LOG.open("a", encoding="utf-8") as stream:
            stream.write(time.strftime("[%Y-%m-%dT%H:%M:%SZ] ", time.gmtime()) + line)


def _e26_logged_copy(function):
    # Reuse the exact function bytecode with a LOCAL print sink. Do not change
    # builtins.print, sys.stdout, the shared client or other notebook cells.
    owner = getattr(function, "__self__", None)
    source = getattr(function, "__func__", function)
    if not isinstance(source, types.FunctionType):
        raise RuntimeError("Expected the existing Python request/validation function for compact logging.")
    namespace = dict(source.__globals__)
    namespace["print"] = _e26_detail_print
    local = types.FunctionType(source.__code__, namespace, source.__name__,
                               source.__defaults__, source.__closure__)
    local.__kwdefaults__ = copy.copy(source.__kwdefaults__)
    return types.MethodType(local, owner) if owner is not None else local


class _E26Progress:
    def __init__(self):
        self.handle = None
        self.started = False
        self.last_update = float("-inf")

    def show(self, line, *, force=False):
        now = time.monotonic()
        interval = LLM_FROZEN_EVAL_PROGRESS_SECONDS if self.handle is not None else 30
        if not force and now - self.last_update < interval:
            return
        self.last_update = now
        if not self.started:
            # Kaggle updates this single display instead of appending lines.
            # Plain consoles fall back to a summary at most every 30 seconds.
            self.handle = display({"text/plain": line}, raw=True, display_id=True)
            self.started = True
        elif self.handle is not None:
            self.handle.update({"text/plain": line}, raw=True)
        else:
            print(line, flush=True)


def _e26_immutable(path, value):
    with LLM_DISK_LOCK:
        if path.exists():
            if _llm_digest(_llm_strict_json(path.read_text())) != _llm_digest(value):
                raise RuntimeError("Saved final-evaluation artifact differs: " + str(path) + ". Preserve it and inspect.")
        else:
            llm_atomic_write_json(path, value)


def _e26_check_protocol():
    protocol = _llm_strict_json(LLM_FROZEN_EVAL_PROTOCOL_PATH.read_text())
    if (_llm_digest(protocol) != LLM_FROZEN_EVAL_PROTOCOL_HASH
            or _llm_digest(LLM_FROZEN_EVAL_PROTOCOL) != LLM_FROZEN_EVAL_PROTOCOL_HASH
            or protocol["protocol_id"] != "frozen_eval_v1" or protocol["source_stage"] != "FROZEN_EVAL"):
        raise RuntimeError("Saved final-evaluation protocol differs from the frozen setup.")
    gate, gate_dates = _g24_check_protocol()
    gate_results = _llm_strict_json(LLM_GATE_RESULTS_PATH.read_text())
    selection = _llm_strict_json(LLM_GATE_SELECTION_PATH.read_text())
    amendment_path = Path(LLM_API_RUN_DIR) / "protocols" / "memory_gate_v1_citation_count_amendment_v1.json"
    amendment = _llm_strict_json(amendment_path.read_text())
    if (_llm_digest(gate) != protocol["gate_protocol_hash"]
            or _llm_digest(gate_results) != protocol["gate_results_hash"]
            or _llm_digest(selection) != protocol["selection_hash"]
            or _llm_digest(LLM_GATE_SELECTION) != protocol["selection_hash"]
            or selection["status"] != "FROZEN_FOR_FINAL_EVALUATION"
            or selection["gate_protocol_hash"] != protocol["gate_protocol_hash"]
            or selection["gate_results_hash"] != protocol["gate_results_hash"]
            or not gate_results["complete"]):
        raise RuntimeError("The saved gate results or selected memory changed after freezing.")
    if (amendment != protocol["validation_amendment_document"]
            or _llm_digest(amendment) != protocol["validation_amendment"]["amendment_hash"]
            or protocol["validation_amendment"] != gate_results["validation_amendment"]
            or protocol["validation_amendment"] != LLM_GATE_VALIDATION_AMENDMENT):
        raise RuntimeError("The recorded response-validation amendment differs.")
    chosen = _g25_select(gate, gate_results["metrics"])
    if any(selection.get(key) != value for key, value in chosen.items()):
        raise RuntimeError("Selection no longer matches the frozen gate rule.")
    dates = pd.DatetimeIndex(pd.to_datetime(protocol["source_dates"], utc=True))
    if (not dates.equals(pd.DatetimeIndex(LLM_FROZEN_EVAL_DATES))
            or protocol["source_dates"] != gate["frozen_eval_source_dates"]
            or not dates.is_unique or not dates.is_monotonic_increasing
            or not dates.equals(pd.date_range(dates.min(), dates.max(), freq="D", tz="UTC"))
            or not LLM_MEMORY_ASSIGNMENT.loc[dates].eq("FROZEN_EVAL").all()
            or gate_dates.max() + pd.Timedelta(days=1) >= dates.min()):
        raise RuntimeError("Unexpected final evaluation dates or gate boundary.")
    expected = _g25_build_final_protocol(gate, gate_results, amendment, selection, dates)
    if _llm_digest(expected) != _llm_digest(protocol):
        raise RuntimeError("Final prompts, controls, memory, metrics or settings differ from Block 25.")
    return protocol, dates


def _e26_prepare_parent_citation_amendment(protocol):
    # This is NOT a change to learned memory or a claim of preregistration.
    # Keep the original final protocol immutable and attach a dated-stage addendum.
    amendment = {
        "schema_version": 1,
        "amendment_id": "supplied-ppo-group-citation-v1",
        "base_protocol_hash": _llm_digest(protocol),
        "prior_validation_amendment_hash": protocol["validation_amendment"]["amendment_hash"],
        "origin": "Assistant-authored recovery for a received parent-dictionary citation.",
        "timing": "Introduced after final evaluation started; report this post-freeze amendment.",
        "trigger_request_id": "747ef70f780cb48511bfec4962c743aef5fb9c44d764a38e80cbe0144832a466",
        "trigger_purpose": "frozen_eval_v1/llm_raw_memory/2025-08-12T00:00:00+00:00/review",
        "accepted_parent": "model.ppo_seed_proposals",
        "required_leaf_names": ["ppo_seed0", "ppo_seed1", "ppo_seed2"],
        "hard_rule": "Parent must exist in the supplied state as exactly these three finite numeric scores; all three leaf paths must already be allowed. Other unsupported references remain invalid.",
        "overlap_rule": "Reject a parent cited together with any of its leaves; duplicate or empty citations remain invalid.",
        "application": "Every FROZEN_EVAL condition, including restored and pending responses; no change to completed gate selection.",
        "validation_view": "Only a private validation copy adds the verified parent to allowed_evidence_fields. Never send or save that copy as an API request. Link original and validation-view hashes in the audit.",
        "preservation": "Preserve original response and action verbatim, original citation count, cache identity, prompts, memories, selection and accounting; no response regeneration.",
        "reporting": "Record per-request resolution and per-condition counts. Valid references do not certify the explanation's truth or trading utility.",
    }
    path = Path(LLM_API_RUN_DIR) / "protocols" / "frozen_eval_v1_ppo_group_citation_amendment_v1.json"
    if LLM_FROZEN_EVAL_RESULTS_PATH.exists() and not path.exists():
        raise RuntimeError("Final results already exist without this addendum. Preserve the completed run; do not amend it retrospectively.")
    _e26_immutable(path, amendment)
    info = {
        "base_protocol_hash": _llm_digest(protocol),
        "amendment_id": amendment["amendment_id"],
        "amendment_hash": _llm_digest(amendment),
        "amendment_relative_path": "protocols/" + path.name,
        "introduced_during_final_evaluation": True,
    }
    info["effective_validation_hash"] = _llm_digest({
        "base_protocol_hash": info["base_protocol_hash"],
        "prior_validation_amendment_hash": amendment["prior_validation_amendment_hash"],
        "additional_validation_amendment_hash": info["amendment_hash"],
    })
    return info


def _e26_parent_citation_resolution(data, payload):
    # Deliberately support only this one known, fully supplied dictionary.
    # Never guess aliases, recursively admit arbitrary parents or drop citations.
    parent = "model.ppo_seed_proposals"
    fields = data.get("evidence_fields") if isinstance(data, dict) else None
    if not isinstance(fields, list) or parent not in fields:
        return {}
    keys = ["ppo_seed0", "ppo_seed1", "ppo_seed2"]
    leaves = [parent + "." + key for key in keys]
    group = payload.get("state", {}).get("model", {}).get("ppo_seed_proposals")
    if (not isinstance(group, dict) or set(group) != set(keys)
            or not all(isinstance(v, (int, float)) and not isinstance(v, bool)
                       and math.isfinite(v) for v in group.values())
            or not set(leaves).issubset(payload["allowed_evidence_fields"])):
        raise ValueError("PPO group citation requires exactly three supplied finite scores and three allowed leaf paths.")
    if any(leaf in fields for leaf in leaves):
        raise ValueError("Overlapping PPO parent and leaf citations are invalid; preserve the response for inspection.")
    return {parent: leaves}


# These pure/shared helpers are the same implementations used in the gate.
# They receive this stage's protocol, dates and account; no gate globals are reassigned.
_e26_inputs = _g24_inputs
_e26_cached = _g24_cached
_e26_scripted = _g24_scripted
_e26_snapshot = _g24_snapshot
_e26_make_entry = _g24_make_entry


def _e26_purpose(condition, date):
    return "frozen_eval_v1/" + condition["condition_id"] + "/" + date.isoformat() + "/review"


def _e26_checked_decision(result, payload, condition, date):
    try:
        resolution = _e26_parent_citation_resolution(result["data"], payload)
        if not resolution:
            return _e26_validate(result["data"], payload)
        validation_view = copy.deepcopy(payload)
        validation_view["allowed_evidence_fields"] = sorted(
            set(payload["allowed_evidence_fields"]) | set(resolution))
        # Reuse every existing structural check and style policy. Raw response
        # text/count stays unchanged. Any style event's validation-view hash is
        # explicitly linked to the actual sent input hash in the resolution audit.
        decision = _e26_validate(result["data"], validation_view)
        event = {
            "schema_version": 1, "event_type": "supplied_ppo_group_citation_resolved",
            "validation_amendment_hash": LLM_FROZEN_EVAL_PARENT_CITATION_AMENDMENT["amendment_hash"],
            "condition_id": condition["condition_id"], "source_timestamp_utc": date.isoformat(),
            "request_id": result["request_id"], "purpose": _e26_purpose(condition, date),
            "original_review_input_hash": _llm_digest(payload),
            "validation_view_hash": _llm_digest(validation_view),
            "original_decision_hash": _llm_digest(result["data"]),
            "original_response": copy.deepcopy(result["data"]),
            "resolution": resolution, "original_citation_count": len(decision["evidence_fields"]),
            "supplied_group": copy.deepcopy(payload["state"]["model"]["ppo_seed_proposals"]),
            "response_modified": False, "prompt_modified": False,
            "structural_validation": "PASSED_UNDER_RECORDED_POST_START_AMENDMENT",
            "meaning": "Citation resolution only; no claim about explanation correctness or action quality.",
        }
        path = (Path(LLM_API_RUN_DIR) / "events" / "frozen_eval_ppo_group_citations"
                / (result["request_id"] + ".json"))
        _e26_immutable(path, event)
        return decision
    except ValueError as error:
        raise RuntimeError(
            "Final evaluation paused before execution: " + condition["condition_id"] + "/" + date.isoformat()
            + " | " + str(error) + " | Cached request: " + result["request_id"]
            + " | Supplied response evidence_fields: " + json.dumps(
                result["data"].get("evidence_fields") if isinstance(result["data"], dict) else None)
        ) from None


def _e26_prepare_account(condition, protocol, dates, source_hash):
    root = LLM_FROZEN_EVAL_RUN_DIR / condition["condition_id"]
    plan = {"schema_version": 1, "runner_version": "parallel-frozen-eval-v1", "stage": "FROZEN_EVAL",
            "protocol_hash": _llm_digest(protocol), "source_fingerprint": source_hash,
            "condition": copy.deepcopy(condition), "dates": [d.isoformat() for d in dates]}
    _e26_immutable(root / "plan.json", plan)
    context = {"condition": condition, "root": root, "plan_hash": _llm_digest(plan),
               "replay": LLMDailyReplay(dates), "entries": [], "chain": _llm_digest(plan)}
    replay, entries = context["replay"], context["entries"]
    snapshots = [_e26_snapshot(context)]
    files = sorted((root / "steps").glob("*.json"))
    if len(files) > len(dates) or [p.name for p in files] != [f"{i:06d}.json" for i in range(len(files))]:
        raise RuntimeError("Final daily files have gaps or unexpected names: " + condition["condition_id"])
    for index, path in enumerate(files):
        saved = _llm_strict_json(path.read_text())
        messages, payload = _e26_inputs(replay.get_state(), index, condition, dates, protocol)
        result = None
        if condition["kind"] == "LLM":
            _, result = _e26_cached(messages, _e26_purpose(condition, dates[index]), protocol)
            if result is None:
                raise RuntimeError("Completed final step lacks its response cache. Restore the full backup.")
            decision = _e26_checked_decision(result, payload, condition, dates[index])
        else:
            decision = _e26_scripted(condition)
        record = replay.step(decision["action"])
        _cost_pilot_check_execution(payload, record)
        expected = _e26_make_entry(context, decision, payload, result, record)
        if saved != expected:
            raise RuntimeError("Restored final step or accounting differs: " + condition["condition_id"] + "/" + path.name)
        context["chain"] = _llm_digest(saved)
        entries.append(saved)
        snapshots.append(_e26_snapshot(context))
    progress_path = root / "progress.json"
    if progress_path.exists():
        progress = _llm_strict_json(progress_path.read_text())
        n = progress.get("next_index")
        if type(n) is not int or not 0 <= n < len(snapshots) or progress != snapshots[n]:
            raise RuntimeError("Final progress pointer differs from its daily files: " + condition["condition_id"])
    # A crash between the daily file and pointer write is repaired from the daily file.
    llm_atomic_write_json(progress_path, snapshots[-1])
    # Inspect a paid response received just before an earlier interruption before any
    # accounts dispatch more requests. Never automatically redo an invalid response.
    if not replay.done and condition["kind"] == "LLM":
        messages, payload = _e26_inputs(replay.get_state(), replay._i, condition, dates, protocol)
        _, pending = _e26_cached(messages, _e26_purpose(condition, dates[replay._i]), protocol)
        if pending is not None:
            _e26_checked_decision(pending, payload, condition, dates[replay._i])
    _e26_detail_print(f"[restore] {condition['condition_id']} | {len(entries)}/{len(dates)} saved")
    return context


def _e26_advance(context, protocol, dates, end_index, stop):
    replay, condition = context["replay"], context["condition"]
    try:
        while replay._i < end_index and not stop.is_set():
            index = replay._i
            messages, payload = _e26_inputs(replay.get_state(), index, condition, dates, protocol)
            result = None
            if condition["kind"] == "LLM":
                result = _e26_request(messages, purpose=_e26_purpose(condition, dates[index]),
                                          max_tokens=int(protocol["api_config"]["decision_max_tokens"]))
                try:
                    decision = _e26_checked_decision(result, payload, condition, dates[index])
                except ValueError as error:
                    raise RuntimeError("Final evaluation paused before execution: " + condition["condition_id"] + "/"
                                       + dates[index].isoformat() + " | " + str(error)
                                       + " | Received response remains cached: " + result["request_id"]) from None
            else:
                decision = _e26_scripted(condition)
            record = replay.step(decision["action"])
            _cost_pilot_check_execution(payload, record)
            entry = _e26_make_entry(context, decision, payload, result, record)
            chain = _llm_digest(entry)
            # Both writes use the shared disk lock; daily files remain authoritative
            # if a process/storage interruption occurs before the pointer update.
            with LLM_DISK_LOCK:
                _e26_immutable(context["root"] / "steps" / f"{index:06d}.json", entry)
                llm_atomic_write_json(context["root"] / "progress.json", _e26_snapshot(context, chain))
            context["chain"] = chain
            context["entries"].append(entry)
            if (index + 1) % 5 == 0 or index + 1 == end_index:
                _e26_detail_print(f"[saved] {condition['condition_id']} | {index + 1}/{len(dates)} | "
                                  f"{dates[index].date()} | {decision['action']}")
        return condition["condition_id"]
    except BaseException:
        stop.set()
        raise


def _e26_backup(label):
    global LLM_FROZEN_EVAL_LATEST_BACKUP, _e26_backup_display
    if llm_request_status()["active_requests"]:
        raise RuntimeError("Wait for every request worker before creating a consistent ZIP.")
    backup, count = backup_llm_experiment()
    LLM_FROZEN_EVAL_LATEST_BACKUP = backup
    _e26_detail_print(f"[backup] {label} | {count} files | {backup}")
    link = FileLink(os.path.relpath(backup, Path.cwd()),
                    result_html_prefix="Latest backup — " + label + ": ")
    if _e26_backup_display is None:
        _e26_backup_display = display(link, display_id=True)
    else:
        _e26_backup_display.update(link)


def _e26_metrics(context, protocol):
    result = _g24_metrics(context, {"gate_metrics": protocol["metrics"],
                                    "replay_protocol": protocol["replay_protocol"]})
    result.update(alias_of=None, independent_replay=True,
                  ppo_group_citation_resolutions=sum(
                      "model.ppo_seed_proposals" in e["decision"].get("evidence_fields", [])
                      for e in context["entries"]))
    return result


def _e26_alias_metric(condition, source, protocol_hash):
    # An empty selected-memory condition is the SAME policy as no-memory.
    # Copy its trajectory metrics without charging or counting another run.
    if condition["kind"] != "ALIAS" or condition["alias_of"] != "llm_no_memory" or condition["memories"]:
        raise RuntimeError("Unexpected final evaluation alias.")
    metric = copy.deepcopy(source)
    metric.update(condition_id=condition["condition_id"], kind="ALIAS", selectable=False,
                  alias_of=condition["alias_of"], independent_replay=False,
                  prompt_tokens=0, completion_tokens=0, api_attempts=0, api_accounted_usd=0.0,
                  api_cost_note="Alias of the same no-memory trajectory; no extra requests or independent replicate.")
    artifact = {"schema_version": 1, "protocol_hash": protocol_hash,
                "condition_id": condition["condition_id"], "alias_of": condition["alias_of"],
                "source_plan_hash": source["plan_hash"], "source_final_step_hash": source["final_step_hash"]}
    _e26_immutable(LLM_FROZEN_EVAL_RUN_DIR / condition["condition_id"] / "alias.json", artifact)
    return metric


def _run_frozen_evaluation_v1():
    global _e26_request, _e26_validate, LLM_FROZEN_EVAL_PARENT_CITATION_AMENDMENT
    protocol, dates = _e26_check_protocol()
    LLM_FROZEN_EVAL_PARENT_CITATION_AMENDMENT = _e26_prepare_parent_citation_amendment(protocol)
    _e26_request = _e26_logged_copy(LLM_REQUEST_CLIENT.request)
    _e26_validate = _e26_logged_copy(_g24_validate_with_citation_warning)
    # Bind the SAME amended validation to the final logger without replacing
    # the original notebook validator or any gate-run functions/globals.
    _e26_validate.__globals__["_g24_base_validate"] = _e26_logged_copy(validate_review_decision_v2)
    _e26_validate.__globals__["LLM_GATE_VALIDATION_AMENDMENT"] = copy.deepcopy(protocol["validation_amendment"])
    _e26_validate.__globals__["_g24_immutable"] = _e26_immutable
    _e26_detail_print("[run] start/resume | frozen final protocol " + LLM_FROZEN_EVAL_PROTOCOL_HASH)
    # Evaluator-only integrity fingerprint. Future outcomes are never supplied
    # to the reviewer, its memory, or the request/cache purpose.
    fresh = LLMDailyReplay(dates)
    rows = []
    for date in dates:
        outcome = LLM_EVALUATOR_DAILY.loc[date]
        exit_date = pd.Timestamp(outcome["exit_price_row_utc"])
        if not bool(outcome["eligible"]) or exit_date != date + pd.Timedelta(days=1):
            raise RuntimeError("Final evaluation requires exact one-calendar-day outcomes.")
        rows.append({"state_with_fresh_account": build_reviewer_state(date, portfolio_state=fresh._portfolio),
                     "outcome_price_row_utc": exit_date.isoformat(), "market_return_1d": float(outcome["market_return_1d"])})
    source_hash = _llm_digest(rows)
    _e26_immutable(LLM_FROZEN_EVAL_RUN_DIR / "run_manifest.json", {
        "runner_version": "parallel-frozen-eval-v1", "protocol_hash": _llm_digest(protocol),
        "source_fingerprint": source_hash, "source_stage": "FROZEN_EVAL",
        "backup_barrier_dates": LLM_FROZEN_EVAL_BACKUP_INTERVAL,
    })
    # Verify all accounts and any pending cached response before paid dispatch.
    contexts = [_e26_prepare_account(c, protocol, dates, source_hash)
                for c in protocol["conditions"] if c["kind"] != "ALIAS"]
    total = len(contexts) * len(dates)
    restored = sum(len(c["entries"]) for c in contexts)
    llm_contexts = [c for c in contexts if c["condition"]["kind"] == "LLM"]
    llm_total = len(llm_contexts) * len(dates)
    llm_restored = sum(len(c["entries"]) for c in llm_contexts)
    before, started = llm_request_status(), time.monotonic()
    print(f"FROZEN_EVAL | {len(contexts)} independent accounts | {restored}/{total} decisions restored")
    print(f"LLM decisions: {llm_total}; separate accounts run in parallel, dates sequential within each account.")
    print("Frozen memory and prompt; fresh accounts; daily saves; ZIP backup every 50 dates.")
    print("Ctrl+C pauses dispatch; in-flight requests finish/save before the paused-run backup.")
    print("Progress and latest backup update in place. Full log:", LLM_FROZEN_EVAL_DETAIL_LOG)
    print("Performance appears only after all scheduled conditions finish.")
    print("Recorded post-start amendment: supplied PPO-group citations resolve to the three allowed seed scores; original decisions retained.")
    progress = _E26Progress()
    progress.show(f"[progress] restored={restored}/{total} | LLM saved={llm_restored}/{llm_total}", force=True)
    workers = min(6, int(protocol["api_config"]["max_parallel_requests"]))
    while any(not c["replay"].done for c in contexts):
        earliest = min(c["replay"]._i for c in contexts if not c["replay"].done)
        end_index = min((earliest // LLM_FROZEN_EVAL_BACKUP_INTERVAL + 1) * LLM_FROZEN_EVAL_BACKUP_INTERVAL, len(dates))
        jobs = [c for c in contexts if c["replay"]._i < end_index]
        stop = threading.Event()
        pool = ThreadPoolExecutor(max_workers=workers)
        pending = set()
        errors = []
        try:
            pending = {pool.submit(_e26_advance, c, protocol, dates, end_index, stop) for c in jobs}
            while pending:
                finished, pending = wait(pending, timeout=5, return_when=FIRST_COMPLETED)
                for future in finished:
                    try:
                        future.result()
                    except Exception as error:
                        stop.set()
                        errors.append(error)
                saved = sum(len(c["entries"]) for c in contexts)
                llm_saved = sum(len(c["entries"]) for c in llm_contexts)
                newly_saved_llm = llm_saved - llm_restored
                elapsed = time.monotonic() - started
                eta = (f"{elapsed / newly_saved_llm * (llm_total - llm_saved) / 60:.1f}m (rough)"
                       if newly_saved_llm else "unknown")
                status = llm_request_status()
                progress.show(
                    f"[progress] saved={saved}/{total} ({100 * saved / total:.1f}%) | "
                    f"LLM={llm_saved}/{llm_total} | accounts complete={sum(c['replay'].done for c in contexts)}/{len(contexts)} | "
                    f"active={status['active_requests']} | accounted=${status['accounted_usd']:.4f}/${status['budget_target_usd']:.2f} | ETA~{eta}",
                    force=not pending,
                )
            if errors:
                raise RuntimeError("Final evaluation paused. " + " | ".join(str(e) for e in errors)) from errors[0]
        except BaseException:
            stop.set()
            for future in pending:
                future.cancel()
            raise
        finally:
            pool.shutdown(wait=True, cancel_futures=True)
        if end_index < len(dates):
            _e26_backup("all accounts saved through " + str(dates[end_index - 1].date()))

    indexed_metrics = {c["condition"]["condition_id"]: _e26_metrics(c, protocol) for c in contexts}
    metrics = []
    for condition in protocol["conditions"]:
        cid = condition["condition_id"]
        if condition["kind"] == "ALIAS":
            metric = _e26_alias_metric(condition, indexed_metrics[condition["alias_of"]], _llm_digest(protocol))
        else:
            metric = indexed_metrics[cid]
        metrics.append(metric)
        _e26_immutable(LLM_FROZEN_EVAL_RUN_DIR / cid / "summary.json", metric)
    results = {
        "schema_version": 1, "protocol_id": "frozen_eval_v1", "protocol_hash": _llm_digest(protocol),
        "source_stage": "FROZEN_EVAL", "source_fingerprint": source_hash, "complete": True,
        "dates": [d.isoformat() for d in dates], "metrics": metrics,
        "selection_hash": protocol["selection_hash"], "selected_gate_condition": protocol["selected_gate_condition"],
        "validation_amendment": copy.deepcopy(protocol["validation_amendment"]),
        "additional_validation_amendments": [copy.deepcopy(LLM_FROZEN_EVAL_PARENT_CITATION_AMENDMENT)],
        "memory_updated": False, "selection_changed": False,
        "interpretation": "Final-period comparison of frozen prototype policies with a recorded citation-validation amendment introduced during final evaluation. Report all conditions and amendments; do not reselect or tune on these outcomes. Known base-model leakage remains.",
    }
    _e26_immutable(LLM_FROZEN_EVAL_RESULTS_PATH, results)
    table = pd.DataFrame([{
        "condition": m["condition_id"], "memories": m["memory_count"], "days": m["decisions"],
        "Sharpe": m["sharpe"], "net_return_pct": 100 * m["total_net_return"],
        "max_drawdown_pct": 100 * m["max_drawdown_frac"], "mean_abs_target": m["mean_absolute_target_exposure"],
        "KEEP": m["action_counts"]["KEEP"], "HALVE": m["action_counts"]["HALVE"], "FLAT": m["action_counts"]["FLAT"],
        "API_accounted_usd": m["api_accounted_usd"], "citation_count_warnings": m["evidence_count_style_warnings"],
        "ppo_group_citations": m["ppo_group_citation_resolutions"],
        "alias_of": m["alias_of"] or "",
    } for m in metrics])
    print("\nCOMPLETE FROZEN_EVAL REPLAYS — FIXED POLICIES, PROTOTYPE")
    with pd.option_context("display.max_columns", None, "display.max_rows", 20):
        display(table)
    print("New HTTP attempts this execution:", llm_request_status()["attempts"] - before["attempts"])
    print("Final checkpoint folder:", LLM_FROZEN_EVAL_RUN_DIR)
    print("Results:", LLM_FROZEN_EVAL_RESULTS_PATH)
    display(FileLink(os.path.relpath(LLM_FROZEN_EVAL_RESULTS_PATH, Path.cwd())))
    print("Request accounting:", json.dumps(llm_request_status(), indent=2))
    return contexts, results, table


if not LLM_FROZEN_EVAL_RUN_LOCK.acquire(blocking=False):
    raise RuntimeError("The final evaluation runner is already active.")
try:
    try:
        LLM_FROZEN_EVAL_ACCOUNT_CONTEXTS, LLM_FROZEN_EVAL_RESULTS, LLM_FROZEN_EVAL_METRICS_TABLE = _run_frozen_evaluation_v1()
        LLM_FROZEN_EVAL_RUN_READY = True
    except BaseException:
        print("Final evaluation paused. Received responses and completed daily files remain saved.", flush=True)
        try:
            _e26_backup("paused final evaluation")
        except Exception as error:
            print("Could not create a new ZIP:", type(error).__name__, "| existing saved files remain on disk.")
        raise
finally:
    LLM_FROZEN_EVAL_RUN_LOCK.release()

_e26_backup("complete final evaluation")
print("Block 26 complete. Download the final backup and share the results table. No memory or selection updates occurred.")


FROZEN_EVAL | 6 independent accounts | 2998/3792 decisions restored
LLM decisions: 2528; separate accounts run in parallel, dates sequential within each account.
Frozen memory and prompt; fresh accounts; daily saves; ZIP backup every 50 dates.
Ctrl+C pauses dispatch; in-flight requests finish/save before the paused-run backup.
Progress and latest backup update in place. Full log: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/logs/frozen_eval_v1.log
Performance appears only after all scheduled conditions finish.
Recorded post-start amendment: supplied PPO-group citations resolve to the three allowed seed scores; original decisions retained.


[progress] saved=3792/3792 (100.0%) | LLM=2528/2528 | accounts complete=6/6 | active=0 | accounted=$2.0387/$10.00 | ETA~0.0m (rough)

/kaggle/working/eth_llm_backup_20260929T152624_467484Z.zip


COMPLETE FROZEN_EVAL REPLAYS — FIXED POLICIES, PROTOTYPE


,condition,memories,days,Sharpe,net_return_pct,max_drawdown_pct,mean_abs_target,KEEP,HALVE,FLAT,API_accounted_usd,citation_count_warnings,ppo_group_citations,alias_of
0,model_keep,0,632,5.638020,10009.310377,18.592855,0.685036,632,0,0,0.000000,0,0,
1,fixed_halve,0,632,5.555591,959.250505,9.607544,0.342518,0,632,0,0.000000,0,0,
2,llm_no_memory,0,632,5.002934,4292.639278,17.485842,0.589384,473,159,0,0.264689,0,0,
3,llm_raw_memory,20,632,5.254374,1019.821815,10.889991,0.382040,96,536,0,0.327131,0,2,
4,llm_screen_only,5,632,4.995636,1530.847563,9.607544,0.414491,166,466,0,0.292593,1,0,
5,llm_gate_selected,1,632,5.762607,10762.609083,18.592855,0.668619,552,80,0,0.282457,0,0,


New HTTP attempts this execution: 529
Final checkpoint folder: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/checkpoints/frozen_eval_v1
Results: /kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/evaluator_only/frozen_eval_v1_results.json


/kaggle/working/llm_memory_experiment/llm_runs/eth_memory_flash_v1/evaluator_only/frozen_eval_v1_results.json

Request accounting: {
  "active_requests": 0,
  "saved_responses": 4291,
  "attempts": 4293,
  "usage_based_usd": 2.027229324,
  "reserved_or_uncertain_usd": 0.0115005,
  "accounted_usd": 2.038729824,
  "budget_target_usd": 10.0
}
Block 26 complete. Download the final backup and share the results table. No memory or selection updates occurred.


In [49]:
# BLOCK 26A — Recover transport-interrupted final-evaluation requests.
# PAID RETRY: an earlier attempt may already have been billed. Its reservation
# remains recorded. Retry only identical inputs for a verified unexecuted day.
# No retry of received answers, no new trading actions, no prompt/memory edits.
# Run after Block 26 pauses, then rerun Block 26 to resume the evaluation.

import copy
import json
import uuid
from pathlib import Path
import pandas as pd

if not globals().get("LLM_FROZEN_EVAL_PROTOCOL_READY", False) or "_e26_prepare_account" not in globals():
    raise RuntimeError("Run Block 26 setup before this recovery cell.")
if llm_request_status()["active_requests"] or LLM_GATE_RUN_LOCK.locked():
    raise RuntimeError("Wait for all running requests/replays to stop first.")


def _r26a_record(messages, purpose, protocol):
    request_payload = _g24_request_payload(messages, protocol)
    key = _llm_digest({"config_hash": LLM_REQUEST_CLIENT.config_hash,
                       "purpose": purpose, "payload": request_payload})
    with LLM_DISK_LOCK:
        record = copy.deepcopy(LLM_REQUEST_CLIENT.records.get(key))
        if record is not None:
            saved = _llm_strict_json((Path(LLM_API_RUN_DIR) / "responses" / (key + ".json")).read_text())
            if (saved != record or record["request_id"] != key
                    or record["purpose"] != purpose or record["payload"] != request_payload):
                raise RuntimeError("Saved interrupted request identity differs: " + key)
    return record


def _r26a_inspect_cached(messages, purpose, protocol):
    record = _r26a_record(messages, purpose, protocol)
    if (record and record["attempts"]
            and record["attempts"][-1]["status"] in {"started", "transport_unknown"}):
        # This is for checkpoint inspection ONLY. A completed historical day
        # still requires a received response and will fail restoration if absent.
        return record, None
    return _e26_cached(messages, purpose, protocol)


def _recover_final_transport_requests():
    protocol, dates = _e26_check_protocol()
    with LLM_DISK_LOCK:
        uncertain = {key: copy.deepcopy(record) for key, record in LLM_REQUEST_CLIENT.records.items()
                     if record["purpose"].startswith("frozen_eval_v1/") and record["attempts"]
                     and record["attempts"][-1]["status"] in {"started", "transport_unknown"}}
    if not uncertain:
        print("No interrupted final-evaluation requests need a retry. No API calls made.")
        print("Rerun Block 26 to continue or reconstruct the completed results.")
        return
    manifest_path = Path(LLM_FROZEN_EVAL_RUN_DIR) / "run_manifest.json"
    manifest = _llm_strict_json(manifest_path.read_text())
    if (manifest["protocol_hash"] != _llm_digest(protocol)
            or manifest["source_stage"] != "FROZEN_EVAL"):
        raise RuntimeError("Final run manifest differs from the frozen protocol.")
    # Restore only saved history. Use a local copy so Block 26's normal refusal
    # to resend uncertain requests remains enabled after this cell completes.
    restore = _e26_logged_copy(_e26_prepare_account)
    restore.__globals__["_e26_cached"] = _r26a_inspect_cached
    contexts = [restore(c, protocol, dates, manifest["source_fingerprint"])
                for c in protocol["conditions"] if c["kind"] != "ALIAS"]
    targets = []
    for context in contexts:
        replay, condition = context["replay"], context["condition"]
        if condition["kind"] != "LLM" or replay.done:
            continue
        index = replay._i
        messages, payload = _e26_inputs(replay.get_state(), index, condition, dates, protocol)
        purpose = _e26_purpose(condition, dates[index])
        record = _r26a_record(messages, purpose, protocol)
        if record and record["request_id"] in uncertain:
            if len(record["attempts"]) >= int(LLM_REQUEST_CLIENT.config["max_attempts"]):
                raise RuntimeError("Request already reached the frozen attempt limit: " + purpose
                                   + ". Keep the saved records; do not reset attempts or delete the cache.")
            targets.append((record, messages, payload, condition, dates[index]))
    if {record["request_id"] for record, *_ in targets} != set(uncertain):
        raise RuntimeError("An uncertain request does not match the next saved account state. No retries made; inspect it first.")
    print("Interrupted requests matching unexecuted next days:", len(targets))
    print("The prior attempts may have been billed; their reservations stay in accounting.")
    print("The same messages, model, settings, cache key, spending guard and attempt limit apply.")
    request = _e26_logged_copy(LLM_REQUEST_CLIENT.request)
    for record, messages, payload, condition, date in targets:
        key = record["request_id"]
        with LLM_DISK_LOCK:
            if LLM_REQUEST_CLIENT.records[key] != record:
                raise RuntimeError("Request changed during recovery inspection: " + key)
        audit = {
            "schema_version": 1, "event_type": "explicit_transport_recovery",
            "protocol_hash": _llm_digest(protocol), "request_id": key, "purpose": record["purpose"],
            "request_payload_hash": _llm_digest(record["payload"]),
            "prior_attempts_hash": _llm_digest(record["attempts"]),
            "prior_attempt_count": len(record["attempts"]),
            "prior_accounted_usd": sum(a["accounted_usd"] for a in record["attempts"]),
            "started_utc": _llm_now(), "status": "RETRY_REQUESTED",
            "reason": "No completed response received; exact next unexecuted decision reconstructed from checkpoints.",
            "billing_note": "Duplicate provider billing is possible; no prior reservation is released.",
            "response_selection": "Accept the first completed structurally valid response; no outcome or performance-based retries.",
            "original_response_available": False, "new_action_executed_by_recovery": False,
        }
        path = (Path(LLM_API_RUN_DIR) / "events" / "transport_recovery"
                / (key + "_" + uuid.uuid4().hex + ".json"))
        llm_atomic_write_json(path, audit)
        print(f"[recover] {condition['condition_id']} | {date.date()} | {key[:12]}", flush=True)
        try:
            result = request(messages, purpose=record["purpose"],
                             max_tokens=int(record["payload"]["max_tokens"]), retry_uncertain=True)
            if result["request_id"] != key:
                raise RuntimeError("Retry produced a different request identity.")
            _e26_checked_decision(result, payload, condition, date)
            audit["status"] = "RESPONSE_SAVED_AND_VALIDATED"
            print("[recovered] Response saved. Block 26 will execute this day once.", flush=True)
        except BaseException as error:
            audit["status"] = "PAUSED"
            audit["error_type"] = type(error).__name__
            audit["error"] = str(error)
            raise
        finally:
            with LLM_DISK_LOCK:
                current = copy.deepcopy(LLM_REQUEST_CLIENT.records[key])
            if current["attempts"][:len(record["attempts"])] != record["attempts"]:
                raise RuntimeError("Prior attempt history changed unexpectedly; preserve and inspect the files.")
            audit["finished_utc"] = _llm_now()
            audit["attempt_count_after"] = len(current["attempts"])
            audit["accounted_usd_after"] = sum(a["accounted_usd"] for a in current["attempts"])
            audit["last_attempt_status"] = current["attempts"][-1]["status"]
            llm_atomic_write_json(path, audit)
    print("Recovery complete. No new daily trading actions executed. Rerun Block 26.")


if not LLM_FROZEN_EVAL_RUN_LOCK.acquire(blocking=False):
    raise RuntimeError("Block 26 is still running. Wait for it to finish pausing.")
try:
    _recover_final_transport_requests()
finally:
    try:
        _e26_backup("after interrupted-request recovery")
    finally:
        LLM_FROZEN_EVAL_RUN_LOCK.release()


Interrupted requests matching unexecuted next days: 1
The prior attempts may have been billed; their reservations stay in accounting.
The same messages, model, settings, cache key, spending guard and attempt limit apply.
[recover] llm_raw_memory | 2026-05-14 | c00e054de8a7
[recovered] Response saved. Block 26 will execute this day once.
Recovery complete. No new daily trading actions executed. Rerun Block 26.


In [51]:
from pathlib import Path
from IPython.display import display, FileLink
import os

backup = Path("/kaggle/working/eth_llm_backup_20260929T152624_467484Z.zip")

if not backup.is_file():
    raise FileNotFoundError(f"Backup not found: {backup}")

display(FileLink(
    os.path.relpath(backup, Path.cwd()),
    result_html_prefix="Download completed experiment: "
))

/kaggle/working/eth_llm_backup_20260929T152624_467484Z.zip